# 02 — Chunking experiments: Steps 1–12

Steps 1–4 prepare the approved, frozen experiment setup. **Step 5 implements only the strict
fixed-length character baseline**, comparing nine configurations on the four complete documents
reserved in that setup. The 12 frozen windows select what to inspect; they never determine cuts.
Step 6 adds a conservative, uncapped structural baseline on the same inputs.
Step 7 compares explicit parent-child relationships on the same source.
Step 8 compares controlled oversized-unit splitting experiments.
Step 9 investigates separate history annotations while preserving all original text.
Step 10 explores document-registry and selective metadata joins.
Step 11 compares existing strategies, then reads mentor references for evaluation only.
Step 12 tests an evidence-gated provisional hybrid using existing units and child spans.
Stop after Step 12 for review. No production-optimal strategy or parameters are claimed.

**Corpus input:** `data/processed/01_parsed_docx_structure.json`. Step 5 also reads the frozen
Stage 2 sample manifest as experiment configuration, never as an alternative text source.
No DOCX reparsing, source-URL requests, cleaning, tokenization,
embeddings, retrieval, or model calls occur. Source text stays unchanged. Mentor excerpts enter
only the separate read-only evaluation section after the independent Step 11 comparison.

Run from the project or `notebooks` directory with the project Python kernel, then **Restart
Kernel and Run All Cells**. Expand full-text and source-map panels. Steps 1–4 save/recheck the
original manifest; Steps 5–12 save separate deterministic diagnostic reports under the ignored
`data/processed/02_chunking_experiments/` directory. Keep tracked notebook outputs cleared.
See [the approved Stage 2 plan](../docs/stage-02-chunking-experiments-plan.md).

## Step 1 — Explain the strategies and experiment boundaries

| Strategy | Main question | Main trade-off |
|---|---|---|
| Fixed-length | What happens when length determines boundaries? | Predictable size versus broken provisions and duplicated overlap |
| Structural | What happens when evidenced legal units determine boundaries? | Legal continuity versus uneven sizes and ambiguous markers |
| Parent-child | Can small units remain connected to useful larger context? | Focused child spans versus potentially excessive parent context |
| Hybrid | Which observed failures justify combining approaches? | Adaptability versus additional rules requiring evidence |

Parent-child is a **relationship model**, not a competing length metric: a child still needs a
splitting method. In a later system, children could be embedded/searched and linked parents
could supply context. None of those operations happens in this notebook.

Official conceptual references: [LangChain splitting overview](https://docs.langchain.com/oss/python/integrations/splitters)
and [LlamaIndex hierarchical parser](https://docs.llamaindex.ai/en/v0.10.22/api_reference/node_parsers/hierarchical/).
They describe mechanisms, not optimal Bulgarian legal-corpus parameters. No framework dependency is needed here.

Every later experiment must answer:
1. **What** are we testing?
2. **Why** might it help this corpus?
3. **How** are boundaries or relationships constructed?
4. What intermediate output should we inspect?
5. Which design choices and trade-offs matter?
6. Which observations favor or disfavor the strategy?
7. What remains unproven?

For this setup, favor samples exposing complete provisions, edge cases, and ambiguous evidence.
Question a selection if it hides surrounding text or mistakes an inline citation for a boundary.
Step 5 below implements the fixed-length experiment only. Step 5 below implements the fixed-length experiment only. No final chunk size, legal hierarchy, metadata schema, or strategy is selected.

## Step 2 — Load and verify the Stage 1 interface

**What / why:** Verify the approved, fidelity-validated artifact before deriving any views.
**How:** Locate the project, read the JSON once, check its schema and cross-references, and
record both the artifact SHA-256 and the original DOCX fingerprint stored inside it.
The stored DOCX fingerprint is provenance; this notebook does not reopen the archive to verify it.

**Design choice:** Pin the inspected artifact bytes. If Stage 1 changes, stop and review the
sample locations instead of silently applying old positions to new source content.
The validator also checks structure, so the recorded `PASS` flag alone is not sufficient.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from bisect import bisect_left, bisect_right
import hashlib
import html
import json
import math
from IPython.display import HTML, display


def require(condition, message):
    if not condition:
        raise ValueError(message)


def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def fingerprint(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Start Jupyter in the project or its notebooks directory.")


ROOT = project_root()
ARTIFACT = ROOT / "data/processed/01_parsed_docx_structure.json"
EXPECTED_ARTIFACT_SHA256 = "caa696a76ea403bf1ddf074aa503fd4362162d128e32a2c755570464bc95ebdc"
require(ARTIFACT.is_file(), "Run and validate Stage 1 first: the parsed artifact is missing.")
artifact_bytes = ARTIFACT.read_bytes()
artifact_sha256 = hashlib.sha256(artifact_bytes).hexdigest()
require(artifact_sha256 == EXPECTED_ARTIFACT_SHA256,
        "Stage 1 artifact changed. Review source identity and sample positions before updating the pin.")
try:
    corpus = json.loads(artifact_bytes)
except (UnicodeDecodeError, json.JSONDecodeError) as exc:
    raise ValueError("Stage 1 artifact is not valid UTF-8 JSON.") from exc

In [ ]:
def validate_stage1(value):
    require(isinstance(value, dict), "Artifact root must be an object.")
    require(value.get("schema_version") == 1, "Unsupported Stage 1 schema version.")
    for name in ("source", "parser", "documents", "blocks", "diagnostics", "metadata_provenance"):
        require(name in value, f"Missing Stage 1 field: {name}")
    require(value["diagnostics"].get("stage1_validation", {}).get("result") == "PASS",
            "Stage 1 must report PASS before Stage 2.")
    require(isinstance(value["source"].get("sha256"), str)
            and len(value["source"]["sha256"]) == 64, "Missing source DOCX fingerprint.")
    docs, blocks = value["documents"], value["blocks"]
    require(isinstance(docs, list) and docs, "Missing document records.")
    require(isinstance(blocks, list) and blocks, "Missing ordered blocks.")
    doc_ids = [d["document_id"] for d in docs]
    require(len(set(doc_ids)) == len(doc_ids), "Duplicate document IDs.")
    ordinals = [d["ordinal"] for d in docs]
    require(len(set(ordinals)) == len(ordinals), "Duplicate document ordinals.")
    ids = [b["id"] for b in blocks]
    require(len(set(ids)) == len(ids), "Duplicate body block IDs.")
    positions = [b["position"] for b in blocks]
    require(positions == sorted(set(positions)), "Body positions must be unique and increasing.")
    by_id = {b["id"]: b for b in blocks}
    allowed_regions = {"front_matter", "document_metadata", "reference_rules", "full_text"}
    for b in blocks:
        require(b.get("region") in allowed_regions, f"Unknown region at {b['position']}.")
        require(b.get("document_id") in doc_ids or
                (b.get("document_id") is None and b["region"] == "front_matter"),
                f"Invalid document membership at {b['position']}.")
        require("source" in b and "structural_annotations" in b, "Missing block provenance/annotations.")
        if b["region"] == "full_text":
            require(b["type"] == "paragraph" and isinstance(b.get("raw_text"), str),
                    "Unexpected full-text block type: review a faithful representation before proceeding.")
        for marker in b["structural_annotations"]["legal_marker_candidates"]:
            start, end = marker["character_span"]
            require(marker["block_id"] == b["id"] and marker["document_id"] == b["document_id"],
                    "Candidate marker references the wrong block/document.")
            require(0 <= start < end <= len(b["raw_text"]), "Invalid candidate character span.")
            require(b["raw_text"][start:end] == marker["matched_label"], "Candidate label differs from source.")
    assigned = set()
    for d in docs:
        lo, hi = d["block_range"]["start_index"], d["block_range"]["end_index_exclusive"]
        require(0 <= lo < hi <= len(blocks), "Invalid document block range.")
        indices = set(range(lo, hi))
        require(not assigned.intersection(indices), "Overlapping document ranges.")
        assigned.update(indices)
        require(all(b["document_id"] == d["document_id"] for b in blocks[lo:hi]),
                "Document range and block membership disagree.")
        require(d.get("source_titles") and d.get("source_metadata"), "Missing document metadata.")
        for evidence_id in d["boundary_evidence"].values():
            if evidence_id is not None:
                require(evidence_id in by_id and by_id[evidence_id]["document_id"] == d["document_id"],
                        "Invalid document boundary evidence.")
        marker = by_id[d["boundary_evidence"]["full_text_marker_block_id"]]
        require(marker["region"] == "full_text" and marker["raw_text"] == "Пълен текст / Full text",
                "Full-text delimiter differs from the inspected standalone marker; review it.")
    require(assigned == {i for i, b in enumerate(blocks) if b["document_id"] is not None},
            "Document ranges do not account for every assigned block.")
    require(dict(Counter(b["region"] for b in blocks)) == value["diagnostics"]["region_counts"],
            "Region counts disagree with Stage 1 diagnostics.")
    return {"result": "PASS", "documents": len(docs), "body_blocks": len(blocks)}


validation = validate_stage1(corpus)
corpus_fingerprint_before = fingerprint(corpus)
DOCUMENTS = {d["document_id"]: d for d in corpus["documents"]}
DOC_BY_ORDINAL = {d["ordinal"]: d for d in corpus["documents"]}
BLOCKS = {b["id"]: b for b in corpus["blocks"]}
BY_POSITION = {b["position"]: b for b in corpus["blocks"]}
print(validation)
print("Artifact SHA-256:", artifact_sha256)
print("Original DOCX SHA-256 (stored):", corpus["source"]["sha256"])
print("Schema:", corpus["schema_version"], "| recorded Stage 1 result:",
      corpus["diagnostics"]["stage1_validation"]["result"])

### 2a. Inspect the inventory and preserved evidence

The lightweight HTML helpers below format data only. Text is HTML-escaped; source strings
are never interpreted as markup. Tables default to complete output, with any explicit preview
limit labeled. Full descriptors and annotations are expandable.

In [ ]:
def esc(value):
    return html.escape(str(value), quote=True)


def table_html(rows, columns=None, limit=None):
    rows = list(rows)
    columns = list(columns or (rows[0].keys() if rows else []))
    visible = rows if limit is None else rows[:limit]
    header = "".join(f"<th>{esc(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(row.get(c, ''))}</td>" for c in columns)
                   + "</tr>" for row in visible)
    note = f"Showing {len(visible)} of {len(rows)} rows."
    return f"<p>{note}</p><div style='overflow:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"


def show_table(rows, columns=None, limit=None):
    display(HTML(table_html(rows, columns, limit)))


def details_html(label, text):
    return f"<details><summary>{esc(label)}</summary><pre style='white-space:pre-wrap;overflow-wrap:anywhere'>{esc(text)}</pre></details>"


def show_json(label, value):
    display(HTML(details_html(label, json.dumps(value, ensure_ascii=False, indent=2))))


show_table([{
    "document": d["ordinal"], "title": d["source_titles"]["title"]["raw_text"],
    "full_text_blocks_including_delimiter": sum(b["document_id"] == d["document_id"]
                                              and b["region"] == "full_text" for b in corpus["blocks"])
} for d in corpus["documents"]])
show_table([{"candidate_kind": k, "count": v}
            for k, v in corpus["diagnostics"]["candidate_marker_counts"].items()])
show_table(corpus["diagnostics"]["warnings"])
show_json("Complete Document 1 descriptor: source values and evidence locations", DOC_BY_ORDINAL[1])
for position in (56, 57, 59, 61, 112, 2051, 2848, 25783):
    b = BY_POSITION[position]
    show_json(f"Block {position}: source text and unchanged Stage 1 annotations", {
        k: b[k] for k in ("id", "position", "document_id", "region", "source", "raw_text", "structural_annotations")
    })

### 2b. Build reversible document views — not chunks

**How:** Concatenate each document's full-text paragraphs with exactly one added `\n`
between blocks. Store separate map segments for source text and added separators, including
zero-length entries for empty blocks. Only the evidence-linked standalone full-text delimiter
is omitted from the view and recorded in a separate ledger. Everything else in full text,
including navigation and publication history, stays present.

Offsets are zero-based, half-open `[start, end)`, measured in **Python Unicode characters**,
not bytes or tokens. A source map for a selected span includes block-relative offsets and
added-separator records. Empty blocks remain in the complete map even though they occupy no
characters. Metadata/reference regions remain in `corpus`, outside the text views.

**Inspect:** Region accounting, excluded delimiters, the map around an Article boundary, and
exact reconstruction. A map failure argues against proceeding to experiments.

In [ ]:
def build_document_view(document, ordered_blocks):
    document_id = document["document_id"]
    delimiter_id = document["boundary_evidence"]["full_text_marker_block_id"]
    selected = [b for b in ordered_blocks if b["document_id"] == document_id and b["region"] == "full_text"]
    delimiter = [b for b in selected if b["id"] == delimiter_id]
    require(len(delimiter) == 1 and delimiter[0]["raw_text"] == "Пълен текст / Full text",
            "Expected one evidence-linked standalone delimiter.")
    kept = [b for b in selected if b["id"] != delimiter_id]
    pieces, segments, offset = [], [], 0
    for i, b in enumerate(kept):
        if i:
            segments.append({"kind": "separator", "start": offset, "end": offset + 1,
                             "left_block_id": kept[i - 1]["id"], "right_block_id": b["id"]})
            pieces.append("\n")
            offset += 1
        text = b["raw_text"]
        segments.append({"kind": "source", "start": offset, "end": offset + len(text),
                         "block_id": b["id"], "position": b["position"]})
        pieces.append(text)
        offset += len(text)
    return {"document_id": document_id, "text": "".join(pieces), "segments": segments,
            "block_ids": [b["id"] for b in kept], "delimiter_block_id": delimiter_id}


VIEWS = {did: build_document_view(d, corpus["blocks"]) for did, d in DOCUMENTS.items()}
BLOCK_SPANS = {seg["block_id"]: (did, seg["start"], seg["end"])
               for did, view in VIEWS.items() for seg in view["segments"] if seg["kind"] == "source"}
# Bisect supports repeated starts/ends caused by empty paragraphs.
SEGMENT_ENDS = {did: [seg["end"] for seg in view["segments"]] for did, view in VIEWS.items()}
EXCLUDED_DELIMITERS = [{"document_id": did, "block_id": v["delimiter_block_id"],
                       "text": BLOCKS[v["delimiter_block_id"]]["raw_text"],
                       "reason": "Stage 1 evidence-linked standalone full-text delimiter"}
                      for did, v in VIEWS.items()]


def source_map(document_id, start, end):
    view = VIEWS[document_id]
    require(isinstance(start, int) and isinstance(end, int) and 0 <= start <= end <= len(view["text"]),
            "Invalid document character span.")
    if start == end:
        return []
    rows = []
    for seg in view["segments"][bisect_right(SEGMENT_ENDS[document_id], start):]:
        if seg["start"] >= end:
            break
        lo, hi = max(start, seg["start"]), min(end, seg["end"])
        if lo >= hi:
            continue
        if seg["kind"] == "source":
            b = BLOCKS[seg["block_id"]]
            rows.append({"kind": "source", "stream_start": lo, "stream_end": hi,
                         "block_id": b["id"], "position": b["position"],
                         "block_start": lo - seg["start"], "block_end": hi - seg["start"],
                         "source": b["source"]})
        else:
            rows.append({"kind": "separator", "stream_start": lo, "stream_end": hi,
                         "text": "\n", "left_block_id": seg["left_block_id"],
                         "right_block_id": seg["right_block_id"]})
    return rows


def reconstruct(mapped):
    return "".join(BLOCKS[r["block_id"]]["raw_text"][r["block_start"]:r["block_end"]]
                   if r["kind"] == "source" else r["text"] for r in mapped)


for did, view in VIEWS.items():
    require(reconstruct(source_map(did, 0, len(view["text"]))) == view["text"],
            "Full-document reconstruction failed.")
    for seg in view["segments"]:
        if seg["kind"] == "source":
            require(view["text"][seg["start"]:seg["end"]] == BLOCKS[seg["block_id"]]["raw_text"],
                    "Source paragraph changed in view.")
    require(len(view["segments"]) == max(0, 2 * len(view["block_ids"]) - 1), "Missing map entries.")

show_table([{"region": k, "blocks": v, "view_policy":
             "Retain all except evidence-linked delimiter" if k == "full_text" else "Retain in artifact only"}
            for k, v in Counter(b["region"] for b in corpus["blocks"]).items()])
show_table([{"document": DOCUMENTS[did]["ordinal"], "characters": len(v["text"]),
             "source_blocks": len(v["block_ids"]), "added_newlines": max(0, len(v["block_ids"]) - 1),
             "empty_blocks_preserved": sum(BLOCKS[bid]["raw_text"] == "" for bid in v["block_ids"])}
            for did, v in VIEWS.items()])
show_json("Excluded delimiter ledger (only these full-text blocks are omitted)", EXCLUDED_DELIMITERS)
did, start, _ = BLOCK_SPANS[BY_POSITION[61]["id"]]
end = BLOCK_SPANS[BY_POSITION[63]["id"]][2]
show_json("Article boundary map: original blocks versus added newlines", source_map(did, start, end))
print("PASS: every document view reconstructs exactly from source blocks plus documented separators.")

## Step 3 — Select and freeze representative content

### 3a. Inventory lengths without creating structural chunks

**What / why:** Inspect variation before accepting a small demonstration set as representative.
Actual paragraph lengths are exact. Candidate-unit lengths below are **rough scouting estimates**:
from an existing Stage 1 marker to the next marker or Word heading (or document end), excluding
the trailing added separator. This is intentionally not a legal-unit parser, does not resolve
hierarchy, and does not partition the corpus into chunks. In particular, Part/Chapter estimates
cover only the immediate run after their heading, not all descendants. A heading can prematurely
end an estimate; all such estimates remain unreviewed.

Percentiles use a documented nearest-rank definition (including the median column). Select an
additional median-sized Article if Чл. 1 lies outside the candidate Article interquartile range.
This is a sampling rule, not a chunk-size threshold. Keep deterministic ties in source order.

In [ ]:
def length_summary(values):
    values = sorted(values)
    if not values:
        return {"count": 0, "min": None, "p25": None, "median_p50": None, "p90": None,
                "p95": None, "p75": None, "max": None}
    def percentile(p):
        return values[max(0, math.ceil(p * len(values)) - 1)]
    return {"count": len(values), "min": values[0], "p25": percentile(.25),
            "median_p50": percentile(.5), "p75": percentile(.75), "p90": percentile(.9),
            "p95": percentile(.95), "max": values[-1]}


paragraph_inventory = [{"document": DOCUMENTS[did]["ordinal"], "position": BLOCKS[bid]["position"],
                        "characters": len(BLOCKS[bid]["raw_text"]), "preview": BLOCKS[bid]["raw_text"][:140]}
                       for did, v in VIEWS.items() for bid in v["block_ids"]]
candidate_inventory = []
for did, view in VIEWS.items():
    source_blocks = [BLOCKS[bid] for bid in view["block_ids"]]
    stops = [i for i, b in enumerate(source_blocks)
             if b["structural_annotations"]["legal_marker_candidates"]
             or b["structural_annotations"]["word_heading_level"] is not None]
    for i, b in enumerate(source_blocks):
        markers = b["structural_annotations"]["legal_marker_candidates"]
        if not markers:
            continue
        later = bisect_right(stops, i)
        next_index = stops[later] if later < len(stops) else len(source_blocks)
        first, last = b, source_blocks[next_index - 1]
        lo, hi = BLOCK_SPANS[first["id"]][1], BLOCK_SPANS[last["id"]][2]
        for marker in markers:
            candidate_inventory.append({"document": DOCUMENTS[did]["ordinal"], "kind": marker["kind"],
                "label": marker["matched_label"], "first_position": first["position"],
                "last_position": last["position"], "estimated_characters": hi - lo,
                "status": "unreviewed marker-to-next-anchor estimate; not a chunk"})

show_table([{"inventory": "all source paragraphs", **length_summary(r["characters"] for r in paragraph_inventory)}])
show_table([{"candidate_kind": kind, **length_summary(r["estimated_characters"] for r in candidate_inventory if r["kind"] == kind)}
            for kind in sorted({r["kind"] for r in candidate_inventory})])
show_table(sorted(paragraph_inventory, key=lambda r: (-r["characters"], r["position"])), limit=8)
show_table(sorted(candidate_inventory, key=lambda r: (-r["estimated_characters"], r["first_position"])), limit=8)

### 3b. Explicit source-window selection

Each sample is an inspection window, **not a generated chunk**. Its endpoints are explicit,
source-checked choices. We preserve full block IDs, document-relative offsets, marker evidence,
and a pending human-review status. An unclassified heading is shown as unclassified even when
its wording suggests additional or transitional provisions.

The § samples deliberately distinguish `Допълнителни разпоредби` from `Преходни разпоредби`.
Repeated § numbering in later amendment sections is not a global identifier. The weakly
structured sample is the **whole full-text view of Document 16**, retaining navigation; there
is no manufactured Article parent. Complete-document inputs are frozen separately for future
fair comparisons: later strategies must not restart their windows at these sample boundaries.

**Inspect before accepting:** the first/last blocks, neighboring blocks outside each sample,
nearby headings (evidence only, not an asserted ancestor tree), and the expandable entire text.
Sample freezing fixes identity for repeatability; it does not mean legal-boundary approval.

In [ ]:
# Endpoints are inclusive body positions, not list indexes or page numbers.
# Guards prevent a changed marker from being silently treated as the same example.
SAMPLE_SPECS = [
    ("normal_article", 1, 61, 62, "Чл. 1.", "Complete two-paragraph Article with inline amendment note.",
     "Check whether both paragraphs provide enough legal context; cross-references may need more."),
    ("short_neighbors", 1, 63, 65, "Чл. 2.", "Short Article and Cyrillic-suffixed Чл. 2а expose merging and label risks.",
     "This comparison window deliberately contains two Articles; it is not one legal unit."),
    ("nested_structure", 1, 110, 116, "Глава трета.", "Chapter/Section labels, separate titles, and Article 10.",
     "Heading-title adjacency is observed; no automatic ancestor tree has been assigned."),
    ("long_article", 2, 2779, 2810, "Чл. 7.", "Long list-bearing Article ends immediately before Чл. 8 at 2811.",
     "Review numbered subitems and exceptions; boundary fidelity is not legal interpretation."),
    ("additional_provision", 1, 2051, 2055, "Допълнителни разпоредби", "Additional-provision heading plus § 1.",
     "Heading is unclassified by Stage 1; distinguish this scope from later amendment sections."),
    ("transitional_provision", 1, 2158, 2166, "Преходни разпоредби", "Transitional heading plus § 6 before § 7 at 2167.",
     "Do not identify § 6 globally or infer effective legal status from these source words."),
    ("annex", 2, 2848, 2893, "Приложение № 1", "Complete form-like Annex before Annex 2 at 2894.",
     "Form blanks and numbered entries are preserved; they are not automatically Articles."),
    ("long_paragraph", 11, 25737, 25737, "Понастоящем", "A 3,600-character prose paragraph exceeds small candidate sizes.",
     "The paragraph is an excerpt in a wider recital sequence, not an independent provision."),
    ("publication_history", 1, 56, 56, "Обн. ДВ.", "A 4,077-character publication list can dominate a future chunk.",
     "No history text is removed and dates are not interpreted as verified legal status."),
    ("eu_article", 11, 25783, 25785, "Член 1", "EU Article heading, separate subject title, and body before Член 2.",
     "Different heading convention; verify that the separate title remains attached."),
    ("unclassified_heading", 1, 2051, 2051, "Допълнителни разпоредби", "Isolate a real unclassified Stage 1 heading.",
     "Text suggests a role but Stage 1 has no legal-marker candidate here; keep uncertainty explicit."),
]
weak_doc = DOC_BY_ORDINAL[16]
weak_blocks = VIEWS[weak_doc["document_id"]]["block_ids"]
SAMPLE_SPECS.append(("weak_structure", 16, BLOCKS[weak_blocks[0]]["position"], BLOCKS[weak_blocks[-1]]["position"],
                     "Начало", "Entire weakly structured source-page text including navigation.",
                     "No Article hierarchy; navigation remains present and no cleaning decision is made."))
article_estimates = [r for r in candidate_inventory if r["kind"] == "article"]
article_distribution = length_summary(r["estimated_characters"] for r in article_estimates)
normal_size = BLOCK_SPANS[BY_POSITION[62]["id"]][2] - BLOCK_SPANS[BY_POSITION[61]["id"]][1]
normal_is_atypical = not article_distribution["p25"] <= normal_size <= article_distribution["p75"]
median_selection = {"rule": "add nearest median estimate if normal Article is outside [p25, p75]",
                    "normal_characters": normal_size, "distribution": article_distribution,
                    "additional_sample_needed": normal_is_atypical}
if normal_is_atypical:
    median_row = min(article_estimates, key=lambda r: (abs(r["estimated_characters"] - article_distribution["median_p50"]),
                                                       r["first_position"]))
    SAMPLE_SPECS.append(("median_article_candidate", median_row["document"], median_row["first_position"],
                         median_row["last_position"], median_row["label"],
                         "Deterministic near-median Article candidate balances unusually sized examples.",
                         "Endpoint comes from the scouting estimate and requires human boundary review."))
show_json("Is an additional median Article needed?", median_selection)

In [ ]:
def observed_context(document_id, position, limit=6):
    # Nearby evidence only: deliberately no stack, nesting, or parent assignment.
    blocks = [BLOCKS[bid] for bid in VIEWS[document_id]["block_ids"]
              if BLOCKS[bid]["position"] <= position
              and (BLOCKS[bid]["structural_annotations"]["word_heading_level"] is not None
                   or BLOCKS[bid]["structural_annotations"]["legal_marker_candidates"])]
    return [{"block_id": b["id"], "position": b["position"], "text": b["raw_text"][:180],
             "heading_level": b["structural_annotations"]["word_heading_level"],
             "candidate_labels": [c["matched_label"] for c in b["structural_annotations"]["legal_marker_candidates"]]}
            for b in blocks[-limit:]]


def select_sample(spec):
    name, ordinal, first, last, prefix, why, question = spec
    document = DOC_BY_ORDINAL[ordinal]
    did = document["document_id"]
    blocks = [b for b in corpus["blocks"] if first <= b["position"] <= last]
    require(blocks and blocks[0]["position"] == first and blocks[-1]["position"] == last,
            f"Missing endpoint for {name}.")
    require(all(b["document_id"] == did and b["id"] in BLOCK_SPANS for b in blocks),
            f"Sample {name} crosses a document or includes non-corpus content.")
    require(blocks[0]["raw_text"].startswith(prefix), f"Start label changed for {name}.")
    lo, hi = BLOCK_SPANS[blocks[0]["id"]][1], BLOCK_SPANS[blocks[-1]["id"]][2]
    return {"sample_id": name, "document_id": did, "document_ordinal": ordinal,
            "first_position": first, "last_position_inclusive": last,
            "start": lo, "end": hi, "characters": hi - lo,
            "block_ids": [b["id"] for b in blocks], "source_spans": source_map(did, lo, hi),
            "source_text_sha256": hashlib.sha256(VIEWS[did]["text"][lo:hi].encode("utf-8")).hexdigest(),
            "selection_rationale": why, "unresolved_boundary_question": question,
            "review_status": "source-window identity checked; human review pending",
            "observed_context_not_ancestry": observed_context(did, first)}


samples = [select_sample(spec) for spec in SAMPLE_SPECS]
require(len({s["sample_id"] for s in samples}) == len(samples), "Duplicate sample names.")
require(not BY_POSITION[2051]["structural_annotations"]["legal_marker_candidates"],
        "Unclassified-heading sample no longer matches its purpose.")
for position, prefix in ((63, "Чл. 2."), (66, "Чл. 3."), (117, "Чл. 11."), (2811, "Чл. 8."),
                         (2056, "§ 1а."), (2167, "§ 7."), (2894, "Приложение № 2"), (25786, "Член 2")):
    require(BY_POSITION[position]["raw_text"].startswith(prefix), "Expected following boundary changed.")


def window_rows(sample, padding=2):
    view = VIEWS[sample["document_id"]]
    ids = view["block_ids"]
    first = ids.index(sample["block_ids"][0])
    last = ids.index(sample["block_ids"][-1])
    # Both edges plus adjacent blocks; full body is shown in a separate expandable panel.
    indices = sorted(set(range(max(0, first - padding), min(len(ids), first + padding + 1)))
                     | set(range(max(0, last - padding), min(len(ids), last + padding + 1))))
    return [{"position": BLOCKS[ids[i]]["position"], "within_sample": first <= i <= last,
             "characters": len(BLOCKS[ids[i]]["raw_text"]),
             "style": BLOCKS[ids[i]]["style"]["name"],
             "candidate_labels": [c["matched_label"] for c in BLOCKS[ids[i]]["structural_annotations"]["legal_marker_candidates"]],
             "preview": BLOCKS[ids[i]]["raw_text"][:220]} for i in indices]


show_table([{"sample": s["sample_id"], "document": s["document_ordinal"],
             "positions_inclusive": f"{s['first_position']}–{s['last_position_inclusive']}",
             "characters": s["characters"], "why": s["selection_rationale"]} for s in samples])
for s in samples:
    text = VIEWS[s["document_id"]]["text"][s["start"]:s["end"]]
    display(HTML(f"<h4>{esc(s['sample_id'])} — source selection, not a chunk</h4>"
                 f"<p>{esc(s['unresolved_boundary_question'])}</p>"))
    show_table(window_rows(s))
    show_json("Nearby heading/marker evidence — not inferred ancestry", s["observed_context_not_ancestry"])
    display(HTML(details_html(f"Complete sample: {len(text):,} characters; no truncation", text)))

### 3c. Freeze the manifest and full-document inputs

Canonical JSON plus a SHA-256 fingerprint freezes sample definitions, exact source maps,
selection rationale, uncertainty, and full-document view identities. No timestamps are added.
Changing a source pin, sample endpoint, or selection rationale changes the manifest fingerprint.
The display helpers below reject spans bearing a different setup fingerprint, so stale views
cannot quietly appear as results from a new setup.

The persisted filename includes the fingerprint. Rerunning the same setup recreates identical
bytes; a changed setup creates a separate manifest. Persistence happens only after Step 4 checks.

In [ ]:
selected_document_ids = sorted({s["document_id"] for s in samples}, key=lambda did: DOCUMENTS[did]["ordinal"])
manifest = {
    "manifest_version": 1, "scope": "Stage 2 Steps 1–4: source selections only; no chunks",
    "artifact_path": str(ARTIFACT.relative_to(ROOT)), "artifact_sha256": artifact_sha256,
    "source_docx_sha256": corpus["source"]["sha256"],
    "offset_convention": "zero-based half-open Python Unicode characters",
    "view_policy": "full_text paragraphs; one added newline; omit only evidence-linked delimiter",
    "median_selection": median_selection, "samples": samples,
    "complete_document_inputs": [{"document_id": did, "ordinal": DOCUMENTS[did]["ordinal"],
        "characters": len(VIEWS[did]["text"]), "block_ids": VIEWS[did]["block_ids"],
        "text_sha256": hashlib.sha256(VIEWS[did]["text"].encode("utf-8")).hexdigest()}
        for did in selected_document_ids],
    "review_status": "pending user review after Step 4",
}
FROZEN_MANIFEST_JSON = canonical(manifest)
SETUP_ID = hashlib.sha256(FROZEN_MANIFEST_JSON.encode("utf-8")).hexdigest()


def frozen_samples():
    # Return fresh objects, protecting the canonical frozen selection from display-side mutation.
    return json.loads(FROZEN_MANIFEST_JSON)["samples"]


print("Frozen setup SHA-256:", SETUP_ID)
print("Sample count:", len(frozen_samples()), "| complete documents reserved:", len(selected_document_ids))
show_table(manifest["complete_document_inputs"], ["ordinal", "document_id", "characters", "text_sha256"])
show_json("Complete frozen manifest: expand to inspect all block IDs and source spans", json.loads(FROZEN_MANIFEST_JSON))

## Step 4 — Create shared inspection/display helpers

**What / why:** Prepare consistent, source-linked displays so later strategies can be compared
without hiding boundaries, duplicated context, or uncertainty.

**How:** The helpers consume explicit spans supplied by their caller. They never choose cut
points, form a hierarchy, generate chunks, or fetch a parent. A future caller must supply any
parent and child records, and explicitly reviewed legal-unit spans. Current demonstrations
use whole selected source windows; a tiny separate synthetic fixture checks highlighting.

Each card shows source/document identity, character count, beginning/end, full text, mapping,
observed context, actual overlap with neighboring supplied spans, and any reviewed-unit split
or combination diagnostics. A missing legal-boundary review is labeled **unknown**, not false.
Paragraph splitting is a mechanical source-block check, separate from legal-meaning review.

Generated labels and highlights are display-only. Gold highlights mean overlapping supplied
spans; cut markers are annotations, not characters inserted into stored source. A selected
window is blue. Content outside that window stays visible and its character count is reported.

**Future interfaces (experimental only):** a display record contains `id`, `ordinal`,
`document_id`, `start`, `end`, `configuration`, `setup_id`, and `record_kind`.
Optional `parent_id` supports caller-supplied relationship displays. It is not a production schema.

In [ ]:
def highlight_html(text, origin=0, cuts=(), overlaps=(), focus=None):
    end = origin + len(text)
    boundaries = {origin, end}
    cut_set = {c for c in cuts if origin <= c <= end}
    boundaries.update(cut_set)
    for lo, hi in list(overlaps) + ([focus] if focus is not None else []):
        if lo < end and hi > origin:
            boundaries.update((max(origin, lo), min(end, hi)))
    ordered = sorted(boundaries)
    output = []
    for index, lo in enumerate(ordered):
        if lo in cut_set:
            output.append("<b title='Supplied boundary; not source text' style='color:#a00'>│</b>")
        if index + 1 == len(ordered):
            break
        hi = ordered[index + 1]
        fragment = esc(text[lo - origin:hi - origin])
        color = "#ffe39a" if any(a <= lo and hi <= b for a, b in overlaps) else (
            "#dcefff" if focus is not None and focus[0] <= lo and hi <= focus[1] else None)
        output.append(f"<mark style='background:{color};color:#111'>{fragment}</mark>" if color else fragment)
    return "<pre style='white-space:pre-wrap;overflow-wrap:anywhere'>" + "".join(output) + "</pre>"


def check_display_record(record):
    require(record.get("setup_id") == SETUP_ID, "Stale display record: recreate it for the current frozen setup.")
    require(record.get("document_id") in VIEWS, "Display record references an unknown document.")
    require(isinstance(record.get("start"), int) and isinstance(record.get("end"), int), "Offsets must be integers.")
    require(0 <= record["start"] <= record["end"] <= len(VIEWS[record["document_id"]]["text"]),
            "Display span is outside its source document.")
    for key in ("id", "ordinal", "configuration", "record_kind"):
        require(key in record, f"Missing display field: {key}")


def intersection_length(a, b):
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))


def span_diagnostics(record, neighbors=(), reviewed_units=None):
    check_display_record(record)
    did, lo, hi = record["document_id"], record["start"], record["end"]
    for other in neighbors:
        check_display_record(other)
    peers = [other for other in neighbors if other["document_id"] == did and other["id"] != record["id"]]
    peers.append(record)
    require(len({p["id"] for p in peers}) == len(peers), "Duplicate display IDs in neighbor list.")
    peers.sort(key=lambda p: (p["start"], p["ordinal"], p["id"]))
    index = next(i for i, p in enumerate(peers) if p["id"] == record["id"])
    before = peers[index - 1] if index else None
    after = peers[index + 1] if index + 1 < len(peers) else None
    previous = intersection_length((lo, hi), (before["start"], before["end"])) if before else 0
    following = intersection_length((lo, hi), (after["start"], after["end"])) if after else 0
    overlaps = [(max(lo, other["start"]), min(hi, other["end"])) for other in (before, after)
                if other is not None and intersection_length((lo, hi), (other["start"], other["end"]))]
    mapped = source_map(did, lo, hi)
    paragraphs_cut = sorted({r["position"] for r in mapped if r["kind"] == "source"
                             and (r["block_start"] > 0 or r["block_end"] < len(BLOCKS[r["block_id"]]["raw_text"]))})
    split_units, touched_units = [], []
    if reviewed_units is not None:
        for unit in reviewed_units:
            require(unit.get("review_status") == "reviewed" and unit.get("setup_id") == SETUP_ID,
                    "Legal-unit diagnostics require explicit current-setup reviewed units.")
            require(unit["document_id"] in VIEWS and
                    0 <= unit["start"] < unit["end"] <= len(VIEWS[unit["document_id"]]["text"]),
                    "Reviewed unit has an invalid source span.")
            if unit["document_id"] == did and intersection_length((lo, hi), (unit["start"], unit["end"])):
                touched_units.append(unit["id"])
                if unit["start"] < lo < unit["end"] or unit["start"] < hi < unit["end"]:
                    split_units.append(unit["id"])
    return {"actual_previous_overlap": previous, "actual_next_overlap": following,
            "overlap_ranges": overlaps, "paragraph_positions_cut": paragraphs_cut,
            "reviewed_legal_units_cut (supplied ledger only)": split_units if reviewed_units is not None else "unknown — no reviewed unit ledger supplied",
            "combines_reviewed_units (supplied ledger only)": len(touched_units) > 1 if reviewed_units is not None else "unknown — no reviewed unit ledger supplied"}


def span_card(record, sample=None, neighbors=(), reviewed_units=None):
    check_display_record(record)
    did, lo, hi = record["document_id"], record["start"], record["end"]
    text = VIEWS[did]["text"][lo:hi]
    diagnostic = span_diagnostics(record, neighbors, reviewed_units)
    focus = None
    outside = "not applicable"
    if sample is not None:
        require(sample["document_id"] == did, "Sample and display record belong to different documents.")
        focus = (sample["start"], sample["end"])
        outside = len(text) - intersection_length((lo, hi), focus)
    mapped = source_map(did, lo, hi)
    positions = [r["position"] for r in mapped if r["kind"] == "source"]
    fields = {"kind": record["record_kind"], "configuration": record["configuration"],
              "sample": sample["sample_id"] if sample else "not supplied", "ordinal": record["ordinal"],
              "ID": record["id"], "document ID": did,
              "document title": DOCUMENTS[did]["source_titles"]["title"]["raw_text"],
              "characters": len(text), "span [start, end)": [lo, hi], "outside sample characters": outside,
              "beginning": text[:180], "end": text[-180:], **diagnostic}
    context = observed_context(did, positions[0]) if positions else []
    body = table_html([{"field": k, "value": v} for k, v in fields.items() if k != "overlap_ranges"])
    body += "<details><summary>Complete text with display-only boundary/overlap highlights</summary>"
    body += highlight_html(text, lo, (lo, hi), diagnostic["overlap_ranges"], focus) + "</details>"
    body += details_html("Reversible source map", json.dumps(mapped, ensure_ascii=False, indent=2))
    body += details_html("Observed context — not inferred ancestry", json.dumps(context, ensure_ascii=False, indent=2))
    body += "<p>Legal hierarchy remains uncertain; candidate labels are source evidence only.</p>"
    return "<section style='border:1px solid #aaa;padding:10px;min-width:0'>" + body + "</section>"


def comparison_html(sample, panels, reviewed_units=None):
    # panels: {label: [caller-supplied span records]}. No splitting occurs here.
    frozen = {s["sample_id"]: s for s in frozen_samples()}
    require(sample == frozen.get(sample["sample_id"]), "Sample differs from frozen manifest.")
    did = sample["document_id"]
    text = VIEWS[did]["text"][sample["start"]:sample["end"]]
    source = f"<h4>Same source: {esc(sample['sample_id'])}</h4>" + details_html("Full selected source", text)
    columns = []
    for label, records in panels.items():
        for record in records:
            check_display_record(record)
            require(record["document_id"] == did, "Comparison panel mixes documents.")
        shown = [r for r in records if intersection_length((r["start"], r["end"]), (sample["start"], sample["end"]))]
        cards = "".join(span_card(r, sample, records, reviewed_units) for r in shown)
        columns.append(f"<div style='flex:1;min-width:280px'><h4>{esc(label)}</h4>"
                       + (cards or "<p>No strategy results supplied — not implemented.</p>") + "</div>")
    return source + "<div style='display:flex;gap:12px;overflow:auto'>" + "".join(columns) + "</div>"


def relationship_html(parent, children, sample=None, reviewed_units=None):
    # Display only: caller supplies records and links. Never derives a parent or splits it.
    check_display_record(parent)
    require(not parent.get("parent_id"), "This display accepts one explicit parent level only.")
    require(len({c["id"] for c in children}) == len(children), "Duplicate child IDs.")
    rows = []
    for child in children:
        check_display_record(child)
        require(child.get("parent_id") == parent["id"] and child["id"] != parent["id"], "Invalid parent link.")
        require(child["document_id"] == parent["document_id"]
                and parent["start"] <= child["start"] <= child["end"] <= parent["end"],
                "Child is not contained within supplied parent.")
        size = child["end"] - child["start"]
        rows.append({"parent ID": parent["id"], "child ID": child["id"], "child order": child["ordinal"],
                     "parent characters": parent["end"] - parent["start"], "child characters": size,
                     "expansion ratio": round((parent["end"] - parent["start"]) / size, 2) if size else "undefined: empty child"})
    ordered = sorted(children, key=lambda c: (c["start"], c["ordinal"]))
    return table_html(rows) + "<div style='display:flex;gap:12px;overflow:auto'><div style='flex:1;min-width:280px'>" + span_card(parent, sample) + "</div><div style='flex:1;min-width:280px'>" + "".join(
        span_card(c, sample, ordered, reviewed_units) for c in ordered) + "</div></div>"

### 4a. Demonstrate source inspection; leave strategy panels empty

Choose a `SAMPLE_ID` below and rerun this cell to inspect another frozen example. The record
is the exact selected window and is labeled **source selection, not a chunk**. The three
comparison panels remain empty. Parent/child display support is defined but is not called on
corpus data, because no parent-child experiment is authorized yet.

In [ ]:
SAMPLE_ID = "normal_article"
selected_sample = next(s for s in frozen_samples() if s["sample_id"] == SAMPLE_ID)
source_record = {"id": "source-selection:" + fingerprint([SETUP_ID, SAMPLE_ID]), "ordinal": 1,
                 "document_id": selected_sample["document_id"], "start": selected_sample["start"],
                 "end": selected_sample["end"], "setup_id": SETUP_ID,
                 "configuration": "Steps 1–4 source inspection; no strategy parameters",
                 "record_kind": "source selection, not a chunk"}
display(HTML(span_card(source_record, selected_sample)))
display(HTML(comparison_html(selected_sample, {"Fixed-length (future)": [], "Structural (future)": [],
                                              "Parent-child (future)": []})))
display(HTML("<h4>Synthetic renderer fixture — not corpus output</h4><p>Gold: supplied overlap; bars: supplied cuts.</p>"
             + highlight_html("АБ <пример> ВГ", cuts=(3, 10), overlaps=((3, 10),))))

### 4b. Validate setup fidelity and display edge cases

These checks validate source mappings, frozen sample identity, error handling, and display
mechanics—not chunking quality or legal correctness. Small synthetic fixtures cover empty
blocks, newline accounting, Cyrillic offsets, HTML escaping, and caller-supplied parent links.
The fixture relationship contains no corpus text and does not implement a splitting strategy.

In [ ]:
def expect_value_error(action, expected):
    try:
        action()
    except ValueError as exc:
        require(expected in str(exc), f"Unexpected error: {exc}")
    else:
        raise AssertionError(f"Expected ValueError containing {expected!r}")


# Validate malformed-interface guards without mutating the actual artifact.
expect_value_error(lambda: validate_stage1({**corpus, "schema_version": 999}), "schema version")
expect_value_error(lambda: validate_stage1({**corpus, "diagnostics": {**corpus["diagnostics"],
    "stage1_validation": {"result": "FAIL"}}}), "must report PASS")
expect_value_error(lambda: validate_stage1({**corpus, "blocks": corpus["blocks"] + [corpus["blocks"][0]]}), "Duplicate body")
expect_value_error(lambda: validate_stage1({**corpus, "blocks": list(reversed(corpus["blocks"]))}), "increasing")
changed_membership = dict(corpus["blocks"][61], document_id="missing-document")
expect_value_error(lambda: validate_stage1({**corpus, "blocks": corpus["blocks"][:61] + [changed_membership] + corpus["blocks"][62:]}), "membership")

# Full-text accounting is by source identity, not by matching repeated strings.
kept_ids = {bid for v in VIEWS.values() for bid in v["block_ids"]}
removed_ids = {r["block_id"] for r in EXCLUDED_DELIMITERS}
full_text_ids = {b["id"] for b in corpus["blocks"] if b["region"] == "full_text"}
require(not kept_ids.intersection(removed_ids) and kept_ids | removed_ids == full_text_ids, "Full-text coverage mismatch.")
require(all(BLOCKS[bid]["region"] == "full_text" for bid in kept_ids), "Reference/metadata contamination.")
for sample in frozen_samples():
    text = VIEWS[sample["document_id"]]["text"][sample["start"]:sample["end"]]
    require(reconstruct(sample["source_spans"]) == text, "Sample source-map reconstruction failed.")
    require(hashlib.sha256(text.encode("utf-8")).hexdigest() == sample["source_text_sha256"], "Sample text changed.")
    require(len(text) == sample["characters"], "Sample character count changed.")
require([select_sample(spec) for spec in SAMPLE_SPECS] == frozen_samples(), "Sample selection is not reproducible.")
require(canonical(manifest) == FROZEN_MANIFEST_JSON, "Manifest mutated after freeze.")
require(fingerprint({**manifest, "view_policy": "changed"}) != SETUP_ID, "Setup fingerprint ignores configuration.")
expect_value_error(lambda: check_display_record({**source_record, "setup_id": "old-setup"}), "Stale")
expect_value_error(lambda: source_map(source_record["document_id"], -1, 0), "Invalid")
require(source_map(source_record["document_id"], 0, 0) == [], "Empty span map must be empty.")
require(span_diagnostics(source_record)["paragraph_positions_cut"] == [], "Whole-block sample unexpectedly cuts paragraphs.")

# Test display relationships with a separate tiny synthetic namespace, then restore it.
real_views, real_blocks, real_spans, real_ends, real_documents = VIEWS, BLOCKS, BLOCK_SPANS, SEGMENT_ENDS, DOCUMENTS
try:
    fixture_doc = {"document_id": "fixture", "ordinal": 0, "source_titles": {"title": {"raw_text": "Synthetic display fixture"}},
                   "boundary_evidence": {"full_text_marker_block_id": "delimiter"}}
    fixture_texts = [("delimiter", "Пълен текст / Full text"), ("a", "Чл. 2а. <x>"), ("empty", ""), ("b", "§ 1. (Изм.)")]
    fixture_blocks = [{"id": bid, "position": i, "document_id": "fixture", "region": "full_text", "raw_text": text,
                       "source": {"part": "fixture", "path": str(i)},
                       "structural_annotations": {"word_heading_level": None, "legal_marker_candidates": []}}
                      for i, (bid, text) in enumerate(fixture_texts)]
    fixture_view = build_document_view(fixture_doc, fixture_blocks)
    require(fixture_view["text"] == "Чл. 2а. <x>\n\n§ 1. (Изм.)", "Empty source block or separator lost.")
    require(sum(seg["kind"] == "source" and seg["start"] == seg["end"] for seg in fixture_view["segments"]) == 1,
            "Zero-length block evidence lost.")
    VIEWS = {"fixture": fixture_view}
    BLOCKS = {b["id"]: b for b in fixture_blocks}
    DOCUMENTS = {"fixture": fixture_doc}
    SEGMENT_ENDS = {"fixture": [seg["end"] for seg in fixture_view["segments"]]}
    # Exhaustive short-span reconstruction catches edge offsets adjacent to empty blocks/newlines.
    for lo in range(len(fixture_view["text"]) + 1):
        for hi in range(lo, len(fixture_view["text"]) + 1):
            require(reconstruct(source_map("fixture", lo, hi)) == fixture_view["text"][lo:hi], "Partial map mismatch.")
    base = {"document_id": "fixture", "setup_id": SETUP_ID, "configuration": "synthetic display test",
            "record_kind": "supplied fixture span"}
    parent = {**base, "id": "p", "ordinal": 0, "start": 0, "end": len(fixture_view["text"])}
    left = {**base, "id": "a", "ordinal": 1, "start": 0, "end": 8, "parent_id": "p"}
    right = {**base, "id": "b", "ordinal": 2, "start": 6, "end": 12, "parent_id": "p"}
    require(span_diagnostics(left, [left, right])["actual_next_overlap"] == 2, "Overlap calculation failed.")
    require(span_diagnostics(right, [left, right])["actual_previous_overlap"] == 2, "Previous overlap failed.")
    require(span_diagnostics(left)["paragraph_positions_cut"] == [1], "Partial paragraph not flagged.")
    reviewed = [{"id": "fixture-unit", "document_id": "fixture", "start": 0, "end": 10,
                 "setup_id": SETUP_ID, "review_status": "reviewed"}]
    require(span_diagnostics(left, reviewed_units=reviewed)["reviewed_legal_units_cut (supplied ledger only)"] == ["fixture-unit"], "Reviewed-unit split not flagged.")
    rendered = relationship_html(parent, [left, right])
    require("expansion ratio" in rendered and "&lt;x&gt;" in rendered and "<x>" not in rendered, "Unsafe or incomplete display.")
    expect_value_error(lambda: relationship_html(parent, [{**left, "parent_id": "wrong"}]), "parent link")
    expect_value_error(lambda: relationship_html(parent, [{**left, "end": parent["end"] + 1}]), "outside")
finally:
    VIEWS, BLOCKS, BLOCK_SPANS, SEGMENT_ENDS, DOCUMENTS = real_views, real_blocks, real_spans, real_ends, real_documents

require(fingerprint(corpus) == corpus_fingerprint_before, "In-memory Stage 1 representation mutated.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == artifact_sha256, "Stage 1 artifact changed on disk.")
print("PASS: source fidelity, region exclusion, complete-document coverage, sample freezing, error guards, and display fixtures.")
print("Setup validation complete. The approved Step 5 fixed-length baseline follows below.")

### 4c. Preserve the approved frozen setup

The manifest is a setup artifact, not a chunk corpus. Its filename identifies the exact source,
selection, and view policy. Existing different setups are retained; identical reruns leave the
same manifest unchanged. No corpus text is written into tracked files.

**Review checklist:**
- Inspect the complete document inventory and unresolved Stage 1 warnings.
- Check the sample table, both boundary windows, and full text for every selected example.
- Confirm additional versus transitional § contexts, separate EU titles, and the long Article/Annex endpoints.
- Inspect the unclassified heading and retained navigation rather than assuming clean legal structure.
- Confirm whether the median sampling rule yields a useful set; any selection change requires a new fingerprint.
- Verify the source map distinguishes original text from added newlines and leaves empty-block evidence intact.
- Confirm the display marks missing legal review as unknown and all strategy panels are empty.

**Steps 1–4 were approved for proceeding to Step 5.** Preserve the original manifest bytes,
including its historical pending-review label; record the approval separately in Step 5.
Structural, Parent-Child, Oversized-Unit, and Hybrid experiments remain out of scope.

In [ ]:
OUTPUT_DIR = ROOT / "data/processed/02_chunking_experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / f"sample_manifest.{SETUP_ID}.json"
manifest_bytes = (FROZEN_MANIFEST_JSON + "\n").encode("utf-8")
if manifest_path.exists():
    require(manifest_path.read_bytes() == manifest_bytes, "Existing content-addressed manifest differs; investigate before overwriting.")
else:
    manifest_path.write_bytes(manifest_bytes)
require(json.loads(manifest_path.read_text(encoding="utf-8")) == json.loads(FROZEN_MANIFEST_JSON), "Manifest round-trip failed.")
print("Frozen manifest:", manifest_path.relative_to(ROOT))
print("Setup fingerprint:", SETUP_ID)
print("Approved Steps 1–4 setup verified. Proceeding only to Step 5 below.")

## Step 5 — Fixed-Length Baseline

**What:** Compare lengths **1,000 / 2,000 / 4,000 Unicode characters** and overlap **0% / 10% / 20%**.
**Why:** Establish a simple control with predictable maximum size before evaluating other strategies.
**How:** Start at document offset zero; take `[start, min(start + length, document_length))`;
advance by `length − overlap`; stop as soon as a window reaches the document end. No trimming,
word snapping, whitespace preference, paragraph handling, legal markers, or sample-based resets.
The newline separators established in Step 2 count as characters; no new separators are added.

**Design choices:** Character counts are not token counts. Windows never cross documents. Only
the final window can be short; an already covered tail is never emitted again. Overlap repeats
source positions, not new text. The splitter receives just text and numeric settings.

**Inspect:** Word/provision cuts, detached labels, conditions versus exceptions, duplicated history,
and unrelated rules inside one window. Smaller windows may improve focus but fragment context;
larger windows may preserve more context while combining provisions. Overlap costs duplication
and cannot guarantee completeness. Observations favor a configuration if passages remain
understandable with tolerable duplication; repeated loss of important qualifications argues
against it. No retrieval quality or legal correctness is established by these measurements.

### 5a. Load the approved manifest and lock the experiment inputs

Read the **existing** content-addressed setup manifest and verify both its identity and every
complete-document text hash. The experiment corpus here is the **four frozen complete documents**,
not all 21 Stage 1 documents and not the 12 short sample excerpts. The immutable setup's historical
pending-review field is preserved; the approval below records the user's subsequent decision.

In [ ]:
APPROVED_SETUP_ID = "9282d70912c3c61977502ae88016482ba790e93f324da78806d12d3bfdda7043"
APPROVED_MANIFEST_PATH = OUTPUT_DIR / f"sample_manifest.{APPROVED_SETUP_ID}.json"
require(APPROVED_MANIFEST_PATH.is_file(), "Approved Stage 2 manifest is missing; restore the frozen setup first.")
approved_manifest_bytes = APPROVED_MANIFEST_PATH.read_bytes()
APPROVED_MANIFEST = json.loads(approved_manifest_bytes)
require(fingerprint(APPROVED_MANIFEST) == APPROVED_SETUP_ID == SETUP_ID,
        "Setup changed since approval. Review the new sample manifest before running Step 5.")
require(canonical(APPROVED_MANIFEST) == FROZEN_MANIFEST_JSON, "In-memory setup differs from approved manifest.")
require(APPROVED_MANIFEST["artifact_sha256"] == artifact_sha256, "Manifest points to another Stage 1 artifact.")
SETUP_APPROVAL = {"setup_id": APPROVED_SETUP_ID, "scope": "Steps 1–4 approved; Step 5 only authorized",
                  "legal_validity": "not assessed"}
BASELINE_DOCUMENT_IDS = tuple(d["document_id"] for d in APPROVED_MANIFEST["complete_document_inputs"])
BASELINE_SAMPLES = APPROVED_MANIFEST["samples"]
for entry in APPROVED_MANIFEST["complete_document_inputs"]:
    view = VIEWS[entry["document_id"]]
    require(len(view["text"]) == entry["characters"] and view["block_ids"] == entry["block_ids"],
            "Complete-document input differs from the frozen manifest.")
    require(hashlib.sha256(view["text"].encode("utf-8")).hexdigest() == entry["text_sha256"],
            "Complete-document text hash mismatch.")

CONFIGURATIONS = [{"id": f"chars-{length}-overlap-{percent}pct", "length": length,
                   "overlap_percent": percent, "overlap_characters": length * percent // 100,
                   "stride": length - length * percent // 100}
                  for length in (1000, 2000, 4000) for percent in (0, 10, 20)]
CONFIGURATIONS_JSON = canonical(CONFIGURATIONS)
CONFIG_BY_ID = {c["id"]: c for c in CONFIGURATIONS}
BASELINE_SPEC = {"setup_id": SETUP_ID, "algorithm": "strict_character_windows_v1",
                 "configurations": CONFIGURATIONS,
                 "boundary_policy": "start at document zero; stop on first window reaching document end"}
BASELINE_ID = fingerprint(BASELINE_SPEC)
print("Fixed-length experiment:", BASELINE_ID)
print("Frozen scope:", len(BASELINE_DOCUMENT_IDS), "complete documents;", len(BASELINE_SAMPLES), "inspection samples")
show_table(CONFIGURATIONS)
show_table(APPROVED_MANIFEST["complete_document_inputs"], ["ordinal", "characters", "text_sha256"])

### 5b. Strict splitter and independent boundary checks

Read the function before running it: it has no access to structural annotations or sample
positions. The fixtures deliberately include spaces, line breaks, Cyrillic labels, and unbroken
text. Expected offsets verify that none of these affect its arithmetic. We also check empty
input, exact-size input, a final short chunk, a one-character tail, and large valid overlap.

In [ ]:
def fixed_length_windows(text, length, overlap):
    require(isinstance(text, str), "Text must be a string.")
    require(type(length) is int and length > 0, "Length must be a positive integer.")
    require(type(overlap) is int and 0 <= overlap < length, "Overlap must be an integer in [0, length).")
    windows = []
    start = 0
    while start < len(text):
        end = min(start + length, len(text))
        windows.append({"start": start, "end": end, "text": text[start:end]})
        if end == len(text):
            break
        start += length - overlap
    return windows


def offsets(text, length, overlap):
    return [(w["start"], w["end"]) for w in fixed_length_windows(text, length, overlap)]


require(offsets("", 4, 1) == [], "Empty text must produce no chunks.")
require(offsets("АБВГ", 4, 1) == [(0, 4)], "Exact-size input must not create a duplicate tail.")
require(offsets("АБВГД", 4, 0) == [(0, 4), (4, 5)], "One-character final chunk lost.")
require(offsets("АБВГД", 4, 1) == [(0, 4), (3, 5)], "Overlapped final chunk is wrong.")
require(offsets("АБВГД", 4, 3) == [(0, 4), (1, 5)], "High overlap failed to terminate.")
require(offsets("Чл. 1.\nдългадума § 2.", 7, 2) == [(0, 7), (5, 12), (10, 17), (15, 21)],
        "Whitespace/labels changed strict offsets.")
fixture = "Чл. 2а.\n(Изм. - ДВ) дългадума"
require(offsets(fixture, 9, 2) == offsets("Ж" * len(fixture), 9, 2), "Content affected fixed boundaries.")
require(fixed_length_windows("дългадума", 4, 0)[0]["text"] == "дълг", "Splitter adjusted a word boundary.")
for length, overlap in ((0, 0), (-1, 0), (4, -1), (4, 4), (4, 5), (True, 0), (4, 1.5)):
    expect_value_error(lambda length=length, overlap=overlap: fixed_length_windows("x", length, overlap),
                       "Length" if type(length) is not int or length <= 0 else "Overlap")
print("PASS: strict splitter fixtures; boundaries depend only on character count and numeric settings.")

### 5c. Generate all nine baselines on complete-document inputs

Chunk IDs include setup, configuration, document, and offsets. The text is an exact source-view
slice. Source maps are resolved by the existing helper, not inferred from matching repeated
strings. Validation reconstructs every chunk and each entire document after removing only the
known overlap. These checks establish fidelity, not a preference among configurations.

In [ ]:
def baseline_chunk_id(config_id, document_id, start, end):
    return "fixed:" + fingerprint([BASELINE_ID, config_id, document_id, start, end])


def build_fixed_baseline():
    result = {}
    for config in CONFIGURATIONS:
        result[config["id"]] = {}
        for did in BASELINE_DOCUMENT_IDS:
            windows = fixed_length_windows(VIEWS[did]["text"], config["length"], config["overlap_characters"])
            result[config["id"]][did] = [{**window,
                "id": baseline_chunk_id(config["id"], did, window["start"], window["end"]),
                "ordinal": ordinal, "document_id": did, "configuration": config["id"],
                "setup_id": SETUP_ID, "baseline_id": BASELINE_ID,
                "record_kind": "strict fixed-length character chunk"}
                for ordinal, window in enumerate(windows, 1)]
    return result


FIXED_RESULTS = build_fixed_baseline()


def validate_fixed_baseline(results):
    require(set(results) == set(CONFIG_BY_ID), "Missing/extra baseline configurations.")
    all_ids = set()
    for config in CONFIGURATIONS:
        require(set(results[config["id"]]) == set(BASELINE_DOCUMENT_IDS), "Wrong document population.")
        for did, records in results[config["id"]].items():
            source = VIEWS[did]["text"]
            expected_count = 0 if not source else 1 + max(0, math.ceil((len(source) - config["length"]) / config["stride"]))
            require(len(records) == expected_count, "Unexpected chunk count or redundant tail.")
            rebuilt, covered_end = [], 0
            for index, record in enumerate(records):
                check_display_record(record)
                lo, hi = record["start"], record["end"]
                require(record["ordinal"] == index + 1 and record["document_id"] == did
                        and record["configuration"] == config["id"] and record["baseline_id"] == BASELINE_ID,
                        "Chunk identity/configuration mismatch.")
                require(lo == index * config["stride"] and hi == min(lo + config["length"], len(source)),
                        "A boundary deviated from strict arithmetic.")
                require(record["id"] == baseline_chunk_id(config["id"], did, lo, hi), "Unstable chunk ID.")
                require(record["id"] not in all_ids, "Duplicate chunk ID.")
                all_ids.add(record["id"])
                require(record["text"] == source[lo:hi] == reconstruct(source_map(did, lo, hi)),
                        "Chunk text or source mapping differs from source.")
                require(lo <= covered_end and hi > covered_end, "Gap or wholly redundant chunk.")
                if index:
                    previous = records[index - 1]
                    require(previous["end"] - lo == config["overlap_characters"], "Actual overlap differs from configured overlap.")
                    require(previous["text"][-config["overlap_characters"]:] == record["text"][:config["overlap_characters"]]
                            if config["overlap_characters"] else True, "Overlap text does not match.")
                rebuilt.append(record["text"][covered_end - lo:])
                covered_end = hi
            require(covered_end == len(source) and "".join(rebuilt) == source, "Complete-document reconstruction failed.")
    return len(all_ids)


fixed_chunk_count = validate_fixed_baseline(FIXED_RESULTS)
FIXED_RESULTS_FINGERPRINT = fingerprint(FIXED_RESULTS)
print("PASS:", fixed_chunk_count, "configuration-specific chunks; complete coverage and exact mappings for all nine runs.")

### 5d. Evaluation-only provision ledger — never used by the splitter

To make legal-boundary diagnostics concrete, annotate **nine explicit source spans inside the
approved samples**: six Articles (including the two short neighbors), two § provisions, and one
Annex. These are the already inspected source ranges, not automatically discovered units or a
structural chunking result. Chapter/Section labels remain contextual evidence; no hierarchy is built.

`reviewed` here means source-boundary inspection for this small ledger, **not legal validation**.
Every diagnostic is limited to this ledger. A chunk touching two entries combines at least parts
of two annotated units; another field counts complete units contained. Zero matches outside these
ranges means **unassessed**, not proof of legal coherence. Paragraph cuts are checked independently
against original block spans throughout the four-document experiment corpus.

In [ ]:
# Explicit source annotations used ONLY after all fixed-length chunks have been produced.
PROVISION_SPECS = [
    ("normal_article", "Article", "Чл. 1.", 61, 62),
    ("short_neighbors", "Article", "Чл. 2.", 63, 63),
    ("short_neighbors", "Article", "Чл. 2а.", 64, 65),
    ("nested_structure", "Article", "Чл. 10.", 114, 116),
    ("long_article", "Article", "Чл. 7.", 2779, 2810),
    ("additional_provision", "§ provision", "§ 1.", 2052, 2055),
    ("transitional_provision", "§ provision", "§ 6.", 2159, 2166),
    ("annex", "Annex", "Приложение № 1", 2848, 2893),
    ("eu_article", "Article", "Член 1", 25783, 25785),
]
SAMPLE_BY_ID = {s["sample_id"]: s for s in BASELINE_SAMPLES}
REVIEWED_PROVISIONS = []
for sample_id, kind, label, first, last in PROVISION_SPECS:
    sample = SAMPLE_BY_ID[sample_id]
    did, lo, _ = BLOCK_SPANS[BY_POSITION[first]["id"]]
    end_did, _, hi = BLOCK_SPANS[BY_POSITION[last]["id"]]
    require(did == end_did == sample["document_id"] and sample["start"] <= lo < hi <= sample["end"],
            "Diagnostic provision falls outside its approved sample.")
    require(BY_POSITION[first]["raw_text"].startswith(label), "Diagnostic label changed.")
    REVIEWED_PROVISIONS.append({"id": f"source-unit:{first}:{last}", "kind": kind, "label": label,
        "sample_id": sample_id, "document_id": did, "start": lo, "end": hi, "setup_id": SETUP_ID,
        "first_position": first, "last_position": last, "review_status": "reviewed",
        "review_scope": "Explicit source-boundary inspection only; not legal validity or a corpus-wide segmentation"})
show_table(REVIEWED_PROVISIONS, ["id", "kind", "label", "sample_id", "first_position", "last_position"])

### 5e. Corpus-level and sample-level diagnostics

The top table covers the same **875,477 source-view characters** in the four frozen documents
for every configuration. Sizes include added newlines from Step 2. `extra_character_pct` is
`100 × (sum(chunk sizes) − unique covered characters) / unique covered characters`; it measures
repeated positions from overlap, not repeated phrases already in the source.

`paragraphs_cut` counts distinct original paragraphs containing an internal chunk start or end.
`ledger_units_cut` similarly counts annotated provisions; overlapping windows can cut a unit
while another window contains it completely, so complete containment is reported separately.
`chunks_combining_ledger_units` counts chunks touching at least parts of two ledger entries.
These are descriptive measurements, not scores or evidence of retrieval quality.

Each sample has its own nine-row table. `sample_coverage_pct` must be 100. Sample duplication
counts repeated source positions **inside the window**; `outside_sample_characters` measures
unique context outside it in intersecting chunks. Sample tables overlap each other and must
not be summed into corpus totals. All configurations retain source order; none is ranked.

In [ ]:
def interval_union_length(intervals):
    total, end = 0, None
    for lo, hi in sorted(intervals):
        require(lo <= hi, "Reversed interval.")
        if end is None or lo > end:
            total += hi - lo
        else:
            total += max(0, hi - end)
        end = hi if end is None else max(end, hi)
    return total


def paragraph_cuts(record):
    mapped = source_map(record["document_id"], record["start"], record["end"])
    return sorted({r["block_id"] for r in mapped if r["kind"] == "source"
                   and (r["block_start"] > 0 or r["block_end"] < len(BLOCKS[r["block_id"]]["raw_text"]))})


def legal_diagnostic(record):
    lo, hi, did = record["start"], record["end"], record["document_id"]
    touched = [u for u in REVIEWED_PROVISIONS if u["document_id"] == did
               and intersection_length((lo, hi), (u["start"], u["end"]))]
    return {"touched": [u["id"] for u in touched],
            "cut": [u["id"] for u in touched if u["start"] < lo < u["end"] or u["start"] < hi < u["end"]],
            "contained": [u["id"] for u in touched if lo <= u["start"] and u["end"] <= hi]}


CHUNK_DIAGNOSTICS = {r["id"]: {"paragraphs_cut": paragraph_cuts(r), **legal_diagnostic(r)}
                     for docs in FIXED_RESULTS.values() for records in docs.values() for r in records}
CORPUS_DIAGNOSTICS, DOCUMENT_DIAGNOSTICS, SAMPLE_DIAGNOSTICS = [], [], []
for config in CONFIGURATIONS:
    config_id = config["id"]
    records = [r for did in BASELINE_DOCUMENT_IDS for r in FIXED_RESULTS[config_id][did]]
    sizes = [r["end"] - r["start"] for r in records]
    unique = sum(len(VIEWS[did]["text"]) for did in BASELINE_DOCUMENT_IDS)
    actual_overlaps = [a["end"] - b["start"] for did in BASELINE_DOCUMENT_IDS
                       for a, b in zip(FIXED_RESULTS[config_id][did], FIXED_RESULTS[config_id][did][1:])]
    corpus_row = {"configuration": config_id, "documents": len(BASELINE_DOCUMENT_IDS), "chunks": len(records),
        "size_min": min(sizes), "size_median": length_summary(sizes)["median_p50"], "size_max": max(sizes),
        "coverage_pct": 100.0, "extra_characters": sum(sizes) - unique,
        "extra_character_pct": round(100 * (sum(sizes) - unique) / unique, 2),
        "actual_overlap_min_max": [min(actual_overlaps), max(actual_overlaps)],
        "paragraphs_cut": len({bid for r in records for bid in CHUNK_DIAGNOSTICS[r["id"]]["paragraphs_cut"]}),
        "ledger_units_cut_of_9": len({uid for r in records for uid in CHUNK_DIAGNOSTICS[r["id"]]["cut"]}),
        "ledger_units_contained_somewhere_of_9": len({uid for r in records for uid in CHUNK_DIAGNOSTICS[r["id"]]["contained"]}),
        "chunks_combining_ledger_units": sum(len(CHUNK_DIAGNOSTICS[r["id"]]["touched"]) > 1 for r in records)}
    # Coverage is independently measured, not inferred from the table's fixed population.
    covered = sum(interval_union_length((r["start"], r["end"]) for r in FIXED_RESULTS[config_id][did])
                  for did in BASELINE_DOCUMENT_IDS)
    require(covered == unique, "Corpus coverage is incomplete.")
    corpus_row["coverage_pct"] = round(100 * covered / unique, 2)
    CORPUS_DIAGNOSTICS.append(corpus_row)
    for did in BASELINE_DOCUMENT_IDS:
        doc_records = FIXED_RESULTS[config_id][did]
        DOCUMENT_DIAGNOSTICS.append({"configuration": config_id, "document": DOCUMENTS[did]["ordinal"],
            "characters": len(VIEWS[did]["text"]), "chunks": len(doc_records),
            "last_chunk_characters": doc_records[-1]["end"] - doc_records[-1]["start"]})
    for sample in BASELINE_SAMPLES:
        lo, hi = sample["start"], sample["end"]
        hits = [r for r in FIXED_RESULTS[config_id][sample["document_id"]]
                if intersection_length((lo, hi), (r["start"], r["end"]))]
        clipped = [(max(lo, r["start"]), min(hi, r["end"])) for r in hits]
        covered = interval_union_length(clipped)
        require(covered == hi - lo, "Sample is not fully covered.")
        sample_units = {u["id"] for u in REVIEWED_PROVISIONS if u["document_id"] == sample["document_id"]
                        and intersection_length((lo, hi), (u["start"], u["end"]))}
        units_cut = {uid for r in hits for uid in CHUNK_DIAGNOSTICS[r["id"]]["cut"]} & sample_units
        units_contained = {uid for r in hits for uid in CHUNK_DIAGNOSTICS[r["id"]]["contained"]} & sample_units
        cut_paragraphs = {bid for r in hits for bid in CHUNK_DIAGNOSTICS[r["id"]]["paragraphs_cut"]}
        # Only count boundaries falling inside this sample, not outside context in the same chunk.
        internal_cuts = {cut for r in hits for cut in (r["start"], r["end"]) if lo < cut < hi}
        cut_paragraphs = {bid for bid in cut_paragraphs if any(BLOCK_SPANS[bid][1] < c < BLOCK_SPANS[bid][2]
                                                            for c in internal_cuts)}
        SAMPLE_DIAGNOSTICS.append({"sample": sample["sample_id"], "configuration": config_id,
            "intersecting_chunks": len(hits), "sample_coverage_pct": round(100 * covered / (hi - lo), 2),
            "extra_character_pct_in_sample": round(100 * (sum(b - a for a, b in clipped) - covered) / covered, 2),
            "whole_sample_in_one_chunk": any(r["start"] <= lo and hi <= r["end"] for r in hits),
            "internal_cut_points": len(internal_cuts), "paragraphs_cut_inside_sample": len(cut_paragraphs),
            "ledger_units_cut": len(units_cut) if sample_units else "unassessed",
            "ledger_units_contained": len(units_contained) if sample_units else "unassessed",
            "hit_chunks_combining_ledger_units": sum(len(CHUNK_DIAGNOSTICS[r["id"]]["touched"]) > 1 for r in hits),
            "outside_sample_characters": interval_union_length((r["start"], r["end"]) for r in hits) - covered})

print("Experiment corpus characters:", sum(len(VIEWS[did]["text"]) for did in BASELINE_DOCUMENT_IDS))
show_table(CORPUS_DIAGNOSTICS)
display(HTML("<details><summary>Per-document counts and tail sizes (36 rows)</summary>" + table_html(DOCUMENT_DIAGNOSTICS) + "</details>"))
for sample in BASELINE_SAMPLES:
    rows = [r for r in SAMPLE_DIAGNOSTICS if r["sample"] == sample["sample_id"]]
    display(HTML(f"<details><summary>{esc(sample['sample_id'])}: all nine configurations</summary>"
                 + table_html(rows) + "</details>"))

### 5f. Inspect real chunks, cut points, overlap, and source maps

The gallery shows every frozen sample under **2,000 characters / 10% overlap** solely as a
consistent display setting, not a recommendation. For a long sample, first/middle/last chunks
are shown; the exact number omitted is stated, and the cut map still covers **all** intersecting
chunks. Set `max_cards=None` to inspect every chunk. No text inside a shown chunk is truncated.

Each card uses the Step 4 helper: beginning/end previews, expandable complete text, reversible
block offsets, actual previous/next overlap, paragraph cuts, and provision-ledger diagnostics.
The neighboring chunks used to measure overlap come from the complete document, even when a
neighbor falls outside the selected sample. Bars mark supplied cuts; gold marks overlap; blue
marks the selected source window. Source panels and cut-context tables make mid-word cuts visible.

Change `INSPECT_SAMPLE` below to compare another sample across all nine settings. A fixed-length
chunk may extend outside a sample: this is intentional and its full text remains available.

In [ ]:
def ensure_baseline_current():
    require(SETUP_ID == APPROVED_SETUP_ID and fingerprint(json.loads(FROZEN_MANIFEST_JSON)) == APPROVED_SETUP_ID,
            "Frozen setup changed; rerun and review setup before using these results.")
    require(canonical(CONFIGURATIONS) == CONFIGURATIONS_JSON and fingerprint(BASELINE_SPEC) == BASELINE_ID,
            "Configuration changed; rerun Step 5 generation and diagnostics.")


def show_fixed_sample(sample_id, config_id, max_cards=3):
    ensure_baseline_current()
    require(config_id in CONFIG_BY_ID and sample_id in SAMPLE_BY_ID, "Unknown sample/configuration.")
    require(max_cards is None or type(max_cards) is int and max_cards > 0, "max_cards must be positive or None.")
    sample = SAMPLE_BY_ID[sample_id]
    lo, hi, did = sample["start"], sample["end"], sample["document_id"]
    records = FIXED_RESULTS[config_id][did]
    hits = [r for r in records if intersection_length((lo, hi), (r["start"], r["end"]))]
    cuts = sorted({c for r in hits for c in (r["start"], r["end"]) if lo <= c <= hi})
    overlap_ranges = [(b["start"], a["end"]) for a, b in zip(records, records[1:])
                      if b["start"] < a["end"] and b["start"] < hi and a["end"] > lo]
    cut_rows = []
    for cut in cuts:
        paragraph = next((BLOCKS[r["block_id"]]["position"] for r in source_map(did, max(0, cut - 1),
                         min(len(VIEWS[did]["text"]), cut + 1)) if r["kind"] == "source"
                         and BLOCK_SPANS[r["block_id"]][1] < cut < BLOCK_SPANS[r["block_id"]][2]), None)
        source = VIEWS[did]["text"]
        cut_rows.append({"offset": cut, "left_context": source[max(0, cut - 45):cut],
            "right_context": source[cut:cut + 45], "paragraph_cut_position": paragraph,
            "inside_alphanumeric_run": 0 < cut < len(source) and source[cut - 1].isalnum() and source[cut].isalnum(),
            "ledger_units_cut": [u["id"] for u in REVIEWED_PROVISIONS if u["document_id"] == did and u["start"] < cut < u["end"]]})
    chosen = hits
    if max_cards is not None and len(hits) > max_cards:
        indices = [0] if max_cards == 1 else sorted({round(i * (len(hits) - 1) / (max_cards - 1)) for i in range(max_cards)})
        chosen = [hits[i] for i in indices]
    body = f"<p>{len(hits)} intersecting chunks; showing {len(chosen)}; {len(hits)-len(chosen)} omitted. Use max_cards=None for all.</p>"
    body += "<details><summary>Complete sample with ALL intersecting chunk cuts/overlap</summary>"
    body += highlight_html(VIEWS[did]["text"][lo:hi], lo, cuts, overlap_ranges, (lo, hi)) + "</details>"
    body += "<details><summary>Exact cut offsets and source context</summary>" + table_html(cut_rows) + "</details>"
    for record in chosen:
        require(record["baseline_id"] == BASELINE_ID and record["text"] == VIEWS[did]["text"][record["start"]:record["end"]],
                "Stale chunk record.")
        index = record["ordinal"] - 1
        neighbors = records[max(0, index - 1):index + 2]
        body += span_card(record, sample, neighbors, REVIEWED_PROVISIONS)
        legal = CHUNK_DIAGNOSTICS[record["id"]]
        body += details_html("Evaluation ledger: units touched / cut / fully contained (not exhaustive)",
                             json.dumps(legal, ensure_ascii=False, indent=2))
    display(HTML(f"<details><summary>{esc(sample_id)} — {esc(config_id)}</summary>{body}</details>"))


GALLERY_CONFIGURATION = "chars-2000-overlap-10pct"  # Display convenience only; not selected as winner.
for sample in BASELINE_SAMPLES:
    show_fixed_sample(sample["sample_id"], GALLERY_CONFIGURATION, max_cards=3)

In [ ]:
# All nine settings on the SAME source sample. Change this ID and rerun this cell to inspect another.
INSPECT_SAMPLE = "short_neighbors"
for config in CONFIGURATIONS:
    show_fixed_sample(INSPECT_SAMPLE, config["id"], max_cards=3)
# To inspect every chunk of the long Article in a chosen setting:
# show_fixed_sample("long_article", "chars-1000-overlap-20pct", max_cards=None)

### 5g. Concrete boundary witnesses and interpretation

The following witnesses select the **first source-order matching chunk** for each configuration,
not the best or worst configuration. They point to actual sample intersections where a paragraph
is cut, an annotated provision is cut, or multiple annotated units are combined. `No witness`
would mean only that this small reviewed set did not expose one.

Inspect whether a cut separates a condition from its exception, whether overlap restores the
necessary context, whether two combined provisions are related, and how much publication
history is duplicated. Numerical containment does not establish that all cross-referenced legal
context is present. The long Article necessarily exceeds each tested window length; that is an
observation about this baseline, not authorization to implement an oversized-unit fallback.

In [ ]:
WITNESSES = []
for config in CONFIGURATIONS:
    population = [r for did in BASELINE_DOCUMENT_IDS for r in FIXED_RESULTS[config["id"]][did]
                  if any(s["document_id"] == did and intersection_length((s["start"], s["end"]), (r["start"], r["end"]))
                         for s in BASELINE_SAMPLES)]
    for phenomenon in ("paragraph cut", "provision cut", "multiple provisions combined"):
        def matches(record):
            d = CHUNK_DIAGNOSTICS[record["id"]]
            return bool(d["paragraphs_cut"]) if phenomenon == "paragraph cut" else (
                bool(d["cut"]) if phenomenon == "provision cut" else len(d["touched"]) > 1)
        witness = next((r for r in population if matches(r)), None)
        WITNESSES.append({"configuration": config["id"], "phenomenon": phenomenon,
            "chunk_id": witness["id"] if witness else None,
            "document": DOCUMENTS[witness["document_id"]]["ordinal"] if witness else None,
            "ordinal": witness["ordinal"] if witness else None,
            "sample": next(s["sample_id"] for s in BASELINE_SAMPLES if witness and s["document_id"] == witness["document_id"]
                           and intersection_length((s["start"], s["end"]), (witness["start"], witness["end"]))) if witness else None,
            "evidence": CHUNK_DIAGNOSTICS[witness["id"]] if witness else "No witness in annotated sample intersections"})
show_table(WITNESSES, ["configuration", "phenomenon", "document", "ordinal", "sample", "chunk_id"])
# Two complete evidence cards: source-order examples, not recommendations.
for phenomenon in ("provision cut", "multiple provisions combined"):
    witness = next((w for w in WITNESSES if w["phenomenon"] == phenomenon and w["chunk_id"]), None)
    if witness:
        did = DOC_BY_ORDINAL[witness["document"]]["document_id"]
        records = FIXED_RESULTS[witness["configuration"]][did]
        record = records[witness["ordinal"] - 1]
        display(HTML(f"<h4>Witness: {esc(phenomenon)}</h4>" + span_card(record, SAMPLE_BY_ID[witness["sample"]],
                     records[max(0, witness["ordinal"] - 2):witness["ordinal"] + 1], REVIEWED_PROVISIONS)))
        show_json("Witness diagnostics (source ledger only)", witness["evidence"])

print("Observed across the nine runs:")
print("- Every configuration covers the same frozen document text completely.")
print("- Actual adjacent overlaps match each requested character overlap exactly.")
print("- Chunks with multiple ledger units:", [r["chunks_combining_ledger_units"] for r in CORPUS_DIAGNOSTICS])
print("- Distinct ledger units cut (out of nine):", [r["ledger_units_cut_of_9"] for r in CORPUS_DIAGNOSTICS])
print("- The long Article is", SAMPLE_BY_ID["long_article"]["characters"],
      "characters; none of the tested lengths can contain it in one chunk.")
print("No configuration has been selected. Meaning preservation requires inspecting the actual text.")

### 5h. Final fidelity checks, reproducible report, and approved checkpoint

Validate metrics independently on small interval fixtures, compare source maps and shared
helper diagnostics, and regenerate the baseline to check stable text/IDs. Save a separate
content-addressed report containing scope, settings, ledger, metrics, witness IDs, and result
fingerprint. Full chunk text stays available in `FIXED_RESULTS` and the notebook displays; the
report is a reproducibility/diagnostic artifact, not a production chunk schema.

**Step 5 is approved.** Its nine configurations and report remain unchanged for comparison.
Proceed only to Step 6 below. Parent-Child, Oversized-Unit, Hybrid, embeddings, retrieval, and RAG remain out of scope.

In [ ]:
ensure_baseline_current()
require(interval_union_length([]) == 0 and interval_union_length([(0, 4), (2, 6), (6, 7)]) == 7,
        "Overlap union calculation failed.")
require(interval_union_length([(0, 10), (2, 3), (12, 14)]) == 12, "Contained/disjoint interval union failed.")
require(len(CORPUS_DIAGNOSTICS) == 9 and len(SAMPLE_DIAGNOSTICS) == 9 * len(BASELINE_SAMPLES),
        "Incomplete diagnostics grid.")
require(all(row["extra_characters"] == 0 for row in CORPUS_DIAGNOSTICS if "overlap-0pct" in row["configuration"]),
        "Zero-overlap baseline unexpectedly duplicates characters.")
require(all(row["sample_coverage_pct"] == 100 for row in SAMPLE_DIAGNOSTICS), "Sample coverage incomplete.")
require(all(row["ledger_units_contained"] == 0 for row in SAMPLE_DIAGNOSTICS if row["sample"] == "long_article"),
        "Oversized Article reported as fully contained.")
for witness in WITNESSES:
    if not witness["chunk_id"]:
        continue
    records = FIXED_RESULTS[witness["configuration"]][DOC_BY_ORDINAL[witness["document"]]["document_id"]]
    record = records[witness["ordinal"] - 1]
    shared = span_diagnostics(record, records[max(0, witness["ordinal"] - 2):witness["ordinal"] + 1], REVIEWED_PROVISIONS)
    local = CHUNK_DIAGNOSTICS[record["id"]]
    require(shared["reviewed_legal_units_cut (supplied ledger only)"] == local["cut"], "Display/report legal cuts disagree.")
    require(shared["combines_reviewed_units (supplied ledger only)"] == (len(local["touched"]) > 1), "Combined-unit diagnostics disagree.")
    require(shared["paragraph_positions_cut"] == sorted(BLOCKS[bid]["position"] for bid in local["paragraphs_cut"]),
            "Display/report paragraph cuts disagree.")
require(fingerprint(build_fixed_baseline()) == FIXED_RESULTS_FINGERPRINT == fingerprint(FIXED_RESULTS),
        "Baseline rerun changed IDs/text, or stored results were mutated.")
require(APPROVED_MANIFEST_PATH.read_bytes() == approved_manifest_bytes, "Approved sample manifest changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == artifact_sha256
        and fingerprint(corpus) == corpus_fingerprint_before, "Stage 1 source changed.")

FIXED_REPORT = {"report_version": 1, "scope": "Step 5 only; four frozen complete-document inputs",
    "setup_approval": SETUP_APPROVAL, "baseline_spec": BASELINE_SPEC, "baseline_id": BASELINE_ID,
    "source_artifact_sha256": artifact_sha256, "result_fingerprint": FIXED_RESULTS_FINGERPRINT,
    "configuration_specific_chunk_count": fixed_chunk_count, "diagnostic_ledger": REVIEWED_PROVISIONS,
    "corpus_diagnostics": CORPUS_DIAGNOSTICS, "document_diagnostics": DOCUMENT_DIAGNOSTICS,
    "sample_diagnostics": SAMPLE_DIAGNOSTICS, "witnesses": WITNESSES,
    "validation": "PASS: strict arithmetic, exact reconstruction, complete coverage, overlap, IDs, metrics, immutable inputs",
    "decision": "No winner selected; waiting for Step 5 review"}
report_path = OUTPUT_DIR / f"fixed_length_report.{fingerprint(FIXED_REPORT)}.json"
report_bytes = (canonical(FIXED_REPORT) + "\n").encode("utf-8")
if report_path.exists():
    require(report_path.read_bytes() == report_bytes, "Existing content-addressed report differs.")
else:
    report_path.write_bytes(report_bytes)
require(json.loads(report_path.read_text(encoding="utf-8")) == FIXED_REPORT, "Report round-trip failed.")
print("PASS: all Step 5 checks. Report:", report_path.relative_to(ROOT))
print("Approved Step 5 baseline verified. Step 6 follows; no winning configuration selected.")

## Step 6 — Document-Aware Structural Baseline (uncapped)

**What:** Partition the same four frozen document views at supported source boundaries, without
any length cap, overlap, packing, or fallback. **Why:** Observe natural provision lengths and
boundary ambiguity before introducing size constraints. **How:** First propose and display a
source-linked ledger; only then generate disjoint, provisional structural units.

This is a **flat baseline, not a legal tree**. Articles, § provisions, and Annexes are kept as
units up to the next applicable evidenced boundary. Part/Chapter/Section labels and reviewed
separate titles become **heading-context units**, not overlapping copies of entire Parts or
Chapters. That avoids guessing ancestry from Word heading ranks or counting the same text twice.
Text between a heading/title and a later provision remains an explicit residual unit.

Unclassified headings form explicit unclassified spans; their Word style suggests a transition
but does not assign a legal role. Text before the first boundary, document tails with reviewed
transitions, and marker-free documents remain residual/unclassified. No text is removed.

**Trade-offs / inspect:** Provision integrity may improve while sizes vary widely. Very short
heading units may be poor standalone context; long Articles/Annexes remain long. A lower split
count on nine inspected provisions does not establish retrieval quality or correctness across
the corpus. Favor this strategy if complete rules and qualifications stay together; question it
where ambiguity, context-only fragments, or very large units dominate. No strategy is selected.

### 6a. Lock inputs and document the conservative policy

Use the approved manifest and unchanged Step 5 result fingerprint. Read-only comparison uses
existing fixed-length outputs; no new fixed-length parameters are introduced.

Boundary decisions, in order:
1. Apply explicit, source-pinned overrides and reviewed label/title pairings.
2. Accept one leading Stage 1 legal-marker candidate with a supported rule as **provisional**.
3. Keep non-leading/inline citations and amendment-reference phrases in the current text; log
   suspicious candidates instead of promoting them to legal units.
4. Within an Annex, only another supported Annex or an explicit reviewed transition ends it.
   Internal markers/headings are logged as uncertain evidence, not made into children.
5. Outside Annexes, conflicting/unsupported candidates and unclassified headings start an
   **unclassified** span. Word heading level never determines legal rank or parentage.
6. Preserve all residual text and attach each added inter-block newline to the unit on its left.

Every unit carries source-position identity. Repeated §/Annex labels are not deduplicated.
The nearest observed scope-transition block is recorded only as flat context evidence, never
as a parent relationship. Context resets per document and at explicit scope transitions.

In [ ]:
import re

ensure_baseline_current()
require(fingerprint(FIXED_RESULTS) == FIXED_RESULTS_FINGERPRINT, "Approved fixed-length results changed.")
require(fingerprint(APPROVED_MANIFEST) == APPROVED_SETUP_ID, "Approved sample manifest changed.")
STEP5_REPORT_BYTES = report_path.read_bytes()
STEP5_RESULT_FINGERPRINT = FIXED_RESULTS_FINGERPRINT

# All listed adjacent pairs were inspected in this pinned source snapshot. Other title-like
# text is not automatically promoted to a title. These are positions, not heading ranks.
REVIEWED_TITLE_ANCHORS = [
    57, 59, 95, 110, 112, 132, 194, 240, 260, 267, 293, 326, 359, 407, 414,
    451, 453, 480, 506, 541, 583, 598, 600, 606, 638, 646, 652, 654, 661,
    663, 677, 701, 775, 777, 795, 803, 926, 980, 982, 1067, 1181, 1313,
    1315, 1317, 1342, 1443, 1472, 1489, 1543, 1545, 1598, 1600, 1613,
    1640, 1661, 1670, 1674, 1688, 1690, 1713, 1731, 1734, 1736, 1795,
    1879, 1933, 1944, 2740, 2777, 25781, 25783, 25786, 25817, 25850,
    25852, 25876, 25922, 25941, 25952, 25955, 25957, 25962, 25985, 25997,
    26012, 26014, 26023, 26026, 26032, 26034, 26038, 26054, 26070,
]
TITLE_PAIRS = {BY_POSITION[pos]["id"]: BY_POSITION[pos + 1]["id"] for pos in REVIEWED_TITLE_ANCHORS}
# Manual source-boundary decisions, not source edits or verified legal interpretations.
OVERRIDE_SPECS = [
    (2051, "Допълнителни разпоредби", "boundary", "unclassified", "Sample heading remains unclassified; do not invent its legal rank."),
    (2158, "Преходни разпоредби", "boundary", "unclassified", "Observed transition into transitional provisions, distinct from additional provisions."),
    (26047, "Приложение II се изменя, както следва:", "retain", None,
     "Amendment instruction within Article 17, not a standalone Annex; next Article starts at 26054."),
    (2246, "-------------------------", "boundary", "residual", "Separator and adoption statement between provisions and the next amendment heading."),
    (2722, "Релевантни актове от Европейското законодателство", "boundary", "residual", "Related-act listing after the final § provision; preserve as residual text."),
    (26075, "Съставено в Брюксел", "boundary", "residual", "Signature, footnotes and publication tail after Article 19; retain without guessing footnote relationships."),
]
for sample_id, kind, label, first, last in PROVISION_SPECS:
    source_kind = {"Article": "article", "§ provision": "paragraph_sign", "Annex": "annex"}[kind]
    OVERRIDE_SPECS.append((first, label, "boundary", source_kind,
                           f"Explicit source-start review for frozen sample {sample_id}; endpoint checked after generation."))
STRUCTURAL_OVERRIDES = {}
for pos, prefix, action, kind, reason in OVERRIDE_SPECS:
    b = BY_POSITION[pos]
    require(b["raw_text"].startswith(prefix) and b["id"] in BLOCK_SPANS, "Reviewed override source differs.")
    STRUCTURAL_OVERRIDES[b["id"]] = {"action": action, "kind": kind, "reason": reason,
        "expected_text_sha256": hashlib.sha256(b["raw_text"].encode("utf-8")).hexdigest(), "position": pos}

STRUCTURAL_SPEC = {"algorithm": "conservative_flat_structural_v1", "setup_id": SETUP_ID,
    "source_artifact_sha256": artifact_sha256, "document_ids": list(BASELINE_DOCUMENT_IDS),
    "size_limit": None, "overlap": 0, "title_pairs": TITLE_PAIRS, "overrides": STRUCTURAL_OVERRIDES,
    "heading_semantics": "Part/Chapter/Section heading-context spans, not entire descendant subtrees",
    "annex_policy": "retain internal uncertain markers until next Annex or explicit reviewed transition",
    "separator_policy": "preceding unit owns inter-block newline",
    "approval": "Steps 1–5 approved; Step 6 only authorized"}
STRUCTURAL_ID = fingerprint(STRUCTURAL_SPEC)
STRUCTURAL_CONFIGURATION = "structural-uncapped-provisional"
print("Structural experiment:", STRUCTURAL_ID)
print("Population:", len(BASELINE_DOCUMENT_IDS), "frozen complete documents; size limit: NONE")
show_table([{"position": spec["position"], "action": spec["action"], "role": spec["kind"], "reason": spec["reason"]}
            for spec in STRUCTURAL_OVERRIDES.values()])
show_json("Reviewed title pairs (exact source IDs; no inferred hierarchy)", [
    {"label_position": BLOCKS[a]["position"], "label": BLOCKS[a]["raw_text"],
     "title_position": BLOCKS[b]["position"], "title": BLOCKS[b]["raw_text"]} for a, b in TITLE_PAIRS.items()])

### 6b. Propose boundary decisions — do not generate chunks yet

The planner below produces a **ledger**, not text chunks. It records accepted boundaries,
attached titles, rejected false candidates, unresolved internal Annex markers, and residual
starts. A leading label supports a provisional boundary; it does not prove legal semantics.
Repeated labels receive an explicit warning even when both boundaries are retained.

A concrete false detection is the Stage 1 Annex candidate at body position 26047, inside EU
Article 17. Another subtle case is Chapter V's separate title at 26033: although it has a Word
heading style, the inspected title pairing keeps it with the Chapter label at 26032. Conversely,
the similarly worded heading at 2051 stays unclassified. Neither decision follows heading rank.

In [ ]:
STRUCTURAL_KINDS = {"part", "chapter", "section", "article", "paragraph_sign", "annex"}
CONTEXT_KINDS = {"part", "chapter", "section"}
SUPPORTED_RULES = {"leading_label_and_heading", "leading_numbered_label"}
AMENDMENT_REFERENCE = re.compile(r"^се\s+(?:изменя|изменят|допълва|допълват|отменя|отменят)\b", re.IGNORECASE)


def propose_structural_boundaries(view, block_lookup, overrides=None, title_pairs=None):
    overrides, title_pairs = overrides or {}, title_pairs or {}
    ids = view["block_ids"]
    blocks = [block_lookup[bid] for bid in ids]
    index_of = {bid: i for i, bid in enumerate(ids)}
    starts = {s["block_id"]: s["start"] for s in view["segments"] if s["kind"] == "source"}
    attached_titles, residual_after_header = {}, set()
    for anchor, title in title_pairs.items():
        if anchor not in index_of:
            continue
        require(title in index_of and index_of[title] == index_of[anchor] + 1, "Title must immediately follow its label in the same document.")
        a, t = block_lookup[anchor], block_lookup[title]
        candidates = a["structural_annotations"]["legal_marker_candidates"]
        require(len(candidates) == 1 and candidates[0]["kind"] in CONTEXT_KINDS | {"article"}
                and not t["structural_annotations"]["legal_marker_candidates"] and t["raw_text"],
                "Title pairing has contradictory source evidence.")
        attached_titles[title] = anchor
        if candidates[0]["kind"] in CONTEXT_KINDS and index_of[title] + 1 < len(ids):
            residual_after_header.add(index_of[title] + 1)
    ledger, annex_active, scope_id, seen_labels = [], False, None, set()
    for i, b in enumerate(blocks):
        annotations = b["structural_annotations"]
        candidates = annotations["legal_marker_candidates"]
        heading = annotations["word_heading_level"]
        override = overrides.get(b["id"])
        if override:
            require(hashlib.sha256(b["raw_text"].encode("utf-8")).hexdigest() == override["expected_text_sha256"],
                    "Override text changed; re-inspect source before using it.")
        kind, boundary, decision, reason, uncertainty = None, False, None, None, []
        if override:
            boundary = override["action"] == "boundary"
            kind = override["kind"] if boundary else None
            decision = "reviewed_boundary" if boundary else "reviewed_false_candidate_retained"
            reason = override["reason"]
        elif b["id"] in attached_titles:
            decision, reason = "reviewed_title_attached", "Adjacent title retained with inspected label; Word heading rank ignored."
        elif len(candidates) > 1:
            boundary, kind = not annex_active, "unclassified" if not annex_active else None
            decision, reason = "conflicting_candidates", "Multiple candidate roles: no role selected."
            uncertainty.append("Conflicting markers; retain uncertainty.")
        elif candidates:
            c = candidates[0]
            lo, hi = c["character_span"]
            supported = c["kind"] in STRUCTURAL_KINDS and c.get("rule") in SUPPORTED_RULES
            leading = lo == 0 and 0 < hi <= len(b["raw_text"]) and b["raw_text"][lo:hi] == c["matched_label"]
            tail = b["raw_text"][hi:].lstrip(" .:–—-") if leading else ""
            if not leading:
                decision, reason = "nonleading_candidate_retained", "Inline/non-leading evidence does not establish a boundary."
                uncertainty.append("Candidate is not a verified leading label.")
            elif AMENDMENT_REFERENCE.match(tail):
                decision, reason = "possible_reference_retained", "Leading label is followed by an amendment instruction; not promoted to a standalone provision."
                uncertainty.append("Reference interpretation is provisional unless explicitly overridden.")
            elif annex_active and c["kind"] != "annex":
                decision, reason = "uncertain_marker_inside_annex", "Do not manufacture Annex descendants or infer an Annex exit from this marker."
                uncertainty.append("Internal Annex role remains unresolved; source block retained in Annex.")
            elif supported:
                boundary, kind, decision = True, c["kind"], "candidate_boundary"
                reason = "Single leading Stage 1 marker and recorded matching rule; provisional legal boundary."
                uncertainty.append("Unreviewed legal-boundary interpretation outside explicitly checked samples.")
            else:
                boundary, kind = not annex_active, "unclassified" if not annex_active else None
                decision, reason = "unsupported_candidate", "Unsupported marker rule/role; preserve without selecting a legal role."
                uncertainty.append("Unclassified marker evidence.")
        elif heading is not None:
            boundary, kind = not annex_active, "unclassified" if not annex_active else None
            decision = "unclassified_heading_boundary" if boundary else "uncertain_heading_inside_annex"
            reason = "Heading style supplies a possible transition only; it never supplies legal hierarchy."
            uncertainty.append("Legal role and scope are unclassified.")
        elif i in residual_after_header:
            boundary, kind, decision = True, "residual", "residual_after_heading"
            reason = "Text after the reviewed label/title and before another evidenced legal boundary is residual."
        elif i == 0:
            boundary, kind, decision = True, "residual", "document_start_residual"
            reason = "Initial full-text content or marker-free document; no legal role inferred."
        if decision is None:
            continue
        if i == 0 and not boundary:
            boundary, kind = True, "unclassified"
            uncertainty.append("Document must remain represented despite unresolved initial evidence.")
        label = candidates[0]["matched_label"] if len(candidates) == 1 else b["raw_text"][:160]
        if boundary and kind in STRUCTURAL_KINDS:
            key = (kind, label)
            if key in seen_labels:
                uncertainty.append("Repeated label in this document; source-position identity is required.")
            seen_labels.add(key)
        if boundary:
            # Only explicit/legal transitions change this flat scope hint. It is not a parent link.
            if kind in CONTEXT_KINDS | {"annex", "unclassified", "residual"}:
                scope_id = b["id"]
            annex_active = kind == "annex"
        ledger.append({"id": f"boundary:{view['document_id']}:{b['position']}", "document_id": view["document_id"],
            "block_id": b["id"], "position": b["position"], "block_index": i, "offset": starts[b["id"]],
            "source": b["source"], "source_text": b["raw_text"], "candidate_evidence": candidates,
            "word_heading_level_evidence_only": heading, "label": label, "creates_boundary": boundary,
            "proposed_kind": kind, "decision": decision, "reason": reason, "uncertainty": uncertainty,
            "scope_evidence_block_id_not_parent": scope_id,
            "attached_to_label_block_id": attached_titles.get(b["id"]),
            "reviewed_title_block_id": title_pairs.get(b["id"])})
    return ledger


BOUNDARY_LEDGERS = {did: propose_structural_boundaries(VIEWS[did], BLOCKS, STRUCTURAL_OVERRIDES, TITLE_PAIRS)
                    for did in BASELINE_DOCUMENT_IDS}
BOUNDARY_LEDGER_FINGERPRINT = fingerprint(BOUNDARY_LEDGERS)
print("Proposed ledger only — structural chunks have not been generated yet.")
show_table([{"document": DOCUMENTS[did]["ordinal"], "evidence_rows": len(rows),
             "proposed_boundaries": sum(r["creates_boundary"] for r in rows),
             "uncertain_rows": sum(bool(r["uncertainty"]) for r in rows)} for did, rows in BOUNDARY_LEDGERS.items()])
LEDGER_COLUMNS = ["position", "label", "creates_boundary", "proposed_kind", "decision", "reason", "uncertainty"]
for sample in BASELINE_SAMPLES:
    rows = [r for r in BOUNDARY_LEDGERS[sample["document_id"]]
            if sample["first_position"] - 2 <= r["position"] <= sample["last_position_inclusive"] + 2]
    display(HTML(f"<details><summary>Boundary proposals around {esc(sample['sample_id'])}</summary>"
                 + table_html(rows, LEDGER_COLUMNS) + "</details>"))
for did, rows in BOUNDARY_LEDGERS.items():
    display(HTML(f"<details><summary>Complete proposed ledger: Document {DOCUMENTS[did]['ordinal']}</summary>"
                 + table_html(rows, LEDGER_COLUMNS) + "</details>"))
show_json("Complete evidence-linked ledger, including retained/rejected candidates", BOUNDARY_LEDGERS)

### 6c. Inspect uncertain/rejected evidence before accepting the partition

Expand these records and their unchanged source neighbors. The repeated Annex 3 at 3458 includes
a different form/version note: it stays a distinct source occurrence, without reconciling legal
versions or deleting text. The false Annex at 26047 is retained inside Article 17. Unclassified
headings delimit spans but are not assigned legal ancestry. All remaining accepted candidate
boundaries are provisional; the diagnostics do not hide that uncertainty.

In [ ]:
exception_rows = [r for rows in BOUNDARY_LEDGERS.values() for r in rows
                  if (not r["creates_boundary"] and r["decision"] != "reviewed_title_attached")
                  or r["proposed_kind"] == "unclassified"
                  or any("Repeated label" in warning for warning in r["uncertainty"])]
show_table(exception_rows, LEDGER_COLUMNS, limit=15)
show_json("All exceptional/uncertain boundary records (not just the preview)", exception_rows)
for position in (2051, 2158, 3458, 26033, 26047, 2722, 26075):
    display(HTML(f"<h4>Source evidence around position {position}</h4>"))
    show_table([{"position": q, "text": BY_POSITION[q]["raw_text"],
                 "markers": [c["matched_label"] for c in BY_POSITION[q]["structural_annotations"]["legal_marker_candidates"]]}
                for q in range(position - 1, position + 2)])
require(fingerprint(BOUNDARY_LEDGERS) == BOUNDARY_LEDGER_FINGERPRINT, "Ledger changed after display; rerun ledger review output.")

### 6d. Generate uncapped structural units from the displayed ledger

Each accepted boundary begins a unit; the next accepted boundary ends it. This is a disjoint
partition of the document view, with no overlap, no size-based cuts, no merging by size, and no
parent/child links. Source-block membership includes empty blocks. A trailing added newline may
make a unit one character longer than a sample that ends at the preceding paragraph's last
character; it is explicit separator ownership, not changed source text.

**Inspect:** Unit kind, exact start/end, ledger decisions, full source text and maps. Heading
units are labeled context-only; residual/unclassified units retain their uncertainty. A very
large result is deliberately left intact.

In [ ]:
def assemble_structural_units(view, ledger, experiment_id):
    boundaries = [r for r in ledger if r["creates_boundary"]]
    require(not view["block_ids"] or boundaries and boundaries[0]["block_index"] == 0, "Ledger omits document start.")
    require([r["block_index"] for r in boundaries] == sorted({r["block_index"] for r in boundaries}),
            "Duplicate or unordered structural boundaries.")
    records = []
    for index, boundary in enumerate(boundaries):
        next_boundary = boundaries[index + 1] if index + 1 < len(boundaries) else None
        lo, hi = boundary["offset"], next_boundary["offset"] if next_boundary else len(view["text"])
        end_index = next_boundary["block_index"] if next_boundary else len(view["block_ids"])
        evidence = [r for r in ledger if boundary["block_index"] <= r["block_index"] < end_index]
        records.append({"id": "structural:" + fingerprint([experiment_id, view["document_id"], boundary["block_id"], lo, hi]),
            "ordinal": index + 1, "document_id": view["document_id"], "start": lo, "end": hi,
            "text": view["text"][lo:hi], "block_ids": view["block_ids"][boundary["block_index"]:end_index],
            "configuration": STRUCTURAL_CONFIGURATION, "setup_id": SETUP_ID, "structural_id": experiment_id,
            "record_kind": "provisional structural " + boundary["proposed_kind"], "unit_kind": boundary["proposed_kind"],
            "span_semantics": "heading/context only; not a complete descendant subtree" if boundary["proposed_kind"] in CONTEXT_KINDS
                              else "uncapped flat source span; interpretation provisional",
            "opening_ledger_id": boundary["id"], "closing_ledger_id": next_boundary["id"] if next_boundary else "document_end",
            "evidence_ids": [r["id"] for r in evidence],
            "uncertainty": sorted({warning for r in evidence for warning in r["uncertainty"]}),
            "scope_evidence_block_id_not_parent": boundary["scope_evidence_block_id_not_parent"]})
    return records


STRUCTURAL_RESULTS = {did: assemble_structural_units(VIEWS[did], BOUNDARY_LEDGERS[did], STRUCTURAL_ID)
                      for did in BASELINE_DOCUMENT_IDS}
STRUCTURAL_RESULTS_FINGERPRINT = fingerprint(STRUCTURAL_RESULTS)
STRUCTURAL_GENERATION_CONTEXT = {"structural_id": STRUCTURAL_ID,
    "ledger_fingerprint": BOUNDARY_LEDGER_FINGERPRINT, "setup_id": SETUP_ID}


def validate_structural_results():
    seen = set()
    for did, records in STRUCTURAL_RESULTS.items():
        view = VIEWS[did]
        require("".join(r["text"] for r in records) == view["text"], "Structural text reconstruction failed.")
        require([bid for r in records for bid in r["block_ids"]] == view["block_ids"], "Source block coverage/order changed.")
        end = 0
        for r in records:
            check_display_record(r)
            require(r["start"] == end and r["end"] >= r["start"], "Gap/overlap in structural partition.")
            require(r["id"] not in seen and r["structural_id"] == STRUCTURAL_ID, "Invalid structural identity.")
            seen.add(r["id"])
            require(r["text"] == reconstruct(source_map(did, r["start"], r["end"])), "Structural source map mismatch.")
            require(not paragraph_cuts(r), "A structural boundary cut an original paragraph.")
            require("parent_id" not in r, "Parent-child relationship introduced unexpectedly.")
            end = r["end"]
        require(end == len(view["text"]), "Document tail missing.")
    return len(seen)


structural_count = validate_structural_results()
print("PASS:", structural_count, "uncapped units; exact text/block coverage; no overlaps or paragraph cuts.")
show_table([{"document": DOCUMENTS[did]["ordinal"], "units": len(records),
             "characters": sum(r["end"] - r["start"] for r in records)} for did, records in STRUCTURAL_RESULTS.items()])

### 6e. Natural sizes, residuals, Annexes, § provisions, and uncertainty

Length summaries use the same nearest-rank definition as earlier cells. Bottom/top deciles
(`p10`/`p90`) are descriptive flags, not splitting thresholds. Counts above **1,000 / 2,000 /
4,000 characters** only compare natural units to the Step 5 window lengths; nothing is split
or truncated. Read distributions by kind: short heading contexts and long residual preambles
should not be confused with Article lengths.

§ and Annex tables retain document/source positions and nearest observed scope evidence.
Repeated labels stay separate. The uncertainty table distinguishes unreviewed accepted legal
candidates from unresolved/rejected boundary interpretations; zero text loss does not mean all
legal boundaries are correct.

In [ ]:
structural_records = [r for did in BASELINE_DOCUMENT_IDS for r in STRUCTURAL_RESULTS[did]]
STRUCTURAL_SIZE_ROWS = [{"kind": kind, **length_summary(r["end"] - r["start"] for r in structural_records if r["unit_kind"] == kind)}
                        for kind in sorted({r["unit_kind"] for r in structural_records})]
show_table(STRUCTURAL_SIZE_ROWS)
show_table([{"all_units": structural_count, **length_summary(r["end"] - r["start"] for r in structural_records),
             "coverage_pct": 100, "overlap_characters": 0}])
ordered_sizes = sorted(r["end"] - r["start"] for r in structural_records)
p10, p90 = [ordered_sizes[max(0, math.ceil(q * len(ordered_sizes)) - 1)] for q in (.1, .9)]


def structural_summary(record):
    return {"id": record["id"], "document": DOCUMENTS[record["document_id"]]["ordinal"],
            "kind": record["unit_kind"], "characters": record["end"] - record["start"],
            "first_position": BLOCKS[record["block_ids"][0]]["position"],
            "last_position": BLOCKS[record["block_ids"][-1]]["position"],
            "scope_evidence_position_not_parent": BLOCKS[record["scope_evidence_block_id_not_parent"]]["position"]
                 if record["scope_evidence_block_id_not_parent"] else None,
            "preview": record["text"][:150], "uncertainty": record["uncertainty"]}


SMALL_STRUCTURAL = [structural_summary(r) for r in sorted(structural_records, key=lambda r: (r["end"] - r["start"], r["id"]))
                    if r["end"] - r["start"] <= p10]
LARGE_STRUCTURAL = [structural_summary(r) for r in sorted(structural_records, key=lambda r: (-(r["end"] - r["start"]), r["id"]))
                    if r["end"] - r["start"] >= p90]
print("Empirical flags only — small <= p10:", p10, "; large >= p90:", p90, "characters. No fallback is applied.")
show_table(SMALL_STRUCTURAL, limit=8)
show_table(LARGE_STRUCTURAL, limit=8)
SIZE_COMPARISONS = [{"Step_5_length_for_comparison_only": length,
    "units_exceeding_length": sum(r["end"] - r["start"] > length for r in structural_records),
    "percent": round(100 * sum(r["end"] - r["start"] > length for r in structural_records) / structural_count, 2)}
    for length in (1000, 2000, 4000)]
show_table(SIZE_COMPARISONS)
for kind in ("annex", "paragraph_sign", "unclassified", "residual"):
    rows = [structural_summary(r) for r in structural_records if r["unit_kind"] == kind]
    display(HTML(f"<details><summary>All {esc(kind)} units ({len(rows)}): sizes and scope evidence</summary>"
                 + table_html(rows) + "</details>"))
BOUNDARY_DECISION_COUNTS = Counter(r["decision"] for rows in BOUNDARY_LEDGERS.values() for r in rows)
show_table([{"decision": key, "count": count} for key, count in sorted(BOUNDARY_DECISION_COUNTS.items())])
show_json("All small/large flags (full lists, not just previews)", {"small": SMALL_STRUCTURAL, "large": LARGE_STRUCTURAL})

### 6f. Same-source visual comparison and sample diagnostics

Compare each frozen sample to the existing **2,000-character / 10% overlap** display setting;
this is the same gallery setting used in Step 5, not a recommended configuration. Structural
units can extend outside sample windows, and their full text remains expandable. At most three
representative cards per side are shown by default (first/middle/last); omitted counts are explicit.
Use `max_cards=None` to display all units, or change `fixed_config` to any of the nine settings.

Bars show structural cut positions within the original sample. Structural cards use the existing
source-map, overlap, paragraph/provision-cut helpers; a second metadata panel explains unit kind,
opening/closing ledger evidence, and unresolved interpretation. A Chapter/Section sample should
show multiple heading contexts and a whole Article, without claiming a nested legal tree.

In [ ]:
def ensure_structural_current():
    ensure_baseline_current()
    require(STRUCTURAL_GENERATION_CONTEXT == {"structural_id": STRUCTURAL_ID,
            "ledger_fingerprint": BOUNDARY_LEDGER_FINGERPRINT, "setup_id": SETUP_ID},
            "Generation context is stale; regenerate units from the current displayed ledger.")
    require(fingerprint(STRUCTURAL_SPEC) == STRUCTURAL_ID
            and STRUCTURAL_SPEC["overrides"] == STRUCTURAL_OVERRIDES and STRUCTURAL_SPEC["title_pairs"] == TITLE_PAIRS,
            "Structural policy changed; rerun ledger, generation and diagnostics.")
    require(fingerprint(BOUNDARY_LEDGERS) == BOUNDARY_LEDGER_FINGERPRINT, "Ledger changed; re-display it before regenerating units.")
    require(fingerprint(STRUCTURAL_RESULTS) == STRUCTURAL_RESULTS_FINGERPRINT, "Structural results changed; regenerate diagnostics.")


def compare_structural_sample(sample_id, fixed_config=GALLERY_CONFIGURATION, max_cards=3):
    ensure_structural_current()
    require(sample_id in SAMPLE_BY_ID and fixed_config in CONFIG_BY_ID, "Unknown frozen sample/fixed configuration.")
    require(max_cards is None or type(max_cards) is int and max_cards > 0, "max_cards must be positive or None.")
    sample = SAMPLE_BY_ID[sample_id]
    did, lo, hi = sample["document_id"], sample["start"], sample["end"]
    source = VIEWS[did]["text"][lo:hi]
    columns = []
    for label, records in ((fixed_config, FIXED_RESULTS[fixed_config][did]),
                           (STRUCTURAL_CONFIGURATION, STRUCTURAL_RESULTS[did])):
        hits = [r for r in records if intersection_length((lo, hi), (r["start"], r["end"]))]
        cuts = sorted({c for r in hits for c in (r["start"], r["end"]) if lo <= c <= hi})
        overlaps = [(b["start"], a["end"]) for a, b in zip(records, records[1:])
                    if b["start"] < a["end"] and b["start"] < hi and a["end"] > lo]
        chosen = hits
        if max_cards is not None and len(hits) > max_cards:
            indices = [0] if max_cards == 1 else sorted({round(i * (len(hits) - 1) / (max_cards - 1)) for i in range(max_cards)})
            chosen = [hits[i] for i in indices]
        body = f"<h4>{esc(label)}</h4><p>{len(hits)} intersecting units; {len(chosen)} shown; {len(hits)-len(chosen)} omitted.</p>"
        body += "<details><summary>Same complete sample: ALL cut points and overlaps</summary>"
        body += highlight_html(source, lo, cuts, overlaps, (lo, hi)) + "</details>"
        for r in chosen:
            index = r["ordinal"] - 1
            body += span_card(r, sample, records[max(0, index - 1):index + 2], REVIEWED_PROVISIONS)
            if "unit_kind" in r:
                body += details_html("Structural role and exact ledger evidence (not hierarchy)", json.dumps(
                    {k: r[k] for k in ("unit_kind", "span_semantics", "opening_ledger_id", "closing_ledger_id", "evidence_ids", "uncertainty")},
                    ensure_ascii=False, indent=2))
        columns.append("<div style='flex:1;min-width:320px'>" + body + "</div>")
    display(HTML(f"<details><summary>Same source: {esc(sample_id)} — fixed vs uncapped structural</summary>"
                 + "<div style='display:flex;gap:12px;overflow:auto'>" + "".join(columns) + "</div></details>"))


STRUCTURAL_SAMPLE_DIAGNOSTICS = []
for sample in BASELINE_SAMPLES:
    lo, hi, did = sample["start"], sample["end"], sample["document_id"]
    hits = [r for r in STRUCTURAL_RESULTS[did] if intersection_length((lo, hi), (r["start"], r["end"]))]
    covered = interval_union_length((max(lo, r["start"]), min(hi, r["end"])) for r in hits)
    sample_units = {u["id"] for u in REVIEWED_PROVISIONS if u["document_id"] == did and intersection_length((lo, hi), (u["start"], u["end"]))}
    cut_units = {uid for r in hits for uid in legal_diagnostic(r)["cut"]} & sample_units
    contained_units = {uid for r in hits for uid in legal_diagnostic(r)["contained"]} & sample_units
    require(covered == hi - lo, "Structural sample coverage incomplete.")
    STRUCTURAL_SAMPLE_DIAGNOSTICS.append({"sample": sample["sample_id"], "units": len(hits),
        "kinds": sorted({r["unit_kind"] for r in hits}), "coverage_pct": 100,
        "whole_sample_in_one_unit": any(r["start"] <= lo and hi <= r["end"] for r in hits),
        "ledger_units_cut": len(cut_units) if sample_units else "unassessed",
        "ledger_units_contained": len(contained_units) if sample_units else "unassessed",
        "paragraphs_cut": sum(bool(paragraph_cuts(r)) for r in hits),
        "outside_sample_characters": interval_union_length((r["start"], r["end"]) for r in hits) - covered})
show_table(STRUCTURAL_SAMPLE_DIAGNOSTICS)
for sample in BASELINE_SAMPLES:
    compare_structural_sample(sample["sample_id"])
# Optional: inspect every structural/fixed chunk using another already-tested fixed configuration:
# compare_structural_sample("long_article", fixed_config="chars-1000-overlap-20pct", max_cards=None)

### 6g. Corpus comparison and observed counterexamples

The summary compares one structural partition with all nine existing fixed-length baselines,
using exactly the same character population. Boundary preservation is measured only against
original paragraphs and the small reviewed provision ledger; it is not a retrieval score.
The two strategies expose different risks: fixed windows can fragment a rule, while structural
units can be enormous or too small to explain themselves. No weighted winner is computed.

In [ ]:
structural_cut_units = {uid for r in structural_records for uid in legal_diagnostic(r)["cut"]}
structural_contained_units = {uid for r in structural_records for uid in legal_diagnostic(r)["contained"]}
STRUCTURAL_COMPARISON_ROW = {"configuration": STRUCTURAL_CONFIGURATION, "chunks": structural_count,
    "size_min": min(ordered_sizes), "size_median": length_summary(ordered_sizes)["median_p50"], "size_max": max(ordered_sizes),
    "extra_character_pct": 0.0, "paragraphs_cut": 0,
    "ledger_units_cut_of_9": len(structural_cut_units), "ledger_units_contained_somewhere_of_9": len(structural_contained_units),
    "chunks_combining_ledger_units": sum(len(legal_diagnostic(r)["touched"]) > 1 for r in structural_records)}
show_table(CORPUS_DIAGNOSTICS + [STRUCTURAL_COMPARISON_ROW], list(STRUCTURAL_COMPARISON_ROW))
for record in (min(structural_records, key=lambda r: r["end"] - r["start"]),
               max(structural_records, key=lambda r: r["end"] - r["start"])):
    display(HTML("<h4>Observed size extreme — retained without resizing</h4>" + span_card(record, reviewed_units=REVIEWED_PROVISIONS)))
    show_json("Extreme unit interpretation", structural_summary(record))
print("Observed structural result:", structural_count, "units; sizes", min(ordered_sizes), "to", max(ordered_sizes), "characters.")
print("Residual/unclassified units:", sum(r["unit_kind"] in {"residual", "unclassified"} for r in structural_records))
print("Reviewed provision boundaries cut:", len(structural_cut_units), "of", len(REVIEWED_PROVISIONS), "annotated units.")
print("The false Annex marker in EU Article 17 stays in its Article; repeated Annex labels remain separate source occurrences.")
print("All other candidate boundaries remain provisional. Size variation and detached heading context require human review.")
print("No final strategy selected. No parent-child links, size caps, or fallback splitting introduced.")

### 6h. Validate conservative behavior and save a separate report

Synthetic cases test inline citations, conflicting roles, unknown headings, repeated labels,
Cyrillic suffixes, an Annex containing an internal Article-like marker, and a false Annex phrase.
Changing Word heading ranks must not create a different legal structure. Exact source checks
then verify every real unit and the nine inspected provision ranges. The displayed ledger and
unit IDs must reproduce identically, and approved source/manifest/fixed-length results stay unchanged.

In [ ]:
def structural_fixture(items):
    # Synthetic evidence only. 'kind' supplies Stage 1-like candidate annotations for tests.
    blocks = []
    for i, (text, kind, heading) in enumerate(items):
        candidates = []
        if kind:
            label = re.match(r"(?:Чл\.\s*\d+[а-я]?|§\s*\d+[а-я]?|Приложение\s+[IVX]+|Глава\s+\w+)", text).group()
            candidates = [{"kind": kind, "matched_label": label, "character_span": [0, len(label)],
                           "rule": "leading_numbered_label" if kind in {"article", "paragraph_sign"} else "leading_label_and_heading"}]
        blocks.append({"id": f"fixture-{i}", "position": i, "document_id": "structural-fixture", "raw_text": text,
                       "source": {"part": "fixture", "path": str(i)},
                       "structural_annotations": {"word_heading_level": heading, "legal_marker_candidates": candidates}})
    offset, segments = 0, []
    for i, b in enumerate(blocks):
        if i:
            segments.append({"kind": "separator", "start": offset, "end": offset + 1})
            offset += 1
        segments.append({"kind": "source", "block_id": b["id"], "start": offset, "end": offset + len(b["raw_text"])})
        offset += len(b["raw_text"])
    view = {"document_id": "structural-fixture", "text": "\n".join(b["raw_text"] for b in blocks),
            "block_ids": [b["id"] for b in blocks], "segments": segments}
    return view, {b["id"]: b for b in blocks}


fixture_view, fixture_blocks = structural_fixture([
    ("Преамбюл", None, None), ("Глава първа", "chapter", 2), ("ОБЩИ ПОЛОЖЕНИЯ", None, 3),
    ("междинен текст", None, None), ("Чл. 2а. правило", "article", None),
    ("Виж чл. 99 и § 1 вътре в текста.", None, None), ("Неясен раздел", None, 2),
    ("§ 1. текст", "paragraph_sign", None), ("Друга неясна група", None, 3),
    ("§ 1. друг текст", "paragraph_sign", None), ("Приложение I", "annex", 2),
    ("Чл. 1. образец в приложение", "article", None), ("Приложение I", "annex", 2), ("", None, None)])
fixture_pairs = {"fixture-1": "fixture-2"}
fixture_ledger = propose_structural_boundaries(fixture_view, fixture_blocks, title_pairs=fixture_pairs)
fixture_units = assemble_structural_units(fixture_view, fixture_ledger, "fixture")
require("".join(r["text"] for r in fixture_units) == fixture_view["text"], "Fixture text changed.")
require([bid for r in fixture_units for bid in r["block_ids"]] == fixture_view["block_ids"], "Fixture empty block lost.")
require(not any(r["position"] == 5 for r in fixture_ledger), "Inline citation became a boundary.")
require(next(r for r in fixture_ledger if r["position"] == 2)["decision"] == "reviewed_title_attached", "Detached title not retained.")
require(next(r for r in fixture_ledger if r["position"] == 3)["proposed_kind"] == "residual", "Interstitial residual lost.")
require(next(r for r in fixture_ledger if r["position"] == 6)["proposed_kind"] == "unclassified", "Unknown heading assigned a legal role.")
require(not next(r for r in fixture_ledger if r["position"] == 11)["creates_boundary"], "Annex internal Article manufactured as a separate unit.")
require(sum(r["unit_kind"] == "annex" for r in fixture_units) == 2, "Repeated Annex labels collapsed.")
require(any("Repeated label" in w for r in fixture_ledger for w in r["uncertainty"]), "Repeated labels not flagged.")
changed_ranks = json.loads(json.dumps(fixture_blocks))
for b in changed_ranks.values():
    if b["structural_annotations"]["word_heading_level"] is not None:
        b["structural_annotations"]["word_heading_level"] = 6
changed_ledger = propose_structural_boundaries(fixture_view, changed_ranks, title_pairs=fixture_pairs)
require([(r["offset"], r["proposed_kind"], r["creates_boundary"]) for r in changed_ledger] ==
        [(r["offset"], r["proposed_kind"], r["creates_boundary"]) for r in fixture_ledger], "Word heading rank changed legal boundaries.")
false_view, false_blocks = structural_fixture([("Чл. 1. изменя друг акт", "article", None),
    ("Приложение II се изменя, както следва:", "annex", 2), ("Чл. 2. текст", "article", None)])
false_ledger = propose_structural_boundaries(false_view, false_blocks)
require(not false_ledger[1]["creates_boundary"], "Amendment reference became Annex boundary.")
conflicting = json.loads(json.dumps(false_blocks))
conflicting["fixture-2"]["structural_annotations"]["legal_marker_candidates"].append(
    {**conflicting["fixture-2"]["structural_annotations"]["legal_marker_candidates"][0], "kind": "paragraph_sign"})
require(propose_structural_boundaries(false_view, conflicting)[-1]["proposed_kind"] == "unclassified", "Conflicting markers guessed a legal role.")
expect_value_error(lambda: propose_structural_boundaries(fixture_view, fixture_blocks, title_pairs={"fixture-1": "fixture-3"}), "immediately follow")
expect_value_error(lambda: propose_structural_boundaries(false_view, false_blocks,
    overrides={"fixture-1": {"expected_text_sha256": "changed", "action": "retain", "kind": None, "reason": "test"}}), "Override text changed")

# Reject out-of-order notebook execution rather than comparing old units to a new ledger.
correct_generation_context = STRUCTURAL_GENERATION_CONTEXT
try:
    STRUCTURAL_GENERATION_CONTEXT = {**correct_generation_context, "ledger_fingerprint": "stale"}
    expect_value_error(ensure_structural_current, "Generation context is stale")
finally:
    STRUCTURAL_GENERATION_CONTEXT = correct_generation_context

# Real-corpus invariants and explicit sample endpoint checks (separator ownership is allowed).
for did, rows in BOUNDARY_LEDGERS.items():
    require(all(r["scope_evidence_block_id_not_parent"] is None
                or BLOCKS[r["scope_evidence_block_id_not_parent"]]["document_id"] == did for r in rows),
            "Scope evidence leaked across document boundaries.")
require(validate_structural_results() == structural_count, "Structural count changed.")
for unit in REVIEWED_PROVISIONS:
    hits = [r for r in STRUCTURAL_RESULTS[unit["document_id"]]
            if intersection_length((unit["start"], unit["end"]), (r["start"], r["end"]))]
    require(len(hits) == 1 and hits[0]["start"] == unit["start"] and hits[0]["end"] >= unit["end"],
            "Inspected provision was split or merged with preceding content.")
    require(set(hits[0]["text"][unit["end"] - unit["start"]:]) <= {"\n"}, "Inspected endpoint swallowed following content.")
false_annex = next(r for r in BOUNDARY_LEDGERS[DOC_BY_ORDINAL[11]["document_id"]] if r["position"] == 26047)
require(not false_annex["creates_boundary"], "Reviewed false Annex became a boundary.")
article17 = next(r for r in STRUCTURAL_RESULTS[DOC_BY_ORDINAL[11]["document_id"]] if r["block_ids"][0] == BY_POSITION[26038]["id"])
require(BY_POSITION[26047]["id"] in article17["block_ids"] and BY_POSITION[26054]["id"] not in article17["block_ids"],
        "Article 17/18 boundary mishandled.")
require(len(STRUCTURAL_RESULTS[DOC_BY_ORDINAL[16]["document_id"]]) == 1
        and STRUCTURAL_RESULTS[DOC_BY_ORDINAL[16]["document_id"]][0]["unit_kind"] == "residual", "Marker-free document gained a hierarchy.")
require(max(ordered_sizes) > max(c["length"] for c in CONFIGURATIONS), "Uncapped baseline unexpectedly has a fixed-length cap.")
ensure_structural_current()
repeated_ledgers = {did: propose_structural_boundaries(VIEWS[did], BLOCKS, STRUCTURAL_OVERRIDES, TITLE_PAIRS) for did in BASELINE_DOCUMENT_IDS}
require(fingerprint(repeated_ledgers) == BOUNDARY_LEDGER_FINGERPRINT, "Boundary ledger is not reproducible.")
require(fingerprint({did: assemble_structural_units(VIEWS[did], repeated_ledgers[did], STRUCTURAL_ID)
                     for did in BASELINE_DOCUMENT_IDS}) == STRUCTURAL_RESULTS_FINGERPRINT, "Structural IDs/text are not reproducible.")
require(fingerprint(FIXED_RESULTS) == STEP5_RESULT_FINGERPRINT and report_path.read_bytes() == STEP5_REPORT_BYTES,
        "Approved fixed-length baseline/report changed.")
require(APPROVED_MANIFEST_PATH.read_bytes() == approved_manifest_bytes, "Frozen sample manifest changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == artifact_sha256
        and fingerprint(corpus) == corpus_fingerprint_before, "Stage 1 source changed.")
print("PASS: conservative boundary fixtures; full text/block coverage; sample endpoints; false Annex; reproducibility; unchanged inputs and fixed baseline.")

### 6i. Save the structural report and approved checkpoint

The report stores the complete boundary ledger (including rejected/uncertain candidates), unit
source ranges and roles, natural-size summaries, sample diagnostics, and comparison results.
It leaves the frozen sample manifest and approved fixed-length report unchanged.

**Review checklist:**
- Inspect the ledger before relying on the structural units, especially unclassified headings.
- Compare the 12 frozen samples using the actual full-text cards and source maps.
- Inspect Annex 3's repeated labels, the false EU Annex marker, and repeated § labels/scopes.
- Check small heading/context fragments and long provisions/residuals without assuming either is optimal.
- Confirm all remaining provisional interpretations are visible and every source character is retained.

**Step 6 is approved.** Its ledger, units, and report stay unchanged. Proceed only to the
Step 7 relationship experiment below; size-fallback policies, Hybrid, embeddings, retrieval, and RAG remain out of scope.

In [ ]:
STRUCTURAL_REPORT = {"report_version": 1, "scope": "Step 6 only; uncapped provisional structural baseline",
    "structural_spec": STRUCTURAL_SPEC, "structural_id": STRUCTURAL_ID,
    "approved_fixed_result_fingerprint": STEP5_RESULT_FINGERPRINT,
    "boundary_ledgers": BOUNDARY_LEDGERS, "boundary_ledger_fingerprint": BOUNDARY_LEDGER_FINGERPRINT,
    "structural_result_fingerprint": STRUCTURAL_RESULTS_FINGERPRINT,
    "units": [{k: v for k, v in r.items() if k != "text"} for r in structural_records],
    "size_by_kind": STRUCTURAL_SIZE_ROWS, "comparison_lengths_only": SIZE_COMPARISONS,
    "sample_diagnostics": STRUCTURAL_SAMPLE_DIAGNOSTICS,
    "corpus_comparison": CORPUS_DIAGNOSTICS + [STRUCTURAL_COMPARISON_ROW],
    "validation": "PASS: source fidelity, coverage, no overlap/cap, conservative fixtures, sample boundaries, deterministic ledger/IDs",
    "decision": "No final strategy selected; waiting for Step 6 review"}
structural_report_path = OUTPUT_DIR / f"structural_report.{fingerprint(STRUCTURAL_REPORT)}.json"
structural_report_bytes = (canonical(STRUCTURAL_REPORT) + "\n").encode("utf-8")
if structural_report_path.exists():
    require(structural_report_path.read_bytes() == structural_report_bytes, "Existing structural report differs.")
else:
    structural_report_path.write_bytes(structural_report_bytes)
require(json.loads(structural_report_path.read_text(encoding="utf-8")) == STRUCTURAL_REPORT, "Structural report round-trip failed.")
print("Structural report:", structural_report_path.relative_to(ROOT))
print("Approved Step 6 baseline verified. Step 7 follows; no final strategy selected.")

## Step 7 — Parent-Child Alternatives (relationships only)

**What:** Compare Article/§, Section/Chapter, whole Annex, and reviewed unclassified/residual
parents using the existing source windows and Step 6 evidence. **Why:** A focused child may omit
a condition, definition, or exception, while its parent may restore it—or add unrelated material.
**How:** Preserve complete parent spans, create explicit links to ordered child spans, and inspect
context expansion. Selecting a child here means an **exact dictionary lookup by ID and experiment
set**, not search or retrieval.

A parent is the complete contextual source span. A child is a source span fully contained within
it. In a future system, child text could be embedded/searched and a child hit could resolve to
its parent for LLM context. This notebook implements none of those downstream operations and
makes no claim that a complete parent fits an eventual model budget.

**Controls / trade-offs:** Reuse all nine fixed-length settings unchanged. For larger parents,
also retain contained structural units whole as an uncapped control. An additional controlled
variant windows **every** contained unit with the existing splitter; it has no size-trigger rule
or adaptive fallback. Short units naturally yield one child. This isolates parent context while
allowing identical Article child spans to appear under an Article, Section, or Chapter parent.

Inspect full text, boundary evidence, ordered links, actual overlap, source maps, and added
context. Favor a relationship when expansion restores needed meaning with modest extra context;
question it when it adds mostly unrelated rules or produces enormous parents. Ratios measure
character expansion, not relevance. No parent policy, strategy, or winning size is selected.

### 7a. Preserve approved artifacts and declare evidenced parent candidates

Step 6 Part/Chapter/Section records are **heading contexts**, not whole containers. Larger
parents below are therefore explicit, inspected ranges bracketed by source transitions:

| Larger parent | Start | End (exclusive) | Evidence |
|---|---:|---:|---|
| ЗУТ Chapter I | 59 | 95 | Chapter I → Chapter II |
| ЗУТ Chapter III, Section I | 112 | 132 | Section I → Section II within inspected Chapter III |
| ЗУТ Chapter III | 110 | 451 | Chapter III → Chapter IV |
| Наредба № 3 Chapter II | 2777 | 2818 | Chapter II → additional-provision heading |
| EU Chapter I | 25781 | 25850 | Chapter I → Chapter II |

These scoped hypotheses do not rewrite Step 6 or infer a corpus-wide hierarchy. Parent spans
use exact existing structural-unit boundaries and preserve every contained heading/residual unit.
Article, § and Annex parents reuse the inspected Step 6 units directly. Unclassified/residual
parents have reviewed source extents, **not invented legal roles**. No Annex subsection is
introduced: the approved ledger does not provide a reviewed subsection for the selected Annex.

In [ ]:
ensure_structural_current()
APPROVED_STRUCTURAL_ID = "cf6e1ba11a3b5d373dd8da1a7e5120059c4bdd956123440840c0b391dcbad259"
APPROVED_STRUCTURAL_RESULT = "8d11499a43d36e51ab0570a0a4283397e550b3eba6daacc31bfe41e2b2e44775"
require(STRUCTURAL_ID == APPROVED_STRUCTURAL_ID and STRUCTURAL_RESULTS_FINGERPRINT == APPROVED_STRUCTURAL_RESULT,
        "Structural setup differs from approved Step 6; review before changing parent candidates.")
STEP6_REPORT_BYTES = structural_report_path.read_bytes()
STEP7_FIXED_REPORT_BYTES = report_path.read_bytes()
PC_CONFIGURATIONS = json.loads(CONFIGURATIONS_JSON)
PC_CONFIG_BY_ID = {c["id"]: c for c in PC_CONFIGURATIONS}
STRUCTURAL_BY_ID = {r["id"]: r for r in structural_records}
STRUCTURAL_BY_FIRST_BLOCK = {r["block_ids"][0]: r for r in structural_records}

PARENT_CANDIDATES = []
for sample_id, kind, label, first, last in PROVISION_SPECS:
    unit = STRUCTURAL_BY_FIRST_BLOCK[BY_POSITION[first]["id"]]
    PARENT_CANDIDATES.append({"case": f"unit-{first}", "family": "annex" if kind == "Annex" else "article_or_paragraph_sign",
        "parent_role": unit["unit_kind"], "document_id": unit["document_id"], "start": unit["start"], "end": unit["end"],
        "member_unit_ids": [unit["id"]], "review_basis": f"Inspected {kind} source extent in frozen sample {sample_id}.",
        "opening_position": first, "closing_position_exclusive": None,
        "boundary_evidence_ids": [unit["opening_ledger_id"], unit["closing_ledger_id"]]})

for position, reason in [(54, "Publication/history preamble; no legal-parent role inferred."),
                         (2051, "Reviewed additional-provision heading remains unclassified."),
                         (2158, "Reviewed transitional heading remains unclassified."),
                         (25688, "Complete EU preamble/recital residual containing the long-paragraph sample."),
                         (28934, "Complete weakly structured source-page residual including navigation.")]:
    unit = STRUCTURAL_BY_FIRST_BLOCK[BY_POSITION[position]["id"]]
    require(unit["unit_kind"] in {"unclassified", "residual"}, "Unexpected legal role for unclassified parent.")
    PARENT_CANDIDATES.append({"case": f"unclassified-{position}", "family": "reviewed_unclassified_span",
        "parent_role": unit["unit_kind"], "document_id": unit["document_id"], "start": unit["start"], "end": unit["end"],
        "member_unit_ids": [unit["id"]], "review_basis": reason,
        "opening_position": position, "closing_position_exclusive": None,
        "boundary_evidence_ids": [unit["opening_ledger_id"], unit["closing_ledger_id"]]})

LARGER_PARENT_SPECS = [
    ("chapter-zut-1", "chapter", 59, 95, "Глава първа.", "Глава втора.", "Consecutive Chapter I/II labels."),
    ("section-zut-3-1", "section", 112, 132, "Раздел I.", "Раздел II.", "Section I/II labels inside the inspected Chapter III range."),
    ("chapter-zut-3", "chapter", 110, 451, "Глава трета.", "Глава четвърта.", "Consecutive Chapter III/IV labels; includes multiple Sections."),
    ("chapter-ordinance3-2", "chapter", 2777, 2818, "Глава втора.", "Допълнителни разпоредби",
     "Chapter II ends at the inspected transition into additional provisions; no legal role assigned to that closing heading."),
    ("chapter-eu-1", "chapter", 25781, 25850, "ГЛАВА I", "ГЛАВА II", "Consecutive EU Chapter I/II labels."),
]
for case, role, first, stop, prefix, stop_prefix, reason in LARGER_PARENT_SPECS:
    start_unit = STRUCTURAL_BY_FIRST_BLOCK[BY_POSITION[first]["id"]]
    stop_unit = STRUCTURAL_BY_FIRST_BLOCK[BY_POSITION[stop]["id"]]
    require(BY_POSITION[first]["raw_text"].startswith(prefix) and BY_POSITION[stop]["raw_text"].startswith(stop_prefix),
            "Larger-parent boundary text changed.")
    require(start_unit["unit_kind"] == role and start_unit["document_id"] == stop_unit["document_id"],
            "Larger parent lacks supported opening role or same-document closing evidence.")
    did, lo, hi = start_unit["document_id"], start_unit["start"], stop_unit["start"]
    members = [r for r in STRUCTURAL_RESULTS[did] if lo <= r["start"] and r["end"] <= hi]
    require(members and members[0]["start"] == lo and members[-1]["end"] == hi
            and "".join(r["text"] for r in members) == VIEWS[did]["text"][lo:hi], "Larger parent does not match complete structural units.")
    PARENT_CANDIDATES.append({"case": case, "family": "section_or_chapter", "parent_role": role,
        "document_id": did, "start": lo, "end": hi, "member_unit_ids": [r["id"] for r in members],
        "opening_position": first, "closing_position_exclusive": stop, "review_basis": reason,
        "boundary_evidence_ids": [start_unit["opening_ledger_id"], stop_unit["opening_ledger_id"]]})

PC_SPEC = {"version": 1, "scope": "Step 7 relationship alternatives only", "setup_id": SETUP_ID,
    "structural_id": STRUCTURAL_ID, "structural_result_fingerprint": STRUCTURAL_RESULTS_FINGERPRINT,
    "parent_candidates": PARENT_CANDIDATES, "fixed_configurations": PC_CONFIGURATIONS,
    "unit_parent_mode": "fixed_inside_parent",
    "larger_parent_modes": ["whole_contained_units", "fixed_inside_each_contained_unit"],
    "fallback_policy": None, "annex_subsections": "not established by reviewed evidence; whole Annex only",
    "approval": "Steps 1–6 approved; Step 7 only authorized"}
PC_RUN_ID = fingerprint(PC_SPEC)
show_table([{"case": c["case"], "role": c["parent_role"], "document": DOCUMENTS[c["document_id"]]["ordinal"],
             "characters": c["end"] - c["start"], "contained_units": len(c["member_unit_ids"]),
             "start_position": c["opening_position"], "end_position_exclusive": c["closing_position_exclusive"],
             "review_basis": c["review_basis"]} for c in PARENT_CANDIDATES])
show_json("Parent boundary evidence and exact constituent Step 6 IDs", PARENT_CANDIDATES)
print("Relationship experiment:", PC_RUN_ID)

### 7b. Relationship identity and controlled child construction

There are three distinct IDs:
- **Parent ID:** identifies a complete contextual span and its explicit parent hypothesis.
- **Child ID:** identifies an exact source span independently of parent/configuration, so identical
  spans can be recognized across alternatives. Repeated phrases at different offsets stay distinct.
- **Relationship ID:** identifies an ordered link within a **relationship-set ID** (parent + mode + setting).

A shared child may have several experimental parents. Resolving it therefore requires both its
ID and the chosen relationship-set ID; no arbitrary first parent is selected.

For unit parents, call the unchanged strict splitter on the entire parent. For Section/Chapter
parents, compare (a) all contained units, uncapped and unsplit, with (b) the same fixed splitter
applied uniformly to **every** contained unit. The second is an explicit experiment variant, not
an oversized-unit threshold/fallback policy. It preserves the Article-relative windows from the
unit-parent experiment. Heading/residual units remain included and labeled so parent coverage is
complete; their usefulness as future search targets remains undecided.

In [ ]:
def build_parent_child_experiment():
    parents, children, groups, links = {}, {}, {}, {}

    def register_child(did, lo, hi):
        child_id = "pc-child:" + fingerprint([SETUP_ID, did, lo, hi])
        if child_id not in children:
            children[child_id] = {"id": child_id, "document_id": did, "start": lo, "end": hi,
                "text": VIEWS[did]["text"][lo:hi], "source_spans": source_map(did, lo, hi),
                "setup_id": SETUP_ID, "record_kind": "parent-child experimental child span"}
        return child_id

    for parent_order, candidate in enumerate(PARENT_CANDIDATES, 1):
        did, lo, hi = candidate["document_id"], candidate["start"], candidate["end"]
        parent_id = "pc-parent:" + fingerprint([PC_RUN_ID, candidate["case"], did, lo, hi])
        parents[parent_id] = {**candidate, "id": parent_id, "ordinal": parent_order,
            "configuration": "complete experimental parent", "setup_id": SETUP_ID,
            "record_kind": "complete experimental " + candidate["parent_role"] + " parent",
            "text": VIEWS[did]["text"][lo:hi], "source_spans": source_map(did, lo, hi),
            "source_artifact_sha256": artifact_sha256, "source_docx_sha256": corpus["source"]["sha256"],
            "block_ids": [bid for uid in candidate["member_unit_ids"] for bid in STRUCTURAL_BY_ID[uid]["block_ids"]]}
        modes = [("fixed_inside_parent", c["id"]) for c in PC_CONFIGURATIONS]
        if candidate["family"] == "section_or_chapter":
            modes = [("whole_contained_units", "uncapped-structural-units")] + [
                ("fixed_inside_each_contained_unit", c["id"]) for c in PC_CONFIGURATIONS]
        for mode, config_id in modes:
            group_id = "pc-set:" + fingerprint([PC_RUN_ID, parent_id, mode, config_id])
            group = {"id": group_id, "parent_id": parent_id, "case": candidate["case"],
                     "mode": mode, "configuration": config_id, "relationship_ids": []}
            groups[group_id] = group
            # Every container is an existing unit. Direct unit parents have exactly one member.
            members = [STRUCTURAL_BY_ID[uid] for uid in candidate["member_unit_ids"]]
            if mode == "fixed_inside_parent":
                require(len(members) == 1, "Direct mode expects one complete structural-unit parent.")
            order = 0
            for member in members:
                if mode == "whole_contained_units":
                    spans = [(member["start"], member["end"])]
                else:
                    config = PC_CONFIG_BY_ID[config_id]
                    spans = [(member["start"] + w["start"], member["start"] + w["end"])
                             for w in fixed_length_windows(member["text"], config["length"], config["overlap_characters"])]
                for child_lo, child_hi in spans:
                    order += 1
                    child_id = register_child(did, child_lo, child_hi)
                    relation_id = "pc-link:" + fingerprint([group_id, parent_id, child_id, order])
                    links[relation_id] = {"id": relation_id, "relationship_set_id": group_id,
                        "parent_id": parent_id, "child_id": child_id, "ordinal": order,
                        "source_structural_unit_id": member["id"], "source_unit_kind": member["unit_kind"]}
                    group["relationship_ids"].append(relation_id)
    return {"parents": parents, "children": children, "groups": groups, "links": links}


PC_DATA = build_parent_child_experiment()
PC_DATA_FINGERPRINT = fingerprint(PC_DATA)
PC_GENERATION_CONTEXT = {"run_id": PC_RUN_ID, "setup_id": SETUP_ID,
                         "structural_result": STRUCTURAL_RESULTS_FINGERPRINT}
PC_PARENTS, PC_CHILDREN, PC_GROUPS, PC_LINKS = [PC_DATA[k] for k in ("parents", "children", "groups", "links")]
PC_PARENT_BY_CASE = {p["case"]: p for p in PC_PARENTS.values()}
PC_GROUP_BY_KEY = {(g["case"], g["mode"], g["configuration"]): g for g in PC_GROUPS.values()}
print(len(PC_PARENTS), "parent candidates;", len(PC_GROUPS), "relationship sets;",
      len(PC_CHILDREN), "distinct child source spans;", len(PC_LINKS), "ordered relationships.")

### 7c. Validate links, containment, ordering, full coverage, and provenance

All children must remain within their chosen parent and original document. Deduplicating only
known overlaps must reconstruct the complete parent. With whole-unit children, the units remain
uncapped and the entire parent is covered—including heading contexts. With fixed-window
children, overlaps occur within each original container and reset at the next contained unit.
Empty source-block evidence remains in the parent membership and original document maps.

In [ ]:
def validate_relationship_group(group, parents, children, links):
    require(group["parent_id"] in parents, "Unknown parent in relationship set.")
    parent = parents[group["parent_id"]]
    require(not parent.get("parent_id"), "Parents cannot themselves have a parent in this two-level experiment.")
    ordered_links = [links[rid] for rid in group["relationship_ids"]]
    require([r["ordinal"] for r in ordered_links] == list(range(1, len(ordered_links) + 1)), "Child order is not contiguous.")
    require(len({r["child_id"] for r in ordered_links}) == len(ordered_links), "Duplicate child within relationship set.")
    covered_end, restored = parent["start"], []
    previous_start = parent["start"]
    for link in ordered_links:
        require(link["relationship_set_id"] == group["id"] and link["parent_id"] == parent["id"], "Relationship points to wrong set/parent.")
        require(link["child_id"] != parent["id"], "Self-link is forbidden.")
        require(link["child_id"] in children, "Unknown child in relationship.")
        child = children[link["child_id"]]
        require(child["document_id"] == parent["document_id"], "Child crosses documents.")
        require(parent["start"] <= child["start"] < child["end"] <= parent["end"], "Child is not fully contained in parent.")
        require(previous_start <= child["start"] <= covered_end and child["end"] > covered_end, "Child gap, redundant span, or source-order error.")
        expected = parent["text"][child["start"] - parent["start"]:child["end"] - parent["start"]]
        require(child["text"] == expected, "Child text differs from parent source slice.")
        restored.append(child["text"][covered_end - child["start"]:])
        previous_start, covered_end = child["start"], child["end"]
    require(covered_end == parent["end"] and "".join(restored) == parent["text"], "Children do not reconstruct complete parent.")


for group in PC_GROUPS.values():
    validate_relationship_group(group, PC_PARENTS, PC_CHILDREN, PC_LINKS)
    parent = PC_PARENTS[group["parent_id"]]
    links = [PC_LINKS[rid] for rid in group["relationship_ids"]]
    for uid in parent["member_unit_ids"]:
        member = STRUCTURAL_BY_ID[uid]
        actual = [(PC_CHILDREN[r["child_id"]]["start"], PC_CHILDREN[r["child_id"]]["end"])
                  for r in links if r["source_structural_unit_id"] == uid]
        if group["mode"] == "whole_contained_units":
            expected = [(member["start"], member["end"])]
        else:
            config = PC_CONFIG_BY_ID[group["configuration"]]
            expected = [(member["start"] + w["start"], member["start"] + w["end"])
                        for w in fixed_length_windows(member["text"], config["length"], config["overlap_characters"])]
        require(actual == expected, "Child spans differ from whole units or already-tested fixed windows.")
for node in list(PC_PARENTS.values()) + list(PC_CHILDREN.values()):
    require(node["text"] == VIEWS[node["document_id"]]["text"][node["start"]:node["end"]]
            == reconstruct(node["source_spans"]), "Node source mapping/text changed.")
require(all("parent_id" not in c for c in PC_CHILDREN.values()), "Shared child registry must not choose one arbitrary parent.")
print("PASS: every relationship has valid IDs, ordered contained children, exact parent reconstruction, and source mappings.")

### 7d. Parent sizes, child counts, and context-expansion diagnostics

For one selected child, **expansion ratio = parent characters / child characters** and added
context is the difference. It is not a relevance score. `other_legal_units` below counts other
Step 6 Article/§/Annex units in that parent; those provisions **may** be useful or unrelated.
The notebook cannot decide relevance without inspecting the actual text and a future task.

Each group's overlap percentage measures repeated child positions relative to that one parent.
It excludes intentional parent/child containment and alternative parents sharing the same text.
Do not sum overlapping parent alternatives into a corpus coverage or storage estimate.
A short parent may yield one identical-size child: this degenerate relationship is shown, not
artificially forced into smaller pieces. The whole-unit control deliberately keeps long children.

In [ ]:
LEGAL_PARENT_KINDS = {"article", "paragraph_sign", "annex"}
PC_GROUP_DIAGNOSTICS, PC_LINK_DIAGNOSTICS = [], {}
for group in PC_GROUPS.values():
    parent = PC_PARENTS[group["parent_id"]]
    parent_size = parent["end"] - parent["start"]
    members = [STRUCTURAL_BY_ID[uid] for uid in parent["member_unit_ids"]]
    legal_members = [m for m in members if m["unit_kind"] in LEGAL_PARENT_KINDS]
    sizes, ratios, legal_child_ratios = [], [], []
    for rid in group["relationship_ids"]:
        relation = PC_LINKS[rid]
        child = PC_CHILDREN[relation["child_id"]]
        size = child["end"] - child["start"]
        ratio = parent_size / size
        touched = [m for m in legal_members if intersection_length((child["start"], child["end"]), (m["start"], m["end"]))]
        PC_LINK_DIAGNOSTICS[rid] = {"parent_characters": parent_size, "child_characters": size,
            "context_expansion_ratio": round(ratio, 4), "added_context_characters": parent_size - size,
            "parent_legal_unit_count": len(legal_members), "other_legal_units": len(legal_members) - len(touched)}
        sizes.append(size); ratios.append(ratio)
        if relation["source_unit_kind"] in LEGAL_PARENT_KINDS:
            legal_child_ratios.append(ratio)
    PC_GROUP_DIAGNOSTICS.append({"relationship_set_id": group["id"], "case": group["case"],
        "family": parent["family"], "mode": group["mode"], "configuration": group["configuration"],
        "parent_characters": parent_size, "child_count": len(sizes),
        "child_size_min": min(sizes), "child_size_max": max(sizes),
        "expansion_min": round(min(ratios), 4), "expansion_median": round(length_summary(ratios)["median_p50"], 4),
        "expansion_max": round(max(ratios), 4),
        "legal_child_expansion_median": round(length_summary(legal_child_ratios)["median_p50"], 4) if legal_child_ratios else "not applicable",
        "overlap_extra_character_pct": round(100 * (sum(sizes) - parent_size) / parent_size, 2),
        "parent_legal_units": len(legal_members), "one_child_equals_parent": len(sizes) == 1 and sizes[0] == parent_size,
        "coverage_pct": 100})

for family in ("article_or_paragraph_sign", "section_or_chapter", "annex", "reviewed_unclassified_span"):
    rows = [r for r in PC_GROUP_DIAGNOSTICS if r["family"] == family]
    display(HTML(f"<details><summary>{esc(family)}: all settings and child modes ({len(rows)} sets)</summary>"
                 + table_html(rows) + "</details>"))
show_table([r for r in PC_GROUP_DIAGNOSTICS if r["configuration"] in {GALLERY_CONFIGURATION, "uncapped-structural-units"}],
           ["case", "mode", "configuration", "parent_characters", "child_count", "child_size_min", "child_size_max",
            "expansion_median", "expansion_max", "legal_child_expansion_median", "parent_legal_units", "one_child_equals_parent"])

PC_SAMPLE_COVERAGE = []
for sample in BASELINE_SAMPLES:
    lo, hi, did = sample["start"], sample["end"], sample["document_id"]
    parents = [p for p in PC_PARENTS.values() if p["document_id"] == did and intersection_length((lo, hi), (p["start"], p["end"]))]
    covered = interval_union_length((max(lo, p["start"]), min(hi, p["end"])) for p in parents)
    require(covered == hi - lo, "Parent candidate set leaves a frozen sample source character unrepresented.")
    PC_SAMPLE_COVERAGE.append({"sample": sample["sample_id"], "covered_by_union_of_candidates_pct": 100,
        "fully_containing_parent_cases": [p["case"] for p in parents if p["start"] <= lo and hi <= p["end"]],
        "partially_intersecting_parent_cases": [p["case"] for p in parents if not (p["start"] <= lo and hi <= p["end"])]})
show_table(PC_SAMPLE_COVERAGE)

### 7e. Identical children under different parents — controlled comparison

Hold source, splitter settings, and Article-relative child boundaries constant. Then compare
only the parent context. The following table reports every supported Article-to-larger-parent
pair across all nine fixed settings. `identical_child_spans` must equal the Article child count.
Canonical child IDs are identical even though relationship IDs and parent IDs differ.

For example, the same Article 10 child can resolve to Article 10, Section I, or Chapter III.
More context is not automatically better: Chapter III also contains many other legal units.
Tiny Articles also repeat identically across different size settings; that means the window is
larger than the Article, not that those settings produce different evidence.

In [ ]:
IDENTICAL_CHILD_COMPARISONS = []
for narrow in PC_PARENTS.values():
    if narrow["family"] != "article_or_paragraph_sign":
        continue
    uid = narrow["member_unit_ids"][0]
    for larger in PC_PARENTS.values():
        if larger["family"] != "section_or_chapter" or uid not in larger["member_unit_ids"]:
            continue
        for config in PC_CONFIGURATIONS:
            a = PC_GROUP_BY_KEY[(narrow["case"], "fixed_inside_parent", config["id"])]
            b = PC_GROUP_BY_KEY[(larger["case"], "fixed_inside_each_contained_unit", config["id"])]
            narrow_ids = [PC_LINKS[rid]["child_id"] for rid in a["relationship_ids"]]
            larger_ids = [PC_LINKS[rid]["child_id"] for rid in b["relationship_ids"]
                          if PC_LINKS[rid]["source_structural_unit_id"] == uid]
            require(narrow_ids == larger_ids, "Controlled parent comparison changed child boundaries or identity.")
            IDENTICAL_CHILD_COMPARISONS.append({"configuration": config["id"], "unit_parent": narrow["case"],
                "larger_parent": larger["case"], "identical_child_spans": len(narrow_ids),
                "unit_parent_characters": narrow["end"] - narrow["start"],
                "larger_parent_characters": larger["end"] - larger["start"],
                "same_child_parent_size_multiplier": round((larger["end"] - larger["start"]) / (narrow["end"] - narrow["start"]), 4),
                "example_child_id": narrow_ids[0]})
show_table([r for r in IDENTICAL_CHILD_COMPARISONS if r["configuration"] == GALLERY_CONFIGURATION])
display(HTML("<details><summary>Identical-child comparisons across ALL nine settings</summary>"
             + table_html(IDENTICAL_CHILD_COMPARISONS) + "</details>"))
print("PASS:", len(IDENTICAL_CHILD_COMPARISONS), "same-child parent comparisons; no child boundary changes.")

### 7f. Complete parents and ordered children on the frozen samples

Expand a sample's relationship display. The parent and **all ordered children** are present,
with complete expandable text and source maps. No parent is shortened. The explicit link table
adds relationship IDs, absolute and parent-relative child spans, expansion, and additional legal
units. The existing Step 4 relationship helper provides the parent beside its children.

The gallery setting remains 2,000 characters / 10% overlap for display convenience only. Two
samples include an unclassified heading plus a § provision; their gallery shows the § parent and
states its partial sample coverage. Their heading-only parents remain available in the candidate
and diagnostics tables. This does not remove either heading from the experiment.

In [ ]:
def ensure_pc_current():
    ensure_structural_current()
    require(fingerprint(PC_SPEC) == PC_RUN_ID and PC_SPEC["parent_candidates"] == PARENT_CANDIDATES
            and PC_SPEC["fixed_configurations"] == PC_CONFIGURATIONS and canonical(PC_CONFIGURATIONS) == CONFIGURATIONS_JSON,
            "Parent candidates/settings changed; rebuild Step 7 relationships and diagnostics.")
    require(PC_GENERATION_CONTEXT == {"run_id": PC_RUN_ID, "setup_id": SETUP_ID,
            "structural_result": STRUCTURAL_RESULTS_FINGERPRINT}, "Parent-child generation context is stale.")


def relationship_display_records(group_id):
    ensure_pc_current()
    require(group_id in PC_GROUPS, "Unknown relationship set.")
    group = PC_GROUPS[group_id]
    parent = {**PC_PARENTS[group["parent_id"]], "configuration": group["mode"] + " / " + group["configuration"]}
    children = [{**PC_CHILDREN[PC_LINKS[rid]["child_id"]], "parent_id": parent["id"],
                 "ordinal": PC_LINKS[rid]["ordinal"], "relationship_id": rid,
                 "configuration": parent["configuration"]} for rid in group["relationship_ids"]]
    return parent, children


def show_parent_child_set(group_id, sample_id=None):
    parent, children = relationship_display_records(group_id)
    group = PC_GROUPS[group_id]
    sample = SAMPLE_BY_ID[sample_id] if sample_id is not None else None
    if sample:
        require(sample["document_id"] == parent["document_id"], "Gallery sample and parent belong to different documents.")
        pct = round(100 * intersection_length((sample["start"], sample["end"]), (parent["start"], parent["end"])) / sample["characters"], 2)
        note = f"Parent covers {pct}% of this frozen sample. Its full extent is displayed."
    else:
        note = "Complete parent extent; all ordered children are displayed."
    rows = []
    for rid in group["relationship_ids"]:
        link = PC_LINKS[rid]; child = PC_CHILDREN[link["child_id"]]
        rows.append({"order": link["ordinal"], "relationship_id": rid, "parent_id": parent["id"], "child_id": child["id"],
            "source_span": [child["start"], child["end"]],
            "parent_relative_span": [child["start"] - parent["start"], child["end"] - parent["start"]],
            "source_unit_kind": link["source_unit_kind"], **PC_LINK_DIAGNOSTICS[rid]})
    body = f"<p>{esc(note)} Child count: {len(children)}. No parent or child text has been truncated.</p>"
    body += details_html("Parent hypothesis, role, and boundary provenance", json.dumps({k: parent[k] for k in (
        "case", "parent_role", "review_basis", "boundary_evidence_ids", "member_unit_ids", "source_artifact_sha256", "source_docx_sha256")}, ensure_ascii=False, indent=2))
    body += "<details><summary>All ordered relationship IDs, source spans and expansion ratios</summary>" + table_html(rows) + "</details>"
    body += relationship_html(parent, children, sample, REVIEWED_PROVISIONS)
    display(HTML(f"<details><summary>{esc(sample_id or parent['case'])}: {esc(parent['case'])} / {esc(group['mode'])} / {esc(group['configuration'])}</summary>"
                 + body + "</details>"))


GALLERY_PARENT_CASES = {
    "normal_article": "unit-61", "short_neighbors": "chapter-zut-1", "nested_structure": "chapter-zut-3",
    "long_article": "unit-2779", "additional_provision": "unit-2052", "transitional_provision": "unit-2159",
    "annex": "unit-2848", "long_paragraph": "unclassified-25688", "publication_history": "unclassified-54",
    "eu_article": "unit-25783", "unclassified_heading": "unclassified-2051", "weak_structure": "unclassified-28934",
}
for sample in BASELINE_SAMPLES:
    case = GALLERY_PARENT_CASES[sample["sample_id"]]
    mode = "fixed_inside_each_contained_unit" if PC_PARENT_BY_CASE[case]["family"] == "section_or_chapter" else "fixed_inside_parent"
    show_parent_child_set(PC_GROUP_BY_KEY[(case, mode, GALLERY_CONFIGURATION)]["id"], sample["sample_id"])
# Display the whole-unit Section control too; children remain uncapped (no fallback).
show_parent_child_set(PC_GROUP_BY_KEY[("section-zut-3-1", "whole_contained_units", "uncapped-structural-units")]["id"], "nested_structure")
# Any other existing setting/parent can be inspected without generating a new policy, e.g.:
# show_parent_child_set(PC_GROUP_BY_KEY[("chapter-ordinance3-2", "whole_contained_units", "uncapped-structural-units")]["id"])

### 7g. Manual child → parent resolution (no search)

Select a child ID from an ordered relationship table and the relationship-set ID for the
experiment you want to inspect. The defaults below explicitly choose the first child of the
long Article's 2,000-character / 10% run. Replace them with printed IDs to inspect another link.

The lookup checks exact IDs and membership, then returns the stored parent in full. It performs
no text matching, ranking, embeddings, or retrieval. The same child can legitimately appear
under several parent choices, so calling it without a relationship set is rejected.

In [ ]:
def resolve_child_parent(child_id, relationship_set_id=None):
    ensure_pc_current()
    require(relationship_set_id in PC_GROUPS, "Choose a valid relationship-set ID; a shared child can have multiple experimental parents.")
    require(child_id in PC_CHILDREN, "Unknown child ID.")
    group = PC_GROUPS[relationship_set_id]
    matches = [PC_LINKS[rid] for rid in group["relationship_ids"] if PC_LINKS[rid]["child_id"] == child_id]
    require(len(matches) == 1, "Child is not uniquely linked in the selected relationship set.")
    link = matches[0]
    parent, child = PC_PARENTS[link["parent_id"]], PC_CHILDREN[child_id]
    require(parent["document_id"] == child["document_id"] and parent["start"] <= child["start"] < child["end"] <= parent["end"],
            "Linked child is not contained in its parent.")
    return parent, child, link


MANUAL_RELATIONSHIP_SET = PC_GROUP_BY_KEY[("unit-2779", "fixed_inside_parent", GALLERY_CONFIGURATION)]["id"]
MANUAL_CHILD_ID = PC_LINKS[PC_GROUPS[MANUAL_RELATIONSHIP_SET]["relationship_ids"][0]]["child_id"]
manual_parent, manual_child, manual_link = resolve_child_parent(MANUAL_CHILD_ID, MANUAL_RELATIONSHIP_SET)
show_table([{"relationship_set_id": MANUAL_RELATIONSHIP_SET, "selected_child_id": MANUAL_CHILD_ID,
             "resolved_parent_id": manual_parent["id"], "relationship_id": manual_link["id"],
             **PC_LINK_DIAGNOSTICS[manual_link["id"]]}])
manual_parent_display, manual_children_display = relationship_display_records(MANUAL_RELATIONSHIP_SET)
manual_selected_display = next(c for c in manual_children_display if c["id"] == MANUAL_CHILD_ID)
display(HTML("<h4>Manually selected child</h4>" + span_card(manual_selected_display, reviewed_units=REVIEWED_PROVISIONS)
             + "<h4>Resolved COMPLETE parent (no truncation)</h4>" + span_card(manual_parent_display, reviewed_units=REVIEWED_PROVISIONS)))

### 7h. Same child, Article vs Section vs Chapter context

For the inspected Article 10 child, show the same source span under all three supported parents.
The complete parent text and source map are available in each column. Added neighboring legal
units are a concrete prompt for review, not automatically classified as irrelevant.

Also compare the long Article's child to its Chapter II parent in the identical-child table:
the Article parent restores the rest of that long provision; the Chapter additionally includes
other provisions. Neither is assumed to be the eventual LLM context policy.

In [ ]:
shared_group = PC_GROUP_BY_KEY[("unit-114", "fixed_inside_parent", GALLERY_CONFIGURATION)]
SHARED_CHILD_ID = PC_LINKS[shared_group["relationship_ids"][0]]["child_id"]
SHARED_CHILD_CONTEXT_ROWS = []
columns = []
for case, mode in [("unit-114", "fixed_inside_parent"), ("section-zut-3-1", "fixed_inside_each_contained_unit"),
                   ("chapter-zut-3", "fixed_inside_each_contained_unit")]:
    group = PC_GROUP_BY_KEY[(case, mode, GALLERY_CONFIGURATION)]
    parent, child, link = resolve_child_parent(SHARED_CHILD_ID, group["id"])
    parent_view, _ = relationship_display_records(group["id"])
    SHARED_CHILD_CONTEXT_ROWS.append({"same_child_id": child["id"], "parent_case": case, "parent_id": parent["id"],
                                     "relationship_id": link["id"], **PC_LINK_DIAGNOSTICS[link["id"]]})
    columns.append(f"<div style='flex:1;min-width:320px'><h4>{esc(case)}</h4>" + span_card(parent_view, reviewed_units=REVIEWED_PROVISIONS) + "</div>")
show_table(SHARED_CHILD_CONTEXT_ROWS)
display(HTML(details_html("The identical complete child text in all three alternatives", PC_CHILDREN[SHARED_CHILD_ID]["text"])))
display(HTML("<details><summary>Complete Article / Section / Chapter parents for the SAME child</summary>"
             + "<div style='display:flex;gap:12px;overflow:auto'>" + "".join(columns) + "</div></details>"))
print("Observed: one identical source child resolves to parents of",
      [row["parent_characters"] for row in SHARED_CHILD_CONTEXT_ROWS], "characters.")
print("Inspect expansion against the actual rules, conditions, and neighboring provisions. Character ratios do not measure usefulness.")
print("Whole-unit controls keep long children; short unclassified parents can yield a single equal-size child. No fallback policy is applied.")

### 7i. Validate relationship failures, identity, and reproducibility

Checks reject ambiguous lookup, wrong-parent links, missing coverage, cross-document children,
out-of-parent spans, self-links, and stale generation context. Rebuild the relationship graph
to verify stable IDs and spans. Preserve Stage 1, the frozen sample manifest, and both approved
baseline reports. These checks establish relationship/source fidelity, not retrieval quality or
legal correctness of every provisional structural unit.

In [ ]:
expect_value_error(lambda: resolve_child_parent(MANUAL_CHILD_ID), "relationship-set ID")
expect_value_error(lambda: resolve_child_parent("unknown-child", MANUAL_RELATIONSHIP_SET), "Unknown child")
wrong_group = next(g for g in PC_GROUPS.values() if MANUAL_CHILD_ID not in [PC_LINKS[rid]["child_id"] for rid in g["relationship_ids"]])
expect_value_error(lambda: resolve_child_parent(MANUAL_CHILD_ID, wrong_group["id"]), "not uniquely linked")

test_group = PC_GROUPS[MANUAL_RELATIONSHIP_SET]
first_link_id = test_group["relationship_ids"][0]
first_link = PC_LINKS[first_link_id]
first_child = PC_CHILDREN[first_link["child_id"]]
parent = PC_PARENTS[test_group["parent_id"]]
expect_value_error(lambda: validate_relationship_group(test_group, PC_PARENTS, PC_CHILDREN,
    {**PC_LINKS, first_link_id: {**first_link, "parent_id": "wrong-parent"}}), "wrong set/parent")
expect_value_error(lambda: validate_relationship_group(test_group, PC_PARENTS,
    {**PC_CHILDREN, first_child["id"]: {**first_child, "start": parent["start"] - 1}}, PC_LINKS), "fully contained")
expect_value_error(lambda: validate_relationship_group(test_group, PC_PARENTS,
    {**PC_CHILDREN, first_child["id"]: {**first_child, "document_id": "another-document"}}, PC_LINKS), "crosses documents")
expect_value_error(lambda: validate_relationship_group(test_group, PC_PARENTS, PC_CHILDREN,
    {**PC_LINKS, first_link_id: {**first_link, "child_id": parent["id"]}}), "Self-link")
expect_value_error(lambda: validate_relationship_group({**test_group, "relationship_ids": test_group["relationship_ids"][:-1]},
    PC_PARENTS, PC_CHILDREN, PC_LINKS), "reconstruct complete parent")
correct_pc_context = PC_GENERATION_CONTEXT
try:
    PC_GENERATION_CONTEXT = {**correct_pc_context, "structural_result": "stale"}
    expect_value_error(ensure_pc_current, "generation context is stale")
finally:
    PC_GENERATION_CONTEXT = correct_pc_context

require(len(PC_GROUPS) == sum(10 if p["family"] == "section_or_chapter" else 9 for p in PC_PARENTS.values()),
        "Missing parent/mode/configuration relationship sets.")
require(all(r["coverage_pct"] == 100 for r in PC_GROUP_DIAGNOSTICS), "Parent coverage diagnostics failed.")
require(any(r["child_size_max"] > 4000 for r in PC_GROUP_DIAGNOSTICS if r["mode"] == "whole_contained_units"),
        "Whole-unit children unexpectedly size-capped.")
require(any(r["one_child_equals_parent"] for r in PC_GROUP_DIAGNOSTICS), "Short-parent equality case is missing.")
require(len({r["same_child_id"] for r in SHARED_CHILD_CONTEXT_ROWS}) == 1
        and len({r["parent_id"] for r in SHARED_CHILD_CONTEXT_ROWS}) == 3, "Shared-child parent-choice comparison is invalid.")
require(fingerprint(build_parent_child_experiment()) == PC_DATA_FINGERPRINT == fingerprint(PC_DATA),
        "Parent-child IDs, source spans, or links changed on rerun.")
require(fingerprint(STRUCTURAL_RESULTS) == APPROVED_STRUCTURAL_RESULT
        and structural_report_path.read_bytes() == STEP6_REPORT_BYTES, "Approved structural baseline/report changed.")
require(fingerprint(FIXED_RESULTS) == STEP5_RESULT_FINGERPRINT and report_path.read_bytes() == STEP7_FIXED_REPORT_BYTES,
        "Approved fixed baseline/report changed.")
require(APPROVED_MANIFEST_PATH.read_bytes() == approved_manifest_bytes, "Frozen samples changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == artifact_sha256
        and fingerprint(corpus) == corpus_fingerprint_before, "Stage 1 source changed.")
print("PASS: contained ordered children, complete parent reconstruction, exact mappings, deterministic links, safe lookup, and unchanged approved artifacts.")

### 7j. Save relationship results and stop after Step 7

The report stores parent/child source spans, provenance, explicit links, all-setting diagnostics,
and identical-child comparisons. Text can be reconstructed exactly from the unchanged Stage 1
views; it remains fully inspectable in the notebook. Canonical child records do not pick a
single parent because parent choice is precisely what this experiment compares.

**Review:**
- Compare complete Article/§ context with Section/Chapter context for identical children.
- Inspect conditions, definitions, exceptions, and potentially unrelated neighboring provisions.
- Check the whole Annex and reviewed unclassified/residual cases without assigning new legal roles.
- Notice tiny equal-size parent/child pairs and uncapped whole-unit children.
- Verify manual ID lookup and distinguish child overlap from intentional parent-context containment.

**Approved checkpoint — Step 7 complete.** No final parent policy or chunking strategy
selected. Step 8 below now tests oversized-unit alternatives. No final fallback policy, Hybrid, embeddings,
vector storage, search/retrieval, reranking, RAG, or agents are introduced.

In [ ]:
PC_REPORT = {"report_version": 1, "scope": "Step 7 only; parent-child relationship alternatives",
    "run_id": PC_RUN_ID, "spec": PC_SPEC, "data_fingerprint": PC_DATA_FINGERPRINT,
    "parents": {pid: {k: v for k, v in p.items() if k != "text"} for pid, p in PC_PARENTS.items()},
    "children": {cid: {k: v for k, v in c.items() if k != "text"} for cid, c in PC_CHILDREN.items()},
    "relationship_sets": PC_GROUPS, "relationships": PC_LINKS,
    "group_diagnostics": PC_GROUP_DIAGNOSTICS, "relationship_diagnostics": PC_LINK_DIAGNOSTICS,
    "sample_coverage": PC_SAMPLE_COVERAGE, "identical_child_comparisons": IDENTICAL_CHILD_COMPARISONS,
    "shared_child_parent_contexts": SHARED_CHILD_CONTEXT_ROWS,
    "validation": "PASS: source fidelity, containment, order, full coverage, explicit lookup, stable IDs, immutable baselines",
    "decision": "No final parent policy or chunking strategy selected; waiting for Step 7 review"}
pc_report_path = OUTPUT_DIR / f"parent_child_report.{fingerprint(PC_REPORT)}.json"
pc_report_bytes = (canonical(PC_REPORT) + "\n").encode("utf-8")
if pc_report_path.exists():
    require(pc_report_path.read_bytes() == pc_report_bytes, "Existing parent-child report differs.")
else:
    pc_report_path.write_bytes(pc_report_bytes)
require(json.loads(pc_report_path.read_text(encoding="utf-8")) == PC_REPORT, "Parent-child report round-trip failed.")
print("Parent-child report:", pc_report_path.relative_to(ROOT))
print("STOP AFTER STEP 7 — relationship alternatives ready for review; no final parent policy selected.")

## Step 8 — Oversized-unit experiments (stop here for review)

**What:** Keep the complete, uncapped Step 6 units as the control and compare two ways to
make contained children only when a unit is **strictly longer** than 2,000, 4,000, or 8,000
Python Unicode characters. Equality does not trigger splitting. These are experimental probes,
not a fallback policy or a new combined corpus. Unaffected units remain in the control.

**Why:** Complete provisions can be much larger than a future context budget. Smaller children
may help inspection but can separate conditions, lists and exceptions. Preserving paragraphs
is a mechanical property, not proof that legal meaning survives.

**How:** For each affected parent compare exact windows against paragraph/whitespace-aware
windows at 1,000 and 2,000 characters, with requested overlap of 0% and 10%. Exact windows
reuse Step 5's splitter. Parent choice stays fixed to Step 6; Step 7 Section/Chapter alternatives
are not substituted here. There are 24 threshold/method/target/overlap combinations. Child
spans are reused across thresholds because a trigger changes eligibility, not the splitter.

**Inspect first:** the control and affected-unit/type tables below. The experiment covers the
same four complete source documents as the baselines, not all 21 source documents. Frozen
sample windows select displays only; they do not set parent or child boundaries.

In [ ]:
ensure_pc_current()
OS_APPROVED_BYTES = {str(p): p.read_bytes() for p in
                     (APPROVED_MANIFEST_PATH, report_path, structural_report_path, pc_report_path)}
OS_APPROVED_FINGERPRINTS = (fingerprint(STRUCTURAL_RESULTS), fingerprint(FIXED_RESULTS), fingerprint(PC_DATA))
OS_THRESHOLDS = [2000, 4000, 8000]
OS_SETTINGS = [{"id": f"{method}-chars-{target}-overlap-{pct}pct", "method": method,
                "target": target, "overlap_percent": pct, "overlap": target * pct // 100}
               for method in ("exact", "boundary_aware") for target in (1000, 2000) for pct in (0, 10)]
OS_SPEC = {"version": 1, "scope": "Step 8 controlled experiments only", "setup_id": SETUP_ID,
           "structural_id": STRUCTURAL_ID, "structural_result": STRUCTURAL_RESULTS_FINGERPRINT,
           "artifact_sha256": artifact_sha256, "thresholds": OS_THRESHOLDS, "settings": OS_SETTINGS,
           "eligibility": "parent characters strictly greater than threshold",
           "paragraphs": "source block spans; added separator attached to preceding block",
           "boundary_algorithm": "farthest paragraph end, else whitespace, else exact; overlap at safe boundaries within requested budget",
           "final_policy": None}
OS_RUN_ID = fingerprint(OS_SPEC)
OS_CONTROL = [u for did in BASELINE_DOCUMENT_IDS for u in STRUCTURAL_RESULTS[did]]
OS_THRESHOLD_ROWS = []
for threshold in OS_THRESHOLDS:
    affected = [u for u in OS_CONTROL if u["end"] - u["start"] > threshold]
    OS_THRESHOLD_ROWS.append({"trigger": threshold, "control_units": len(OS_CONTROL),
        "affected_units": len(affected), "affected_pct": round(100 * len(affected) / len(OS_CONTROL), 2),
        "unchanged_control_units": len(OS_CONTROL) - len(affected),
        "affected_types": dict(sorted(Counter(u["unit_kind"] for u in affected).items())),
        "affected_parent_sizes": length_summary([u["end"] - u["start"] for u in affected])})
print("Uncapped Step 6 control:", len(OS_CONTROL), "units across", len(BASELINE_DOCUMENT_IDS), "complete documents")
show_table([{"unit_kind": kind, **length_summary([u["end"] - u["start"] for u in OS_CONTROL if u["unit_kind"] == kind])}
            for kind in sorted({u["unit_kind"] for u in OS_CONTROL})])
show_table(OS_THRESHOLD_ROWS)
show_table(OS_SETTINGS)

### 8a. Boundary-aware mechanics and their costs

The splitter receives the parent's exact text plus source-block paragraph spans. Added newlines
stay attached to the preceding paragraph. Empty-block evidence remains in the complete parent
and its Stage 1 membership; it is not invented as nonempty text.

For each window, use the furthest paragraph end within the target. If none advances coverage,
use the furthest whitespace boundary (immediately **after** whitespace); if none exists, use an
exact character cut. The parent's end wins when the remainder fits. A short paragraph can be
left for the next child instead of being cut to fill this one. Children never exceed the target.
A source paragraph already clipped by the parent is not claimed to be restored.

For overlap, prefer a paragraph boundary within the requested overlap budget. Otherwise,
only when the ending cut is inside an oversized paragraph, use a whitespace boundary there.
If neither is available, reduce overlap to zero. Exact cuts are allowed for forward coverage
of unbroken text, but are not forced just to manufacture overlap. Starts and covered ends must
both advance. Requested overlap is thus an upper bound for the aware method; **actual overlap
is measured from source intervals**, including added separators. Exact windows keep their
requested overlap. No characters are stripped, normalized, masked, or rewritten.

This conservative choice can leave short children or less overlap than requested. Compare these
costs with paragraph fragmentation; do not read smaller children as an automatic improvement.

In [ ]:
def oversized_eligible(size, threshold):
    return size > threshold


def aware_windows(text, paragraph_spans, target, overlap):
    # Reuse strict baseline parameter validation, including empty input.
    fixed_length_windows("", target, overlap)
    n = len(text)
    require(all(0 <= a < b <= n for a, b in paragraph_spans), "Invalid paragraph span.")
    require(all(b <= c for (a, b), (c, d) in zip(paragraph_spans, paragraph_spans[1:])),
            "Paragraph spans must be ordered and disjoint.")
    edges = sorted({0, n, *(a for a, b in paragraph_spans), *(b for a, b in paragraph_spans)})
    whitespace = [i + 1 for i, char in enumerate(text) if char.isspace()]
    result, start, covered = [], 0, 0
    while start < n:
        limit = min(start + target, n)
        require(limit > covered, "Boundary splitter cannot advance.")
        if limit == n:
            end, reason = n, "parent_end"
        else:
            candidates = edges[bisect_right(edges, covered):bisect_right(edges, limit)]
            if candidates:
                end, reason = candidates[-1], "paragraph_boundary"
            else:
                candidates = whitespace[bisect_right(whitespace, covered):bisect_right(whitespace, limit)]
                end, reason = (candidates[-1], "whitespace_inside_paragraph") if candidates else (limit, "exact_no_available_boundary")
        result.append({"start": start, "end": end, "text": text[start:end], "end_reason": reason})
        if end == n:
            break
        lower = max(start + 1, end - overlap)
        next_starts = [p for p in edges if lower <= p < end]
        if next_starts:
            next_start, start_reason = next_starts[0], "paragraph_overlap"
        elif overlap and any(a < end < b and b - a > target for a, b in paragraph_spans):
            safe = [p for p in whitespace if lower <= p < end
                    and any(a < p <= end < b and b - a > target for a, b in paragraph_spans)]
            next_start, start_reason = (safe[0], "whitespace_overlap") if safe else (end, "overlap_reduced_to_zero")
        else:
            next_start, start_reason = end, "no_overlap" if not overlap else "overlap_reduced_to_zero"
        result[-1]["next_start_reason"] = start_reason
        require(start < next_start <= end and 0 <= end - next_start <= overlap, "Overlap does not advance safely.")
        start, covered = next_start, end
    return result


def parent_paragraph_spans(parent):
    # Blocks remain distinct. A following added separator belongs to the preceding block.
    lo, hi = parent["start"], parent["end"]
    segments = VIEWS[parent["document_id"]]["segments"]
    spans = []
    for i, segment in enumerate(segments):
        if segment["kind"] != "source":
            continue
        end = segment["end"]
        if i + 1 < len(segments) and segments[i + 1]["kind"] == "separator":
            end = segments[i + 1]["end"]
        a, b = max(lo, segment["start"]), min(hi, end)
        if a < b:
            spans.append((a - lo, b - lo))
    return spans


def os_windows(parent, setting):
    text = VIEWS[parent["document_id"]]["text"][parent["start"]:parent["end"]]
    if setting["method"] == "exact":
        return [{**w, "end_reason": "parent_end" if w["end"] == len(text) else "exact_fixed_window",
                 "next_start_reason": "exact_requested_overlap"} for w in
                fixed_length_windows(text, setting["target"], setting["overlap"])]
    require(setting["method"] == "boundary_aware", "Unknown splitting method.")
    return aware_windows(text, parent_paragraph_spans(parent), setting["target"], setting["overlap"])


def check_os_windows(text, windows, target, overlap):
    if not text:
        require(not windows, "Empty input produced children.")
        return
    require(windows[0]["start"] == 0 and windows[-1]["end"] == len(text), "Missing parent ends.")
    require(interval_union_length([(w["start"], w["end"]) for w in windows]) == len(text), "Parent coverage gap.")
    previous_start, covered, recovered = -1, 0, ""
    for w in windows:
        a, b = w["start"], w["end"]
        require(previous_start < a <= covered < b <= len(text) and b - a <= target, "Invalid child progress/size.")
        require(covered - a <= overlap and w["text"] == text[a:b], "Overlap/text mismatch.")
        recovered += w["text"][covered-a:]
        previous_start, covered = a, b
    require(recovered == text, "Removing measured overlaps failed exact reconstruction.")


# Focused edge cases: Cyrillic, empty paragraphs/separators, whitespace, exact-size and unbroken text.
OS_SYNTHETIC_CASES = [([], 8), (["Чл. 2а. (изм.)", "§ 1. Изключение.", ""], 10),
    (["а" * 73], 10), (["   " * 14, "край"], 10), (["а б в г д е ж з и к л м н о п", "Чл. 3."], 10),
    (["1234567890"], 10), (["аб", "вг", "де", "жз", "и"], 10)]
for paragraphs, target in OS_SYNTHETIC_CASES:
    text, spans, cursor = "\n".join(paragraphs), [], 0
    for i, paragraph in enumerate(paragraphs):
        end = cursor + len(paragraph) + (i < len(paragraphs) - 1)
        if end > cursor: spans.append((cursor, end))
        cursor = end
    for overlap in (0, max(1, target // 10), target - 1):
        windows = aware_windows(text, spans, target, overlap)
        check_os_windows(text, windows, target, overlap)
        require(windows == aware_windows(text, spans, target, overlap), "Nondeterministic synthetic splitter.")
require(not oversized_eligible(2000, 2000) and oversized_eligible(2001, 2000), "Threshold equality changed.")
require(any(w["end_reason"] == "exact_no_available_boundary" for w in aware_windows("я" * 25, [(0,25)], 10, 1)),
        "Unbroken-text exact fallback is not exercised.")
# A small paragraph that does not fit the remaining window is moved whole to the next child.
whole = aware_windows("абвг\nдежз\nий", [(0,5),(5,10),(10,12)], 8, 0)
require([(w["start"],w["end"]) for w in whole] == [(0,5),(5,12)], "Short paragraphs were needlessly split.")
whitespace_case = aware_windows("абвг дежз ийкл", [(0,13)], 8, 0)
require(whitespace_case[0]["end"] == 5 and whitespace_case[0]["end_reason"] == "whitespace_inside_paragraph",
        "Oversized paragraph did not prefer available whitespace.")
print("PASS: synthetic coverage, exact reconstruction, strict progress, equality, Unicode and unbroken-text fallback.")

### 8b. Generate experimental children while retaining every complete parent

The parent registry references the original structural IDs, roles, boundary evidence and uncertainty.
Each child has its own reproducible ID, explicit relationship ID, absolute source span, reversible
block map, split reason, actual previous/next overlap and context-expansion ratio. Provenance remains
available through its parent and the fingerprinted run. No new legal ancestry is inferred.

In [ ]:
OS_PARENTS = {u["id"]: u for u in OS_CONTROL if oversized_eligible(u["end"] - u["start"], min(OS_THRESHOLDS))}
OS_GROUPS = {}
for pid, parent in OS_PARENTS.items():
    for setting in OS_SETTINGS:
        windows = os_windows(parent, setting)
        check_os_windows(parent["text"], windows, setting["target"], setting["overlap"])
        gid = "oversized-set:" + fingerprint([OS_RUN_ID, pid, setting["id"]])
        children = []
        for ordinal, window in enumerate(windows, 1):
            a, b = parent["start"] + window["start"], parent["start"] + window["end"]
            cid = "oversized-child:" + fingerprint([OS_RUN_ID, pid, setting["id"], a, b])
            child = {"id": cid, "ordinal": ordinal, "document_id": parent["document_id"],
                "start": a, "end": b, "text": window["text"], "setup_id": SETUP_ID,
                "configuration": setting["id"], "record_kind": "oversized experiment child",
                "parent_id": pid, "relationship_id": "oversized-link:" + fingerprint([gid, cid, ordinal]),
                "source_spans": source_map(parent["document_id"], a, b),
                "split_reason": window["end_reason"], "next_start_reason": window.get("next_start_reason", "parent_complete"),
                "requested_overlap": setting["overlap"],
                "actual_previous_overlap": max(0, windows[ordinal-2]["end"] - window["start"]) if ordinal > 1 else 0,
                "actual_next_overlap": max(0, window["end"] - windows[ordinal]["start"]) if ordinal < len(windows) else 0,
                "context_expansion_ratio": round((parent["end"] - parent["start"]) / (b-a), 4)}
            require(reconstruct(child["source_spans"]) == child["text"], "Child source reconstruction failed.")
            child["paragraph_blocks_cut"] = paragraph_cuts(child)
            child["short_paragraph_blocks_cut"] = [bid for bid in child["paragraph_blocks_cut"]
                                                    if len(BLOCKS[bid]["raw_text"]) <= setting["target"]]
            child["reviewed_boundary_diagnostics"] = legal_diagnostic(child)
            children.append(child)
        OS_GROUPS[(pid, setting["id"])] = {"id": gid, "parent_id": pid, "setting": setting,
            "eligible_thresholds": [t for t in OS_THRESHOLDS if oversized_eligible(parent["end"]-parent["start"], t)],
            "children": children}
OS_DATA_FINGERPRINT = fingerprint(list(OS_GROUPS.values()))


def ensure_os_current():
    ensure_pc_current()
    require(fingerprint(OS_SPEC) == OS_RUN_ID and OS_SPEC["thresholds"] == OS_THRESHOLDS
            and OS_SPEC["settings"] == OS_SETTINGS, "Step 8 settings changed; rerun generation and diagnostics.")
    require(fingerprint(STRUCTURAL_RESULTS) == OS_SPEC["structural_result"], "Structural input changed; rebuild Step 8.")
    require(fingerprint(list(OS_GROUPS.values())) == OS_DATA_FINGERPRINT, "Step 8 child results changed; rebuild diagnostics.")


def os_group_metrics(group):
    children = group["children"]
    parent = OS_PARENTS[group["parent_id"]]
    size = parent["end"] - parent["start"]
    return {"parent_id": parent["id"], "unit_kind": parent["unit_kind"], "parent_characters": size,
        "setting": group["setting"]["id"], "eligible_thresholds": group["eligible_thresholds"],
        "children": len(children), "child_sizes": length_summary([len(c["text"]) for c in children]),
        "actual_overlap": length_summary([c["actual_previous_overlap"] for c in children[1:]]),
        "duplicated_characters": sum(len(c["text"]) for c in children) - size,
        "paragraph_blocks_cut": len({bid for c in children for bid in c["paragraph_blocks_cut"]}),
        "short_paragraph_blocks_cut": len({bid for c in children for bid in c["short_paragraph_blocks_cut"]}),
        "end_reasons": dict(sorted(Counter(c["split_reason"] for c in children).items()))}
OS_GROUP_METRICS = [os_group_metrics(g) for g in OS_GROUPS.values()]
print("Complete affected parents retained:", len(OS_PARENTS), "; distinct parent/setting experiments:", len(OS_GROUPS))

### 8c. Matched diagnostics — compare methods within each threshold and setting

Each row counts only affected parents. Thresholds change the population, so a lower total child
count at 8,000 does not establish a better splitter. `paragraph_blocks_cut` counts distinct source
blocks cut within each parent experiment; a block may appear in several overlapping children.
The short-paragraph column isolates blocks whose raw text could fit the target.
A raw paragraph exactly at the target can exceed it once its attached separator is included;
inspect this edge case rather than assuming the aware method never cuts a short raw paragraph. Child-end reason
counts explain *how* a cut was chosen, while the source-map diagnostics capture both start and
end cuts. Duplication counts repeated source positions and added separators in children, excluding
intentional parent containment and repeated wording already present in the source.

All figures concern provisional structural units. The small reviewed legal-unit ledger can identify
cuts in reviewed samples only; it cannot validate legal completeness across the corpus.

In [ ]:
ensure_os_current()
OS_DIAGNOSTICS = []
for threshold in OS_THRESHOLDS:
    for setting in OS_SETTINGS:
        groups = [g for g in OS_GROUPS.values() if g["setting"]["id"] == setting["id"] and threshold in g["eligible_thresholds"]]
        children = [c for g in groups for c in g["children"]]
        sizes = [len(c["text"]) for c in children]
        originals = sum(OS_PARENTS[g["parent_id"]]["end"] - OS_PARENTS[g["parent_id"]]["start"] for g in groups)
        duplicate = sum(sizes) - originals
        overlaps = [c["actual_previous_overlap"] for g in groups for c in g["children"][1:]]
        metrics = [os_group_metrics(g) for g in groups]
        OS_DIAGNOSTICS.append({"threshold": threshold, "setting": setting["id"], "affected_parents": len(groups),
            "children": len(children), "child_size_distribution": length_summary(sizes),
            "actual_overlap_distribution": length_summary(overlaps),
            "overlap_below_requested": sum(x < setting["overlap"] for x in overlaps),
            "duplicated_characters": duplicate, "duplication_pct_of_parent_text": round(100*duplicate/originals, 2) if originals else 0,
            "paragraph_blocks_cut": sum(m["paragraph_blocks_cut"] for m in metrics),
            "short_paragraph_blocks_cut": sum(m["short_paragraph_blocks_cut"] for m in metrics),
            "end_reasons": dict(sorted(Counter(c["split_reason"] for c in children).items())),
            "covered_parent_characters_pct": 100 if groups else None})
show_table([{k:r[k] for k in ("threshold", "setting", "affected_parents", "children", "paragraph_blocks_cut",
                             "short_paragraph_blocks_cut", "duplication_pct_of_parent_text", "overlap_below_requested")}
            for r in OS_DIAGNOSTICS])
display(HTML("<details><summary>All 24 configurations: sizes, measured overlap and cut reasons</summary>"
             + table_html(OS_DIAGNOSTICS) + "</details>"))
OS_MATCHED_COMPARISONS = []
for threshold in OS_THRESHOLDS:
    for target in (1000, 2000):
        for pct in (0, 10):
            exact, aware = [next(r for r in OS_DIAGNOSTICS if r["threshold"] == threshold
                                and r["setting"] == f"{method}-chars-{target}-overlap-{pct}pct")
                            for method in ("exact", "boundary_aware")]
            OS_MATCHED_COMPARISONS.append({"threshold": threshold, "target": target, "overlap_pct": pct,
                "aware_minus_exact_children": aware["children"]-exact["children"],
                "aware_minus_exact_paragraphs_cut": aware["paragraph_blocks_cut"]-exact["paragraph_blocks_cut"],
                "aware_minus_exact_duplication_chars": aware["duplicated_characters"]-exact["duplicated_characters"],
                "interpretation": "Trade-offs on identical parents; no winner score"})
show_table(OS_MATCHED_COMPARISONS)

### 8d. Representative parents, children and cut points

The gallery reuses the frozen samples. The long paragraph is inside the **complete residual EU
preamble**, not an invented one-paragraph parent. The publication-history sample uses its existing
residual parent, including its neighboring source text. History remains verbatim; no history-policy
annotations or transformations are introduced. The § sample also includes a separate heading;
that heading remains a separate control unit outside this § parent.

First inspect the trigger table and all eight settings per example. Then expand the two matched
method panels (2,000 / 10% is a display convenience, not a selected setting). Each contains the
complete uncapped parent beside **all ordered children**, with the sampled passage highlighted.
The source overview marks every cut and actual overlap. Relationship tables expose split reasons,
IDs, exact spans, paragraph cuts and reviewed legal-unit cuts. Full source mappings are expandable.
`show_oversized_example` also accepts any of the tested targets/overlaps for manual inspection.

Look for intact short paragraphs, splits inside the 3,600-character paragraph and the history list,
list items or exceptions separated across children, and whitespace cuts that still interrupt a legal
provision. Whole-parent retention prevents source loss; it does not make the child self-contained.

In [ ]:
OS_EXAMPLES = [{"sample_id": sample_id, "parent_id": STRUCTURAL_BY_FIRST_BLOCK[BY_POSITION[position]["id"]]["id"]}
    for sample_id, position in [("long_article",2779),
                                ("transitional_provision",2159), ("annex",2848), ("publication_history",54)]]
# Resolve the long-paragraph parent by containment rather than relying on a preamble position.
OS_EXAMPLES.insert(1, {"sample_id": "long_paragraph", "parent_id": next(u["id"] for u in OS_CONTROL
    if u["document_id"] == SAMPLE_BY_ID["long_paragraph"]["document_id"]
    and u["start"] <= SAMPLE_BY_ID["long_paragraph"]["start"] < u["end"])})
OS_EXAMPLE_ROWS = []
for example in OS_EXAMPLES:
    parent = OS_PARENTS[example["parent_id"]]
    sample = SAMPLE_BY_ID[example["sample_id"]]
    OS_EXAMPLE_ROWS.append({"sample": example["sample_id"], "kind": parent["unit_kind"],
        "parent_characters": len(parent["text"]), "sample_characters": sample["characters"],
        "sample_coverage_pct": round(100 * intersection_length((parent["start"], parent["end"]), (sample["start"], sample["end"])) / sample["characters"], 2),
        **{f"trigger_{t}": "split experiment" if len(parent["text"]) > t else "retain uncapped control" for t in OS_THRESHOLDS}})
show_table(OS_EXAMPLE_ROWS)


def show_oversized_example(sample_id, target=2000, overlap_pct=10):
    ensure_os_current()
    example = next(e for e in OS_EXAMPLES if e["sample_id"] == sample_id)
    parent, sample = OS_PARENTS[example["parent_id"]], SAMPLE_BY_ID[sample_id]
    display(HTML("<h4>" + esc(sample_id) + " — complete uncapped control and matched children</h4>"))
    show_table([m for m in OS_GROUP_METRICS if m["parent_id"] == parent["id"]])
    for method in ("exact", "boundary_aware"):
        setting_id = f"{method}-chars-{target}-overlap-{overlap_pct}pct"
        require((parent["id"], setting_id) in OS_GROUPS, "Choose a tested child setting.")
        group = OS_GROUPS[(parent["id"], setting_id)]
        children = group["children"]
        rows = [{"ordinal": c["ordinal"], "child_id": c["id"], "parent_id": parent["id"],
            "relationship_id": c["relationship_id"], "characters": len(c["text"]), "source_span": [c["start"],c["end"]],
            "parent_relative_span": [c["start"]-parent["start"],c["end"]-parent["start"]],
            "intersects_frozen_sample": bool(intersection_length((c["start"],c["end"]),(sample["start"],sample["end"]))),
            **{k:c[k] for k in ("split_reason", "next_start_reason", "requested_overlap", "actual_previous_overlap",
                                "actual_next_overlap", "context_expansion_ratio", "paragraph_blocks_cut", "reviewed_boundary_diagnostics")}}
            for c in children]
        cuts = sorted({p for c in children for p in (c["start"],c["end"])})
        overlaps = [(b["start"],a["end"]) for a,b in zip(children,children[1:]) if b["start"] < a["end"]]
        body = "<p>Complete parent: " + str(len(parent["text"])) + " characters; children: " + str(len(children)) + ".</p>"
        body += "<details><summary>Source panel: all cuts (bars), overlap (gold), frozen sample (blue)</summary>"
        body += highlight_html(parent["text"], parent["start"], cuts, overlaps, (sample["start"],sample["end"])) + "</details>"
        body += table_html(rows)
        body += relationship_html(parent, children, sample, REVIEWED_PROVISIONS)
        display(HTML("<details><summary>" + esc(setting_id) + " — full parent and all children</summary>" + body + "</details>"))
for example in OS_EXAMPLES:
    show_oversized_example(example["sample_id"])

### 8e. Preservation, reproducibility and limits

The checks below verify every child in every distinct parent/setting experiment. They reconstruct
parents by removing only *measured* overlaps, resolve explicit links and IDs, and verify source maps,
containment, order, coverage and unchanged approved baselines. Synthetic checks also exercise
empty input, final short children, exact-size input, Cyrillic, whitespace and unbroken text.
These are preservation checks, not legal validation. Size and sample changes require regeneration;
freshness guards reject stale settings or child results before displays or report export.

In [ ]:
ensure_os_current()
OS_VALIDATED_CHILDREN = 0
for (pid, setting_id), group in OS_GROUPS.items():
    parent, setting, children = OS_PARENTS[pid], group["setting"], group["children"]
    require("parent_id" not in parent and parent["id"] == pid, "Parent link cycle or unresolved parent.")
    require(parent == STRUCTURAL_BY_ID[pid], "Original uncapped parent changed.")
    rerun = os_windows(parent, setting)
    check_os_windows(parent["text"], rerun, setting["target"], setting["overlap"])
    require(len(rerun) == len(children), "Rerun child count changed.")
    require(len({c["id"] for c in children}) == len(children), "Duplicate child IDs.")
    for i, (child, w) in enumerate(zip(children, rerun), 1):
        require(child["parent_id"] == pid and child["document_id"] == parent["document_id"]
                and parent["start"] <= child["start"] < child["end"] <= parent["end"], "Child escaped parent.")
        require(child["start"] == parent["start"] + w["start"] and child["end"] == parent["start"] + w["end"]
                and child["text"] == w["text"] == reconstruct(child["source_spans"]), "Source spans changed on rerun.")
        require(child["id"] == "oversized-child:" + fingerprint([OS_RUN_ID,pid,setting_id,child["start"],child["end"]]), "Unstable child ID.")
        require(child["relationship_id"] == "oversized-link:" + fingerprint([group["id"],child["id"],i]), "Unstable relationship ID.")
        OS_VALIDATED_CHILDREN += 1
# Verify the guard rejects stale settings and data, then restore the exact experiment.
os_original_target = OS_SETTINGS[0]["target"]
try:
    OS_SETTINGS[0]["target"] += 1
    expect_value_error(ensure_os_current, "Step 8 settings changed")
finally:
    OS_SETTINGS[0]["target"] = os_original_target
os_first_child = next(iter(OS_GROUPS.values()))["children"][0]
os_original_reason = os_first_child["split_reason"]
try:
    os_first_child["split_reason"] = "stale edit"
    expect_value_error(ensure_os_current, "Step 8 child results changed")
finally:
    os_first_child["split_reason"] = os_original_reason
require(OS_APPROVED_FINGERPRINTS == (fingerprint(STRUCTURAL_RESULTS), fingerprint(FIXED_RESULTS), fingerprint(PC_DATA)),
        "An approved baseline changed.")
require(all(Path(path).read_bytes() == data for path,data in OS_APPROVED_BYTES.items()), "An approved report or manifest changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == artifact_sha256 and fingerprint(corpus) == corpus_fingerprint_before,
        "Stage 1 input changed.")
ensure_os_current()
print("PASS:", OS_VALIDATED_CHILDREN, "children: source fidelity, complete parent coverage, containment, links, deterministic regeneration.")
print("PASS: frozen manifest, Steps 5–7 reports/results, Stage 1 source unchanged; stale settings rejected.")

### 8f. Review observations and stop

**What the outputs establish:** trigger tables quantify which provisional types need children;
matched comparisons expose how boundary adjustment changes paragraph cuts, child counts, sizes
and duplication. Exact cuts offer predictable size/overlap. Aware cuts can preserve short source
paragraphs but may leave underfilled children and reduce overlap. Neither method promises to keep
a complete rule, exception or list together. Both preserve the complete uncapped parent.

**What to inspect:**
- Compare the long Article and § source panels for conditions separated from qualifications.
- Inspect the long paragraph: whitespace can avoid cutting a word without preserving its argument.
- Compare Annex form/list boundaries; source paragraphs are not automatically meaningful form units.
- Inspect the publication-history parent and inline notes in original text; dates remain verbatim.
- Check the matched difference table for regressions as well as improvements, including more children
  or less context overlap. A lower paragraph-cut count alone is insufficient to select a method.

**What remains unproven:** legal completeness, useful context, model token budgets and retrieval
quality. No threshold, child size, method, final parent policy or final strategy is selected.
The report below records these experiments independently of approved Steps 1–7.

**Approved checkpoint — Step 8 complete.** Step 9 below investigates annotations only. History policy, metadata
experiments, final strategy comparison, Hybrid, embeddings, retrieval, RAG and agents are not implemented.

In [ ]:
ensure_os_current()
OS_REPORT = {"report_version": 1, "scope": "Step 8 only — oversized-unit experiments", "run_id": OS_RUN_ID,
    "spec": OS_SPEC, "data_fingerprint": OS_DATA_FINGERPRINT,
    "control_units": len(OS_CONTROL), "threshold_diagnostics": OS_THRESHOLD_ROWS,
    "parents": {pid:{k:v for k,v in parent.items() if k != "text"} for pid,parent in OS_PARENTS.items()},
    "relationship_sets": [{**{k:v for k,v in group.items() if k != "children"},
        "children": [{k:v for k,v in child.items() if k != "text"} for child in group["children"]]}
        for group in OS_GROUPS.values()],
    "configuration_diagnostics": OS_DIAGNOSTICS, "matched_comparisons": OS_MATCHED_COMPARISONS,
    "sample_examples": OS_EXAMPLES, "sample_trigger_diagnostics": OS_EXAMPLE_ROWS,
    "parent_setting_diagnostics": OS_GROUP_METRICS,
    "validation": {"status": "PASS", "children_checked": OS_VALIDATED_CHILDREN,
                   "source_fidelity": True, "coverage": True, "containment": True, "legal_validation": False},
    "decision": "No threshold, size or splitting method selected; waiting for Step 8 review"}
os_report_path = OUTPUT_DIR / f"oversized_unit_report.{fingerprint(OS_REPORT)}.json"
os_report_bytes = (canonical(OS_REPORT) + "\n").encode("utf-8")
if os_report_path.exists():
    require(os_report_path.read_bytes() == os_report_bytes, "Existing oversized-unit report differs.")
else:
    os_report_path.write_bytes(os_report_bytes)
require(json.loads(os_report_path.read_text(encoding="utf-8")) == OS_REPORT, "Report round trip failed.")
print("Oversized-unit report:", os_report_path.relative_to(ROOT))
print("STOP AFTER STEP 8 — experiments ready for review; no final threshold, child size or method selected.")

## Step 9 — Amendment and publication-history investigation

**What:** Compare original text against the **same original text plus separate span annotations**.
No text is removed, masked, rewritten, normalized or relocated. These annotations are an inspection
layer, not Step 10 metadata policy, verified effective dates or legal-status assertions.

**Why:** Publication lists and amendment notes occupy space and can become separated from the
provisions they appear to qualify. Character share and boundary inspection can describe that
mechanism; they cannot establish retrieval noise or retrieval quality.

**How:** Reuse the frozen samples and the same four complete document views. Explicitly reviewed
publication-list blocks at positions 56 and 2739 are annotated in full. For inline candidates,
scan balanced parentheses within each source block; require both a Bulgarian amendment cue
(`Изм.`, `Доп.`, `Нов`, `Нова`, `Ново`, `Нови`, `Отм.`, `Обн.`, `Попр.`) and a `ДВ` reference.
Nested parentheses stay intact. Do not classify ordinary Article citations, paragraph numbers or
bare dates as history. Unmatched parentheses and uncovered `ДВ` references enter an uncertainty
ledger, not automatic annotations. This is intentionally incomplete: EU publication headers,
nonparenthesized notes and other conventions are outside the detector's current coverage.

**Inspect:** source context and the annotation ledger first, followed by paired displays, measured
shares and boundary witnesses. An apparent scope is only a source-local proposal, not a legal
finding. Numeric/date strings remain verbatim, including any source phrase “в сила от”.

In [ ]:
ensure_os_current()
H_APPROVED_PATHS = [APPROVED_MANIFEST_PATH, report_path, structural_report_path, pc_report_path, os_report_path]
H_APPROVED_BYTES = {str(p): p.read_bytes() for p in H_APPROVED_PATHS}
H_SOURCE_FINGERPRINT = fingerprint({did: VIEWS[did]["text"] for did in BASELINE_DOCUMENT_IDS})
H_BASELINE_FINGERPRINTS = [fingerprint(FIXED_RESULTS), fingerprint(STRUCTURAL_RESULTS), fingerprint(PC_DATA), OS_DATA_FINGERPRINT]
H_SPEC = {"version":1, "setup_id":SETUP_ID, "artifact_sha256":artifact_sha256,
    "source_fingerprint":H_SOURCE_FINGERPRINT, "fixed_run":BASELINE_ID, "structural_run":STRUCTURAL_ID,
    "reviewed_publication_positions":[56,2739],
    "inline_rule":"balanced block-local parentheses with amendment cue and ДВ; outermost qualifying span",
    "scope_rule":"same-block adjacent excerpt and containing provisional structural unit; no legal attribution",
    "coverage_limit":"Bulgarian pattern candidates in four baseline documents; not exhaustive",
    "text_transformation":None, "legal_status_assertions":False}
H_RUN_ID = fingerprint(H_SPEC)
H_CUE = re.compile(r"(?i)(?:\b(?:изм|доп|отм|обн|попр)\.|\bнов(?:а|о|и)?\b)")
H_DV = re.compile(r"\bДВ\b", re.I)


def history_candidates(text):
    stack, pairs, unresolved = [], [], []
    for i,char in enumerate(text):
        if char == "(": stack.append(i)
        elif char == ")":
            if stack: pairs.append((stack.pop(), i+1))
            else: unresolved.append({"offset":i,"reason":"unmatched closing parenthesis"})
    unresolved.extend({"offset":i,"reason":"unmatched opening parenthesis"} for i in stack)
    candidates = [(a,b) for a,b in sorted(pairs) if H_CUE.search(text[a:b]) and H_DV.search(text[a:b])]
    outer = [(a,b) for a,b in candidates if not any(c <= a and b <= d and (c,d)!=(a,b) for c,d in candidates)]
    return outer, unresolved


H_ANNOTATIONS, H_UNCERTAINTY = [], []
for did in BASELINE_DOCUMENT_IDS:
    for bid in VIEWS[did]["block_ids"]:
        block=BLOCKS[bid];text=block["raw_text"];_,origin,_=BLOCK_SPANS[bid]
        candidates, unresolved = history_candidates(text)
        reviewed_list = block["position"] in H_SPEC["reviewed_publication_positions"]
        spans = [(0,len(text))] if reviewed_list else candidates
        for a,b in spans:
            start,end=origin+a,origin+b
            units=[u for u in STRUCTURAL_RESULTS[did] if u["start"] <= start and end <= u["end"]]
            annotation={"id":"history:"+fingerprint([H_RUN_ID,did,bid,a,b]), "document_id":did,
                "block_id":bid,"position":block["position"],"block_start":a,"block_end":b,
                "start":start,"end":end,"verbatim_text":text[a:b],"source_spans":source_map(did,start,end),
                "kind":"document_publication_list" if reviewed_list else "inline_history_candidate",
                "review_basis":"reviewed list block and surrounding title/provision context" if reviewed_list else "pattern candidate; requires manual review",
                "proposed_scope":"document publication/amendment list" if reviewed_list else "nearby clause or provision; exact scope unresolved",
                "containing_structural_unit_ids":[u["id"] for u in units],
                "uncertainty":"Source placement is evidence only; attribution, completeness, effective dates and legal status are not verified."}
            H_ANNOTATIONS.append(annotation)
        for match in H_DV.finditer(text):
            if not any(a <= match.start() < b for a,b in spans):
                H_UNCERTAINTY.append({"document_id":did,"block_id":bid,"position":block["position"],
                    "block_offset":match.start(),"reason":"uncovered ДВ reference: citation, nonparenthesized history or missed candidate",
                    "context":text[max(0,match.start()-70):match.end()+100]})
        if H_DV.search(text):
            H_UNCERTAINTY.extend({"document_id":did,"block_id":bid,"position":block["position"],
                                  "block_offset":r["offset"],"reason":r["reason"],"context":text[:180]} for r in unresolved)
H_ANNOTATIONS.sort(key=lambda a:(DOCUMENTS[a["document_id"]]["ordinal"],a["start"],a["end"]))
H_ANNOTATION_FINGERPRINT = fingerprint(H_ANNOTATIONS)
H_BY_ID = {a["id"]:a for a in H_ANNOTATIONS}
show_table([{"document":DOCUMENTS[did]["ordinal"],
    "publication_lists":sum(a["document_id"]==did and a["kind"]=="document_publication_list" for a in H_ANNOTATIONS),
    "inline_candidates":sum(a["document_id"]==did and a["kind"]=="inline_history_candidate" for a in H_ANNOTATIONS),
    "uncertainty_entries":sum(a["document_id"]==did for a in H_UNCERTAINTY),
    "coverage":"No matches is not evidence of no history"} for did in BASELINE_DOCUMENT_IDS])
show_table([{k:a[k] for k in ("id","position","block_start","block_end","kind","verbatim_text","proposed_scope","uncertainty")}
            for a in H_ANNOTATIONS if a["position"] in (56,61,64,2739,2779)])
display(HTML("<details><summary>Uncovered references and unresolved punctuation (all entries)</summary>"+table_html(H_UNCERTAINTY)+"</details>"))

### 9a. Original versus original plus annotations

The left panel displays the unchanged source block. The right displays the same block with
span highlighting only. Generated explanations and proposed scope live below the source panels;
they are never prepended to source text. Expand the annotation records for verbatim notes and
reversible stream/block offsets. Check the normal Article, the long Article and both publication
lists. The full ledger remains available as `H_ANNOTATIONS` and in the separate report.

In [ ]:
def ensure_history_current():
    ensure_os_current()
    require(fingerprint(H_SPEC)==H_RUN_ID and fingerprint(H_ANNOTATIONS)==H_ANNOTATION_FINGERPRINT,
            "History configuration/annotations changed; rebuild Step 9 diagnostics.")
    require(fingerprint({did:VIEWS[did]["text"] for did in BASELINE_DOCUMENT_IDS})==H_SOURCE_FINGERPRINT,
            "Source views changed; rebuild history experiment.")


def history_representation(record, annotated=False):
    text=VIEWS[record["document_id"]]["text"][record["start"]:record["end"]]
    return {"text":text,"annotation_ids":[a["id"] for a in H_ANNOTATIONS
        if annotated and a["document_id"]==record["document_id"]
        and intersection_length((record["start"],record["end"]),(a["start"],a["end"]))]}


def show_history_block(position):
    ensure_history_current()
    block=BY_POSITION[position];did,start,end=BLOCK_SPANS[block["id"]]
    record={"document_id":did,"start":start,"end":end}
    original=history_representation(record);annotated=history_representation(record,True)
    require(original["text"]==annotated["text"]==block["raw_text"],"Representation changed source.")
    annotations=[H_BY_ID[aid] for aid in annotated["annotation_ids"]]
    left="<h4>Original unchanged</h4>"+highlight_html(original["text"],start)
    right="<h4>Original unchanged + annotation overlay</h4>"+highlight_html(annotated["text"],start,
        overlaps=[(a["start"],a["end"]) for a in annotations])
    display(HTML(f"<h4>Source body position {position}</h4><div style='display:grid;grid-template-columns:1fr 1fr;gap:16px'>"
                 +"<section>"+left+"</section><section>"+right+"</section></div>"
                 +details_html("Separate annotation records — proposed scope, uncertainty and exact mappings",json.dumps(annotations,ensure_ascii=False,indent=2))))
for position in (56,61,64,2739,2779):show_history_block(position)

### 9b. How much text is annotated?

Measure the **union of annotated character intersections / unchanged record character count**.
Denominators include the existing added block separators; annotations do not include separators.
For inline matches these are *candidate-history shares*, not a verified exhaustive history measure.
Overlapping detections would be counted once. Each overlapping chunk is measured independently;
its annotation characters are not summed into a corpus total.

First compare the same frozen samples and their intersecting uncapped structural units. Then
inspect fixed-length chunks across all nine existing settings on those samples. These measurements
reuse existing boundaries; no new splitter or final strategy comparison is introduced. The share
table distinguishes document lists from inline notes and shows zero-match samples honestly.

In [ ]:
def history_share(record):
    did,lo,hi=record["document_id"],record["start"],record["end"]
    hits=[a for a in H_ANNOTATIONS if a["document_id"]==did and intersection_length((lo,hi),(a["start"],a["end"]))]
    intervals=[(max(lo,a["start"]),min(hi,a["end"])) for a in hits]
    count=interval_union_length(intervals)
    return {"characters":hi-lo,"annotated_characters":count,"annotated_share_pct":round(100*count/(hi-lo),2) if hi>lo else None,
        "publication_list_characters":interval_union_length([(max(lo,a["start"]),min(hi,a["end"])) for a in hits if a["kind"]=="document_publication_list"]),
        "inline_candidate_characters":interval_union_length([(max(lo,a["start"]),min(hi,a["end"])) for a in hits if a["kind"]=="inline_history_candidate"]),
        "annotation_ids":[a["id"] for a in hits],
        "annotations_cut":[a["id"] for a in hits if lo>a["start"] or hi<a["end"]]}
H_SAMPLE_SHARES=[{"sample_id":s["sample_id"],**history_share(s)} for s in BASELINE_SAMPLES]
H_UNIT_SHARES=[];H_FIXED_SHARES=[]
for sample in BASELINE_SAMPLES:
    did=sample["document_id"]
    for unit in STRUCTURAL_RESULTS[did]:
        if intersection_length((sample["start"],sample["end"]),(unit["start"],unit["end"])):
            H_UNIT_SHARES.append({"sample_id":sample["sample_id"],"unit_id":unit["id"],"unit_kind":unit["unit_kind"],**history_share(unit)})
    for setting in CONFIGURATIONS:
        for chunk in FIXED_RESULTS[setting["id"]][did]:
            if intersection_length((sample["start"],sample["end"]),(chunk["start"],chunk["end"])):
                H_FIXED_SHARES.append({"sample_id":sample["sample_id"],"configuration":setting["id"],"chunk_id":chunk["id"],**history_share(chunk)})
share_columns=["sample_id","characters","annotated_characters","annotated_share_pct","publication_list_characters","inline_candidate_characters"]
show_table(H_SAMPLE_SHARES,share_columns)
show_table(H_UNIT_SHARES,["sample_id","unit_kind",*share_columns[1:]])
display(HTML("<details><summary>Representative fixed-length chunks: shares and cut annotation IDs across all nine settings</summary>"
             +table_html(H_FIXED_SHARES)+"</details>"))

### 9c. Boundary witnesses: split note versus separated nearby text

Two distinct mechanical events are recorded: **a cut inside an annotation**, and a chunk containing
a complete inline note while excluding all or part of a nearby source-text anchor. For the second,
the anchor is at most 60 characters immediately after the note, ending before the next annotated
note in the same block (whitespace is skipped only to locate offsets, never removed from source).
It is a *context proxy*, not the legally affected provision. Adjacency cannot establish legal scope.

Search only existing fixed-length results for witnesses and select them deterministically. Show
actual neighboring chunks, overlap, source maps and reviewed-unit cut diagnostics. A witness of
separation does not mean all overlapping chunks lose that context: the table reports whether
another chunk in that same setting contains the entire note and anchor together. The unchanged
structural unit remains available as larger source context. Publication-list splits have no assumed
single-provision target.

In [ ]:
H_ANCHORS={}
for a in H_ANNOTATIONS:
    if a["kind"]!="inline_history_candidate":continue
    text=BLOCKS[a["block_id"]]["raw_text"];origin=a["start"]-a["block_start"]
    end_limit=min([other["block_start"] for other in H_ANNOTATIONS if other["block_id"]==a["block_id"] and other["block_start"]>=a["block_end"]] or [len(text)])
    start=a["block_end"]
    while start<end_limit and text[start].isspace():start+=1
    end=min(start+60,end_limit)
    if start<end:
        H_ANCHORS[a["id"]]={"start":origin+start,"end":origin+end,"verbatim_text":text[start:end],
                              "scope":"following same-block context proxy; legal attachment unverified"}
H_WITNESSES=[]
for kind in ("document_publication_list","inline_history_candidate"):
    for event in (("annotation_cut",) if kind=="document_publication_list" else ("annotation_cut","nearby_context_separated")):
        found=None
        for setting in CONFIGURATIONS:
            if found:break
            for a in H_ANNOTATIONS:
                if a["kind"]!=kind:continue
                chunks=FIXED_RESULTS[setting["id"]][a["document_id"]];anchor=H_ANCHORS.get(a["id"])
                for chunk in chunks:
                    hit=intersection_length((chunk["start"],chunk["end"]),(a["start"],a["end"]))
                    cut=0<hit<a["end"]-a["start"]
                    separated=(hit==a["end"]-a["start"] and anchor is not None and
                               not(chunk["start"]<=anchor["start"] and anchor["end"]<=chunk["end"]))
                    if (event=="annotation_cut" and cut) or (event=="nearby_context_separated" and separated):
                        recovered=bool(anchor and any(c["start"]<=min(a["start"],anchor["start"]) and max(a["end"],anchor["end"])<=c["end"] for c in chunks))
                        found={"kind":kind,"event":event,"configuration":setting["id"],"annotation_id":a["id"],
                               "chunk_id":chunk["id"],"position":a["position"],"context_anchor":anchor,
                               "another_chunk_contains_full_note_and_anchor":recovered}
                        break
                if found:break
        if found:H_WITNESSES.append(found)
        else:H_WITNESSES.append({"kind":kind,"event":event,"status":"No witness found in current tested settings; not proof of absence elsewhere"})
show_table(H_WITNESSES)
for witness in H_WITNESSES:
    if "chunk_id" not in witness:continue
    a=H_BY_ID[witness["annotation_id"]];chunks=FIXED_RESULTS[witness["configuration"]][a["document_id"]]
    index=next(i for i,c in enumerate(chunks) if c["id"]==witness["chunk_id"])
    peers=chunks[max(0,index-1):index+2]
    body=details_html("Verbatim history note / list and uncertain scope",json.dumps(a,ensure_ascii=False,indent=2))
    anchor=witness["context_anchor"]
    if anchor:body+=details_html("Nearby source anchor — not verified legal scope",json.dumps(anchor,ensure_ascii=False,indent=2))
    for chunk in peers:body+=span_card(chunk,neighbors=peers,reviewed_units=REVIEWED_PROVISIONS)
    for uid in a["containing_structural_unit_ids"]:
        body+="<details><summary>Complete unchanged structural context</summary>"+span_card(STRUCTURAL_BY_ID[uid],reviewed_units=REVIEWED_PROVISIONS)+"</details>"
    display(HTML("<details><summary>"+esc(witness["event"]+" / "+witness["kind"]+" / "+witness["configuration"])+"</summary>"+body+"</details>"))

### 9d. Future policy hypotheses — no policy implemented

| Future possibility | Potential benefit | Cost / unresolved question |
|---|---|---|
| Keep history in searchable text | Source and historical qualifications remain directly visible | Repeated dates/references might distract from topical content; this is an untested retrieval hypothesis |
| Metadata only | Could reduce text devoted to publication references | Requires reliable scope attribution, approved removal/transformation, and a way to recover the original; **not implemented here** |
| Both text and metadata | Preserves source while enabling separate annotation access | Duplicated information, maintenance and attribution complexity; no production schema selected |

The current paired display demonstrates only unchanged text with an optional annotation layer.
It does not implement a searchable corpus or move history into metadata. High character share
measures volume, not noise. A split note demonstrates a source-boundary issue, not a measured
retrieval failure. Scope may be a clause, paragraph, Article or larger amendment section; source
position alone does not settle that question. Dates in a note may express different events and
must not be equated with verified effective dates.

Favor further annotation review if it helps identify history and inspect its scope without losing
source fidelity. Disfavor automatic attribution where mixed parentheticals, ordinary citations,
unbalanced punctuation or different publication conventions make the boundary uncertain.

In [ ]:
# Focused detector tests: no legal-status/date interpretation and exact Unicode spans.
positive="Чл. 1. (Изм. - ДВ, бр. 65 от 2003 г.) Текст."
pairs,_=history_candidates(positive)
require(len(pairs)==1 and positive[pairs[0][0]:pairs[0][1]]=="(Изм. - ДВ, бр. 65 от 2003 г.)", "Inline-note detection failed.")
for text in ("", "(1) Текст", "(чл. 7, ал. 3)", "(2003 г.)", "(ДВ, бр. 65 от 2003 г.)"):
    require(not history_candidates(text)[0],"Ordinary numbering/citation/date incorrectly asserted as history.")
nested="(Изм. (уточнение) - ДВ, бр. 1)"
require(history_candidates(nested)[0]==[(0,len(nested))],"Nested note boundaries changed.")
require(not history_candidates("(Изм. - ДВ, бр. 1")[0] and history_candidates("(Изм. - ДВ, бр. 1")[1],"Unclosed note silently accepted.")
require(len(history_candidates("(Нов - ДВ, бр. 1) (Доп. - ДВ, бр. 2)")[0])==2,"Adjacent notes merged.")
require(len(H_BY_ID)==len(H_ANNOTATIONS),"Duplicate annotation IDs.")
for a in H_ANNOTATIONS:
    block=BLOCKS[a["block_id"]]
    require(a["verbatim_text"]==block["raw_text"][a["block_start"]:a["block_end"]]
            ==VIEWS[a["document_id"]]["text"][a["start"]:a["end"]]==reconstruct(a["source_spans"]),"Annotation source mismatch.")
    require(a["id"]=="history:"+fingerprint([H_RUN_ID,a["document_id"],a["block_id"],a["block_start"],a["block_end"]]),"Annotation identity changed.")
for did in BASELINE_DOCUMENT_IDS:
    view={"document_id":did,"start":0,"end":len(VIEWS[did]["text"])}
    require(history_representation(view)["text"]==history_representation(view,True)["text"]==VIEWS[did]["text"],"Representation changed legal text.")
for rows in (H_SAMPLE_SHARES,H_UNIT_SHARES,H_FIXED_SHARES):
    require(all(0<=r["annotated_characters"]<=r["characters"] for r in rows),"Share double counts characters.")
require(interval_union_length([(0,4),(2,6)])==6,"Union share accounting changed.")
require(all(Path(path).read_bytes()==data for path,data in H_APPROVED_BYTES.items()),"Approved report or manifest changed.")
require(H_BASELINE_FINGERPRINTS==[fingerprint(FIXED_RESULTS),fingerprint(STRUCTURAL_RESULTS),fingerprint(PC_DATA),fingerprint(list(OS_GROUPS.values()))],"Approved chunk results changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()==artifact_sha256 and fingerprint(corpus)==corpus_fingerprint_before,"Stage 1 changed.")
old=H_SPEC["version"]
try:
    H_SPEC["version"]=999
    expect_value_error(ensure_history_current,"History configuration/annotations changed")
finally:H_SPEC["version"]=old
ensure_history_current()
print("PASS:",len(H_ANNOTATIONS),"exact annotations; unchanged representations, source maps, IDs, union shares, candidate edge cases and approved artifacts.")

### 9e. Review and stop after Step 9

Review the two explicit publication lists, the normal and long Article notes, share tables and
boundary witnesses. Verify the highlighted text against source offsets. Treat automatic inline
matches as candidates and inspect the uncertainty ledger before considering broader coverage.
The report preserves candidate text, proposed scope and uncertainty separately from all source
and chunk artifacts. It includes no parsed effective dates or legal-status assertions.

**Approved checkpoint — Step 9 complete.** Step 10 below explores metadata views only. No final
history/metadata policy, strategy comparison, Hybrid, embeddings, retrieval, RAG or agents are implemented.

In [ ]:
ensure_history_current()
H_REPORT={"report_version":1,"scope":"Step 9 only — non-destructive history investigation","run_id":H_RUN_ID,
    "spec":H_SPEC,"annotation_fingerprint":H_ANNOTATION_FINGERPRINT,"annotations":H_ANNOTATIONS,
    "uncertainty_ledger":H_UNCERTAINTY,"sample_shares":H_SAMPLE_SHARES,"structural_unit_shares":H_UNIT_SHARES,
    "fixed_chunk_shares":H_FIXED_SHARES,"boundary_witnesses":H_WITNESSES,
    "validation":"PASS: unchanged original text, exact source mappings, union shares, deterministic annotation IDs",
    "decision":"No history policy selected; scope and legal interpretation require review"}
history_report_path=OUTPUT_DIR/f"history_annotation_report.{fingerprint(H_REPORT)}.json"
history_bytes=(canonical(H_REPORT)+"\n").encode("utf-8")
if history_report_path.exists():require(history_report_path.read_bytes()==history_bytes,"History report differs.")
else:history_report_path.write_bytes(history_bytes)
require(json.loads(history_report_path.read_text(encoding="utf-8"))==H_REPORT,"History report round trip failed.")
print("History investigation report:",history_report_path.relative_to(ROOT))
print("STOP AFTER STEP 9 — unchanged text and separate candidate annotations ready for review.")

## Step 10 — Metadata experiments (stop here for review)

**What:** An experimental document registry and a minimal-versus-joined view of the **same existing
child chunk**. The text, chunk ID, boundaries and parent choices remain unchanged. This is a
notebook experiment contract, not a production schema, public API or filtering/search implementation.

**Why:** Metadata can make source identity, citation, uncertain boundaries and generation choices
inspectable. Copying long document fields onto every chunk is unnecessary for these displays.

**How:** Preserve all 21 Stage 1 document descriptors' original title and metadata records in a
registry. Only the four existing baseline documents have chunk experiments; the registry does not
extend the corpus-building scope. Keep parser/source fingerprints at run level. Resolve chunk data,
structural evidence and explicit relationship records through their existing IDs.

| Category | Contents and owner | Inspection purpose |
|---|---|---|
| A. Document | Registry: Bulgarian title, English subtitle/title, jurisdiction, URL, official reference, version/accessed dates, note and source evidence | Identification, display and possible future filtering |
| B. Structural | Existing unit and boundary-ledger references, labels, scope evidence, heading locations and uncertainty | Boundary debugging and qualified citations |
| C. Chunk-generated | Existing child ID, generation configuration, order, character size, source mappings, actual overlap and split mechanism | Traceability and reproducibility |
| D. Parent-child | Explicit relationship and set IDs, parent ID/span, child order and relationship mode | Resolve the selected context without assigning one global parent to a shared child |
| E. Provenance/citation | Registry/run references, artifact and DOCX hashes, source URL, block offsets and evidenced legal labels | Source verification and readable source-location citations |

Dates remain exact strings. **Version date is not effective date.** English titles are display
values, never prefixes added to chunk text. No mentor-reference topic, city, district, `in_force`,
physical page values or unsupported attributes are imported or inferred.

In [ ]:
from urllib.parse import urlsplit
ensure_history_current()
M_APPROVED_PATHS=[APPROVED_MANIFEST_PATH,report_path,structural_report_path,pc_report_path,os_report_path,history_report_path]
M_APPROVED_BYTES={str(p):p.read_bytes() for p in M_APPROVED_PATHS}
M_BASELINE_FINGERPRINTS=[fingerprint(FIXED_RESULTS),fingerprint(STRUCTURAL_RESULTS),fingerprint(PC_DATA),fingerprint(list(OS_GROUPS.values())),fingerprint(H_ANNOTATIONS)]
M_LABELS={"jurisdiction":"Юрисдикция / Jurisdiction","source_url":"Източник / Source URL",
          "official_reference":"ДВ / Official reference","version_date":"Версия / Version date",
          "accessed_date":"Достъпено / Accessed","note":"Бележка / Note"}
M_SPEC={"version":1,"setup_id":SETUP_ID,"artifact_sha256":artifact_sha256,
        "labels":M_LABELS,"registry_population":"all 21 Stage 1 document descriptors",
        "chunk_population":"existing Step 2 baseline documents only",
        "representative_sample":"long_article","display_configuration":GALLERY_CONFIGURATION,
        "date_interpretation":"verbatim source only; version date is not effective date",
        "production_schema":False}
M_RUN_ID=fingerprint(M_SPEC)
M_RUN={"id":M_RUN_ID,"artifact_sha256":artifact_sha256,"source":json.loads(canonical(corpus["source"])),
       "parser":json.loads(canonical(corpus["parser"])),"setup_id":SETUP_ID,
       "fixed_run_id":BASELINE_ID,"structural_run_id":STRUCTURAL_ID,"parent_child_run_id":PC_RUN_ID}
M_REGISTRY={}
for did,document in DOCUMENTS.items():
    # Copy source records, not reconstructed or normalized values; retain labels and table coordinates.
    metadata=json.loads(canonical(document["source_metadata"]))
    index={field:[i for i,row in enumerate(metadata["rows"])
                  if [p["raw_text"] for p in row["label_paragraphs"]]==[label]] for field,label in M_LABELS.items()}
    M_REGISTRY[did]={"document_id":did,"ordinal":document["ordinal"],
        "source_titles":json.loads(canonical(document["source_titles"])),"source_metadata":metadata,
        "field_row_references":index,"run_ref":M_RUN_ID}
M_REGISTRY_FINGERPRINT=fingerprint(M_REGISTRY)


def registry_field(did,field):
    require(did in M_REGISTRY,"Unknown registry document.")
    require(field in M_LABELS,"Unsupported registry field.")
    doc=M_REGISTRY[did]
    return [{"row_reference":i,"row_index":doc["source_metadata"]["rows"][i]["row_index"],
             "value_paragraphs":doc["source_metadata"]["rows"][i]["value_paragraphs"]}
            for i in doc["field_row_references"][field]]


def raw_values(did,field):
    # A list preserves multiple paragraphs/rows and blank values without invented separators.
    return [p["raw_text"] for row in registry_field(did,field) for p in row["value_paragraphs"]]


show_table([{"document":d["ordinal"],"document_id":did,
    "Bulgarian_title":d["source_titles"]["title"]["raw_text"],
    "English_title":d["source_titles"]["subtitle"]["raw_text"],
    "jurisdiction_raw":raw_values(did,"jurisdiction"),"version_date_raw":raw_values(did,"version_date"),
    "missing_or_duplicate_rows":{field:len(refs) for field,refs in d["field_row_references"].items() if len(refs)!=1}}
    for did,d in M_REGISTRY.items()])
M_SAMPLE=SAMPLE_BY_ID[M_SPEC["representative_sample"]]
show_json("Complete representative document registry record — all original metadata and source locations",M_REGISTRY[M_SAMPLE["document_id"]])
show_json("Run-level source/DOCX fingerprints and parser provenance",M_RUN)

### 10a. Source field inventory and selective resolution

`field_row_references` indexes the original table rows; it does not replace their text. Values
remain lists of original paragraphs so multiple or missing values stay visible. An absent field
would resolve to an empty list, not an invented default. The table below shows every requested
field for the representative document, including access date, note and official reference.
Their long strings stay in the registry and are looked up for display rather than copied onto
canonical chunk records. Source locations identify DOCX XML parts/paths and body blocks, not
physical pages. No URLs are fetched: a syntactically linkable source URL is not a verified live URL.

In [ ]:
M_FIELD_INVENTORY=[]
for field in M_LABELS:
    entries=registry_field(M_SAMPLE["document_id"],field)
    M_FIELD_INVENTORY.append({"field":field,"raw_values":raw_values(M_SAMPLE["document_id"],field),
        "registry_row_references":[e["row_reference"] for e in entries],
        "source_evidence":[{"block_id":p["block_id"],"source":p["source"]} for e in entries for p in e["value_paragraphs"]],
        "placement":"selective join: possible future filter" if field in ("jurisdiction","version_date") else "registry lookup"})
show_table(M_FIELD_INVENTORY)

### 10b. The same chunk, two metadata views

Use an existing fixed-length child inside the long Article parent, under the existing 2,000-character /
10%-overlap display setting. This is a representative inspection choice, not a preferred configuration.
The **minimal view** stores only IDs/references and the absolute document span. The **joined view**
resolves the requested relationship explicitly, then adds selected inspection fields grouped by A–E.
A shared child may have several relationship records, so a child ID alone must not silently choose
its parent, order or configuration.

The joined document fields include raw jurisdiction and version date as *possible future filter
inputs*. No filter runs here; their coverage, normalization needs and semantics require later review.
The generated citation labels a provisional unit and gives source positions and character offsets.
It does not claim the child contains a complete Article or manufacture a Section/Chapter hierarchy.

In [ ]:
M_SELECTED_GROUP=next(g for g in PC_GROUPS.values() if g["mode"]=="fixed_inside_parent"
    and g["configuration"]==M_SPEC["display_configuration"]
    and PC_PARENTS[g["parent_id"]]["document_id"]==M_SAMPLE["document_id"]
    and PC_PARENTS[g["parent_id"]]["start"]<=M_SAMPLE["start"]<PC_PARENTS[g["parent_id"]]["end"])
M_SELECTED_LINK=PC_LINKS[M_SELECTED_GROUP["relationship_ids"][min(1,len(M_SELECTED_GROUP["relationship_ids"])-1)]]
M_MINIMAL={"chunk_id":M_SELECTED_LINK["child_id"],"document_id":PC_CHILDREN[M_SELECTED_LINK["child_id"]]["document_id"],
    "document_span":[PC_CHILDREN[M_SELECTED_LINK["child_id"]]["start"],PC_CHILDREN[M_SELECTED_LINK["child_id"]]["end"]],
    "relationship_id":M_SELECTED_LINK["id"],"run_ref":M_RUN_ID}


def join_metadata(minimal):
    require(minimal["run_ref"]==M_RUN_ID,"Stale metadata run reference.")
    require(minimal["chunk_id"] in PC_CHILDREN,"Unknown chunk reference.")
    require(minimal["relationship_id"] in PC_LINKS,"Unknown relationship reference.")
    child=PC_CHILDREN[minimal["chunk_id"]];link=PC_LINKS[minimal["relationship_id"]]
    require(link["child_id"]==child["id"],"Relationship belongs to another child.")
    require(minimal["document_id"]==child["document_id"] and minimal["document_span"]==[child["start"],child["end"]],"Chunk reference/span mismatch.")
    group=PC_GROUPS[link["relationship_set_id"]];parent=PC_PARENTS[link["parent_id"]]
    did=child["document_id"];doc=M_REGISTRY[did];unit=STRUCTURAL_BY_ID[link["source_structural_unit_id"]]
    ledger=next(row for row in BOUNDARY_LEDGERS[did] if row["id"]==unit["opening_ledger_id"])
    ordered=[PC_CHILDREN[PC_LINKS[rid]["child_id"]] for rid in group["relationship_ids"]]
    index=link["ordinal"]-1
    previous=intersection_length((ordered[index-1]["start"],ordered[index-1]["end"]),(child["start"],child["end"])) if index else 0
    following=intersection_length((ordered[index+1]["start"],ordered[index+1]["end"]),(child["start"],child["end"])) if index+1<len(ordered) else 0
    mapped=source_map(did,child["start"],child["end"])
    positions=sorted({s["position"] for s in mapped if s["kind"]=="source"})
    source_urls=raw_values(did,"source_url")
    citation=f"{doc['source_titles']['title']['raw_text']} — {ledger['label']} (provisional source label); body positions {positions}; document characters [{child['start']}, {child['end']})"
    return {"A_document_join":{"document_registry_ref":did,
                "Bulgarian_title":doc["source_titles"]["title"],"English_title":doc["source_titles"]["subtitle"],
                "jurisdiction_raw":raw_values(did,"jurisdiction"),"version_date_raw":raw_values(did,"version_date"),
                "filter_field_evidence":{"jurisdiction":registry_field(did,"jurisdiction"),"version_date":registry_field(did,"version_date")}},
        "B_structural":{"unit_id":unit["id"],"provisional_kind":unit["unit_kind"],"unit_span":[unit["start"],unit["end"]],
                "opening_boundary_evidence":ledger,"closing_ledger_ref":unit["closing_ledger_id"],
                "scope_evidence_block_id_not_parent":unit["scope_evidence_block_id_not_parent"],"uncertainty":unit["uncertainty"],
                "child_is_complete_unit":child["start"]==unit["start"] and child["end"]==unit["end"]},
        "C_chunk_generated":{"id":child["id"],"strategy":"parent-child relationship experiment",
                "configuration_id":group["configuration"],"ordinal_in_selected_relationship":link["ordinal"],
                "character_count":child["end"]-child["start"],"source_spans":mapped,
                "actual_previous_overlap":previous,"actual_next_overlap":following,
                "split_mechanism_from_relationship_mode":group["mode"],"setup_id":SETUP_ID},
        "D_parent_child":{"relationship_id":link["id"],"relationship_set_id":group["id"],"parent_id":parent["id"],
                "child_order":link["ordinal"],"relationship_type":group["mode"],"parent_source_span":[parent["start"],parent["end"]],
                "parent_source_map":parent["source_spans"],"parent_role":parent["parent_role"],
                "parent_review_basis":parent["review_basis"],"parent_uncertainty":parent.get("uncertainty", "See review basis and structural evidence; legal meaning unverified")},
        "E_provenance_citation":{"document_registry_ref":did,"run_ref":M_RUN_ID,"artifact_sha256":M_RUN["artifact_sha256"],
                "docx_sha256":M_RUN["source"]["sha256"],"source_urls_raw":source_urls,
                "official_reference_registry_rows":doc["field_row_references"]["official_reference"],
                "source_block_ids":[s["block_id"] for s in mapped if s["kind"]=="source"],
                "citation_display":citation,"url_verification":"Source-supplied; not fetched or independently verified"}}


M_JOINED=join_metadata(M_MINIMAL)
M_CHILD_TEXT=PC_CHILDREN[M_MINIMAL["chunk_id"]]["text"]
display(HTML("<h4>One unchanged chunk payload (shared by both metadata views)</h4>"+details_html("Complete verbatim child text",M_CHILD_TEXT)))
display(HTML("<div style='display:grid;grid-template-columns:1fr 1fr;gap:16px'><section><h4>Minimal metadata</h4>"
    +details_html("Minimal references",json.dumps(M_MINIMAL,ensure_ascii=False,indent=2))
    +"</section><section><h4>Selectively enriched / joined metadata</h4>"
    +"".join(details_html(category,json.dumps(value,ensure_ascii=False,indent=2)) for category,value in M_JOINED.items())+"</section></div>"))
show_table([{"category":key,"fields":list(value)} for key,value in M_JOINED.items()])
display(HTML("<p>"+esc(M_JOINED["E_provenance_citation"]["citation_display"])+"</p>"))
for url in M_JOINED["E_provenance_citation"]["source_urls_raw"]:
    parsed=urlsplit(url)
    if parsed.scheme in ("https","http") and parsed.netloc:
        display(HTML('<p>Source-supplied URL: <a href="'+html.escape(url,quote=True)+'" rel="noopener noreferrer">'+esc(url)+'</a> (not fetched)</p>'))
    else:display(HTML("<p>Source URL retained as text (not a validated HTTP link): "+esc(url)+"</p>"))

### 10c. Split reasons are generated evidence; source fields are not

The selected Step 7 relationship records its existing splitting *mode*, not a size-trigger policy.
For contrast in **metadata representation only**, the table below reads the already-produced Step 8
child reasons, requested/actual overlap and parent links for this same Article. It does not run a
new experiment or compare strategies. The `split_reason` and `next_start_reason` fields describe
algorithm decisions, whereas titles, dates and notes belong to source metadata.

Official reference, access date, full note and parser details are available by document/run lookup.
They are not added to the minimal child. English titles remain available for bilingual display.
Repeated § labels need document and source-location context; neither a label nor a date is a unique
legal identifier. Unclassified spans retain their evidenced status; metadata does not repair an
uncertain legal hierarchy.

In [ ]:
M_REASON_EXAMPLES=[]
selected_unit=M_SELECTED_LINK["source_structural_unit_id"]
for method in ("exact","boundary_aware"):
    group=OS_GROUPS.get((selected_unit,f"{method}-chars-2000-overlap-10pct"))
    if group:
        child=group["children"][0]
        M_REASON_EXAMPLES.append({k:child[k] for k in ("id","parent_id","configuration","split_reason","next_start_reason",
                                                       "requested_overlap","actual_previous_overlap","actual_next_overlap")})
show_table(M_REASON_EXAMPLES)
M_PLACEMENT=[
 {"fields":"Bulgarian / English titles","owner":"document registry","use":"bilingual display, document identification and citation","caution":"Never prepend to original chunk text"},
 {"fields":"jurisdiction; raw version date","owner":"registry; selective joined view","use":"possible future filtering","caution":"Raw version is not effective date; semantics/coverage need review"},
 {"fields":"URL; official reference","owner":"document registry","use":"source navigation and citation lookup","caution":"Source-provided references do not verify current legal status or URL reachability"},
 {"fields":"accessed date; long note","owner":"document registry","use":"source collection context and limitations","caution":"Not provision-specific dates or demonstrated retrieval filter needs"},
 {"fields":"labels; scope; heading locations; uncertainty","owner":"structural unit and boundary ledger","use":"citation context and boundary debugging","caution":"Provisional source evidence, not manufactured ancestry"},
 {"fields":"IDs; spans; configuration; overlaps; split reasons","owner":"chunk/experiment records","use":"traceability, reproducibility and debugging","caution":"Character counts are not tokens; generation evidence is not legal evidence"},
 {"fields":"parent link; order; relationship mode","owner":"explicit relationship set","use":"context resolution and containment","caution":"Shared children may have multiple parent alternatives"},
 {"fields":"DOCX/artifact fingerprints; parser provenance","owner":"run record","use":"verify exactly which source and parser produced the experiment","caution":"Resolve by run reference; no need to repeat parser details per child"}]
show_table(M_PLACEMENT)

### 10d. Validate references and source fidelity

Check every registry field against the original descriptor, including raw dates, notes and metadata
paragraph locations. Check the representative join's source map, relationship order, containment and
citation references, and reject mismatched IDs. The enrichment is a read-only projection: it must
not mutate canonical chunks, parent links, history annotations or prior reports. These assertions
validate representation and traceability, not metadata completeness or legal correctness.

In [ ]:
def ensure_metadata_current():
    ensure_history_current()
    require(fingerprint(M_SPEC)==M_RUN_ID and fingerprint(M_REGISTRY)==M_REGISTRY_FINGERPRINT,
            "Metadata settings or registry changed; rebuild Step 10 views.")
    require(M_JOINED==join_metadata(M_MINIMAL),"Joined view is stale; rebuild Step 10 display.")


require(len(M_REGISTRY)==len(DOCUMENTS)==21,"Document registry population changed.")
for did,doc in M_REGISTRY.items():
    original=DOCUMENTS[did]
    require(doc["source_titles"]==original["source_titles"] and doc["source_metadata"]==original["source_metadata"],"Registry changed source fields or evidence.")
    for field,label in M_LABELS.items():
        expected=[p["raw_text"] for row in original["source_metadata"]["rows"] if [p["raw_text"] for p in row["label_paragraphs"]]==[label]
                  for p in row["value_paragraphs"]]
        require(raw_values(did,field)==expected,"Field value normalized or dropped.")
require(M_CHILD_TEXT==reconstruct(M_JOINED["C_chunk_generated"]["source_spans"])
        ==VIEWS[M_MINIMAL["document_id"]]["text"][slice(*M_MINIMAL["document_span"])],"Joined representation changed source text.")
link=PC_LINKS[M_MINIMAL["relationship_id"]];parent=PC_PARENTS[link["parent_id"]]
require(parent["document_id"]==M_MINIMAL["document_id"] and parent["start"]<=M_MINIMAL["document_span"][0]<M_MINIMAL["document_span"][1]<=parent["end"],"Relationship containment failed.")
require(M_JOINED["D_parent_child"]["child_order"]==link["ordinal"] and PC_GROUPS[link["relationship_set_id"]]["relationship_ids"][link["ordinal"]-1]==link["id"],"Relationship order mismatch.")
require(M_JOINED["E_provenance_citation"]["docx_sha256"]==corpus["source"]["sha256"],"DOCX fingerprint mismatch.")
expect_value_error(lambda:registry_field("unknown","note"),"Unknown registry document")
expect_value_error(lambda:registry_field(M_MINIMAL["document_id"],"in_force"),"Unsupported registry field")
expect_value_error(lambda:join_metadata({**M_MINIMAL,"document_span":[0,1]}),"Chunk reference/span mismatch")
expect_value_error(lambda:join_metadata({**M_MINIMAL,"relationship_id":"unknown"}),"Unknown relationship reference")
other_link=next(r for r in PC_LINKS.values() if r["child_id"]!=M_MINIMAL["chunk_id"])
expect_value_error(lambda:join_metadata({**M_MINIMAL,"relationship_id":other_link["id"]}),"Relationship belongs to another child")
require(set(M_MINIMAL)=={"chunk_id","document_id","document_span","relationship_id","run_ref"},"Minimal view copied document fields.")
M_FORBIDDEN={"topic","city","district","in_force","page","pages","page_number","effective_date"}
def check_no_unsupported_fields(value):
    if isinstance(value,dict):
        require(not M_FORBIDDEN.intersection(value),"Unsupported metadata field introduced.")
        for v in value.values():check_no_unsupported_fields(v)
    elif isinstance(value,list):
        for v in value:check_no_unsupported_fields(v)
check_no_unsupported_fields(M_REGISTRY);check_no_unsupported_fields(M_MINIMAL);check_no_unsupported_fields(M_JOINED)
require(all(Path(p).read_bytes()==data for p,data in M_APPROVED_BYTES.items()),"An approved report or manifest changed.")
require(M_BASELINE_FINGERPRINTS==[fingerprint(FIXED_RESULTS),fingerprint(STRUCTURAL_RESULTS),fingerprint(PC_DATA),fingerprint(list(OS_GROUPS.values())),fingerprint(H_ANNOTATIONS)],"Canonical chunks/evidence changed.")
require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()==artifact_sha256 and fingerprint(corpus)==corpus_fingerprint_before,"Stage 1 source changed.")
ensure_metadata_current()
print("PASS: 21 source-faithful registry records, raw dates, exact joined chunk, valid explicit relationship, provenance and unchanged approved artifacts.")

### 10e. Review and stop

Inspect the original registry record and every source field, then open both metadata panels for
the same child. The joined view should let you identify its document, raw version, provisional
legal context, parent alternative and exact source location without changing the payload. Decide
which fields actually help inspection and whether the long notes/official-reference strings need
anything more than registry lookup. Possible future filters still need semantics and coverage
review; no retrieval benefit has been measured. No production schema or metadata policy is chosen.

**Approved checkpoint — Step 10 complete.** Step 11 below compares existing results only. Hybrid,
embeddings, vector storage, retrieval, RAG and agents remain unimplemented.

In [ ]:
ensure_metadata_current()
M_REPORT={"report_version":1,"scope":"Step 10 only — metadata representations","run_id":M_RUN_ID,"spec":M_SPEC,
    "run_record":M_RUN,"document_registry":M_REGISTRY,"registry_fingerprint":M_REGISTRY_FINGERPRINT,
    "representative_sample":M_SPEC["representative_sample"],"minimal_view":M_MINIMAL,"joined_view":M_JOINED,
    "source_field_inventory":M_FIELD_INVENTORY,"existing_split_reason_examples":M_REASON_EXAMPLES,
    "placement_observations":M_PLACEMENT,"validation":"PASS: registry fidelity, raw dates, reference joins, unchanged payload and approved artifacts",
    "decision":"Experimental views only; no production schema or final metadata policy selected"}
metadata_report_path=OUTPUT_DIR/f"metadata_experiment_report.{fingerprint(M_REPORT)}.json"
metadata_report_bytes=(canonical(M_REPORT)+"\n").encode("utf-8")
if metadata_report_path.exists():require(metadata_report_path.read_bytes()==metadata_report_bytes,"Existing metadata report differs.")
else:metadata_report_path.write_bytes(metadata_report_bytes)
require(json.loads(metadata_report_path.read_text(encoding="utf-8"))==M_REPORT,"Metadata report round trip failed.")
print("Metadata experiment report:",metadata_report_path.relative_to(ROOT))
print("STOP AFTER STEP 10 — registry and joined views ready for review; no production schema selected.")

## Step 11 — Side-by-side strategy comparison, then mentor references

**What:** Inspect the same 12 frozen source spans using already-generated fixed-length, uncapped
structural and parent-child results. No new chunks or strategy are created. The gallery uses the
existing 2,000-character / 10%-overlap setting for comparability, not as a recommendation. Tables
retain the earlier all-setting results and oversized-unit, history and metadata evidence.

**Why:** Preservation checks alone do not tell us whether a child keeps a condition beside its
exception, or whether a parent supplies too much unrelated text. Separate mechanical evidence
from legal/context judgments. There is no weighted winner score or production-strategy decision.

**How to review:** source above, three aligned panels below; expand full texts and mappings.
The same frozen sample is highlighted in every strategy. Extra text remains visible and its size
is reported. Parent choice is explicit: prefer reviewed smaller parents whose union covers the
sample; otherwise show the smallest existing enclosing parent. Heading-plus-provision samples
can need several parents. Large enclosing parents are not truncated or silently substituted.
Alternative parent evidence remains in a separate table, not mixed into overlap metrics.

| Rubric | Meaning here |
|---|---|
| PASS | The stated, bounded check is supported by source spans or a validated invariant |
| CONCERN | A visible boundary/context/size trade-off needs review; not proof of legal error |
| FAIL | A specific preservation/containment requirement fails (none should pass validation if so) |
| UNCERTAIN | Available source evidence cannot establish legal completeness, attribution or retrieval benefit |

Ratings below are initial reviewer observations with source IDs/offsets, not legal certification.
Mentor loading occurs only after these independent observations are frozen.

In [ ]:
ensure_metadata_current()
C_APPROVED_PATHS=[APPROVED_MANIFEST_PATH,report_path,structural_report_path,pc_report_path,os_report_path,history_report_path,metadata_report_path]
C_APPROVED_BYTES={str(p):p.read_bytes() for p in C_APPROVED_PATHS}
def comparison_input_fingerprint():
    return fingerprint([FIXED_RESULTS,STRUCTURAL_RESULTS,PC_DATA,list(OS_GROUPS.values()),H_ANNOTATIONS,M_REGISTRY])
C_INPUT_FINGERPRINT=comparison_input_fingerprint()
C_SPEC={"version":1,"setup_id":SETUP_ID,"input_fingerprint":C_INPUT_FINGERPRINT,
        "display_configuration":GALLERY_CONFIGURATION,"samples":[s["sample_id"] for s in BASELINE_SAMPLES],
        "parent_choice":"smaller reviewed parents covering sample, else smallest existing enclosing parent",
        "new_strategy":False,"weighted_score":None}
C_RUN_ID=fingerprint(C_SPEC)
C_SELECTIONS={}
for sample in BASELINE_SAMPLES:
    did,lo,hi=sample["document_id"],sample["start"],sample["end"]
    fixed=[c for c in FIXED_RESULTS[GALLERY_CONFIGURATION][did] if intersection_length((lo,hi),(c["start"],c["end"]))]
    units=[u for u in STRUCTURAL_RESULTS[did] if intersection_length((lo,hi),(u["start"],u["end"]))]
    parents=[p for p in PC_PARENTS.values() if p["document_id"]==did and p["family"]!="section_or_chapter"
             and intersection_length((lo,hi),(p["start"],p["end"]))]
    covered=interval_union_length([(max(lo,p["start"]),min(hi,p["end"])) for p in parents])
    if covered<hi-lo:
        enclosing=[p for p in PC_PARENTS.values() if p["document_id"]==did and p["start"]<=lo and hi<=p["end"]]
        if enclosing:parents=[min(enclosing,key=lambda p:(p["end"]-p["start"],p["id"]))]
    parents.sort(key=lambda p:p["start"])
    groups=[PC_GROUP_BY_KEY[(p["case"],"fixed_inside_each_contained_unit" if p["family"]=="section_or_chapter" else "fixed_inside_parent",GALLERY_CONFIGURATION)] for p in parents]
    C_SELECTIONS[sample["sample_id"]]={"fixed":fixed,"structural":units,"parent_group_ids":[g["id"] for g in groups]}


def clipped_coverage(records,sample):
    return interval_union_length([(max(sample["start"],r["start"]),min(sample["end"],r["end"])) for r in records
        if r["document_id"]==sample["document_id"] and intersection_length((sample["start"],sample["end"]),(r["start"],r["end"]))])


def comparison_metrics(sample,strategy,records,parents=()):
    intervals=[(r["start"],r["end"]) for r in records]
    unique=interval_union_length(intervals)
    sizes=[r["end"]-r["start"] for r in records]
    cut_units={u for r in records for u in legal_diagnostic(r)["cut"]}
    return {"sample_id":sample["sample_id"],"strategy":strategy,"records":len(records),
        "sample_coverage_pct":round(100*clipped_coverage(records,sample)/sample["characters"],2),
        "size_distribution":length_summary(sizes),"unique_source_characters_in_displayed_records":unique,
        "extra_characters_outside_sample":unique-clipped_coverage(records,sample),
        "overlap_duplicate_characters":sum(sizes)-unique,
        "paragraph_blocks_cut":len({b for r in records for b in paragraph_cuts(r)}),
        "reviewed_legal_units_cut":sorted(cut_units),
        "records_combining_reviewed_units":sum(len(legal_diagnostic(r)["touched"])>1 for r in records),
        "parent_sizes":[p["end"]-p["start"] for p in parents],
        "parent_containment_not_overlap":bool(parents),
        "source_evidence":[{"id":r["id"],"document_id":r["document_id"],"start":r["start"],"end":r["end"]} for r in records]}
C_METRICS=[]
for sample in BASELINE_SAMPLES:
    selection=C_SELECTIONS[sample["sample_id"]]
    for strategy in ("fixed","structural"):
        C_METRICS.append(comparison_metrics(sample,strategy,selection[strategy]))
    parents=[];children=[]
    for gid in selection["parent_group_ids"]:
        parent,kids=relationship_display_records(gid);parents.append(parent);children.extend(kids)
    C_METRICS.append(comparison_metrics(sample,"parent_child",children,parents))
show_table([{k:v for k,v in row.items() if k not in ("source_evidence","reviewed_legal_units_cut")} for row in C_METRICS])

### 11a. Aligned source and strategy panels

The gallery includes all frozen samples. Fixed-length and structural panels show every intersecting
record; parent-child panels retain complete selected parents and **all** their ordered children.
Thus the amount of displayed context differs intentionally, while the blue focus is always the
same source span. Size/extra-text counts use the complete displayed records, not sample-clipped
payloads. Long-unit handling is visible without replacing the uncapped structural control.

In [ ]:
def show_strategy_comparison(sample_id):
    sample=SAMPLE_BY_ID[sample_id];selection=C_SELECTIONS[sample_id]
    source=VIEWS[sample["document_id"]]["text"][sample["start"]:sample["end"]]
    panels=[]
    for strategy in ("fixed","structural"):
        records=selection[strategy]
        body="".join(span_card(r,sample,records,REVIEWED_PROVISIONS) for r in records)
        panels.append("<section><h4>"+esc(strategy)+"</h4>"+body+"</section>")
    body=""
    for gid in selection["parent_group_ids"]:
        parent,children=relationship_display_records(gid)
        body+=details_html("Explicit parent choice and relationship set",json.dumps({"group_id":gid,"case":PC_GROUPS[gid]["case"],"parent_id":parent["id"],"review_basis":parent["review_basis"]},ensure_ascii=False,indent=2))
        body+=relationship_html(parent,children,sample,REVIEWED_PROVISIONS)
    panels.append("<section><h4>parent-child</h4>"+body+"</section>")
    display(HTML("<details><summary>"+esc(sample_id)+" — identical source focus across three strategies</summary>"
        +"<h4>Frozen source span</h4>"+highlight_html(source,sample["start"])
        +details_html("Frozen source mappings",json.dumps(sample["source_spans"],ensure_ascii=False,indent=2))
        +"<div style='display:grid;grid-template-columns:repeat(3,minmax(320px,1fr));gap:12px;overflow-x:auto'>"+"".join(panels)+"</div></details>"))
for sample in BASELINE_SAMPLES:show_strategy_comparison(sample["sample_id"])
show_table([r for r in PC_GROUP_DIAGNOSTICS if r["configuration"]==GALLERY_CONFIGURATION])

### 11b. Independent evidence and initial reviewer rubric

**Overlap duplication** counts repeat coverage of the same source positions among peer children
or fixed windows. **Parent-child containment** intentionally offers the same positions at two
context scales and is reported as expansion, never added to peer overlap. **Source repetition**
means identical wording at *different* source locations; it remains source evidence. The repeated
paragraph inventory below compares exact nonempty raw blocks within each document (at least
40 characters), without normalizing text. It does not measure semantic redundancy.

Mechanical PASS ratings do not imply legally correct boundaries. The reviewed provision ledger
covers only selected units. Legal completeness and useful context remain UNCERTAIN when the
source has ambiguous headings or a chunk requires outside definitions/cross-references. Size
thresholds are probes, not token budgets. History shares measure volume, not retrieval noise.

In [ ]:
C_SOURCE_REPETITION=[]
for did in BASELINE_DOCUMENT_IDS:
    occurrences=defaultdict(list)
    for bid in VIEWS[did]["block_ids"]:
        if len(BLOCKS[bid]["raw_text"])>=40:occurrences[BLOCKS[bid]["raw_text"]].append(bid)
    repeated=[(text,bids) for text,bids in occurrences.items() if len(bids)>1]
    C_SOURCE_REPETITION.append({"document_id":did,"exact_repeated_paragraph_groups":len(repeated),
        "extra_occurrence_characters":sum(len(text)*(len(bids)-1) for text,bids in repeated),
        "examples":[{"verbatim_text":text,"source_block_ids":bids} for text,bids in repeated[:3]],
        "meaning":"distinct source positions; excluded from overlap duplication"})
show_table(C_SOURCE_REPETITION)
C_RUBRIC=[]
for row in C_METRICS:
    evidence={"sample_id":row["sample_id"],"source_spans":row["source_evidence"]}
    ratings=[("source preservation","PASS" if row["sample_coverage_pct"]==100 else "FAIL",f"Exact reconstruction validated; displayed records cover {row['sample_coverage_pct']}% of frozen sample."),
        ("legal boundary quality","CONCERN" if row["reviewed_legal_units_cut"] or row["records_combining_reviewed_units"] else "UNCERTAIN",
         f"Reviewed units cut: {row['reviewed_legal_units_cut']}; records combining reviewed units: {row['records_combining_reviewed_units']}. Absence of flags is not complete legal validation."),
        ("chunk size","CONCERN" if (row["size_distribution"]["max"] or 0)>4000 else "PASS",
         f"Observed sizes {row['size_distribution']}; 4,000 is an experimental character probe, not a production limit."),
        ("duplication / overlap","CONCERN" if row["overlap_duplicate_characters"] else "PASS",
         f"{row['overlap_duplicate_characters']} repeated peer source characters. Parent containment and source repetition excluded."),
        ("context preservation","UNCERTAIN",f"{row['extra_characters_outside_sample']} distinct characters outside the focus are displayed. Inspect definitions, conditions, exceptions and cross-references; extra context need not be relevant."),
        ("parent expansion","CONCERN" if row["parent_sizes"] and max(row["parent_sizes"])>8000 else "UNCERTAIN",
         f"Complete parent sizes {row['parent_sizes']}; per-child expansion is in Step 7 relationship diagnostics. No parent for flat strategies; no future context budget established."),
        ("metadata usefulness","PASS","Document registry, raw dates, source labels/uncertainty and explicit links can be resolved (Step 10); filtering benefit remains untested."),
        ("traceability / provenance","PASS","Exact block offsets and artifact/DOCX fingerprints resolve through the existing source maps and run registry."),
        ("long-unit handling","CONCERN" if row["strategy"]=="structural" and (row["size_distribution"]["max"] or 0)>8000 else "UNCERTAIN",
         "Uncapped structural control remains visible; Step 8 validates contained fallback children. No size or fallback policy selected.")]
    for criterion,rating,explanation in ratings:
        C_RUBRIC.append({"sample_id":row["sample_id"],"strategy":row["strategy"],"criterion":criterion,"rating":rating,"explanation":explanation,"evidence":evidence})
show_table([{k:r[k] for k in ("sample_id","strategy","criterion","rating","explanation")} for r in C_RUBRIC
            if r["sample_id"] in ("normal_article","long_article","nested_structure")])
display(HTML("<details><summary>All samples: complete rubric and source-backed evidence</summary>"+table_html(C_RUBRIC)+"</details>"))
show_table(OS_THRESHOLD_ROWS)
show_table(OS_MATCHED_COMPARISONS)
show_table(H_SAMPLE_SHARES,["sample_id","characters","annotated_characters","annotated_share_pct"])
show_table(H_WITNESSES)
show_table(M_PLACEMENT)
C_INDEPENDENT={"spec":C_SPEC,"metrics":C_METRICS,"rubric":C_RUBRIC,"source_repetition":C_SOURCE_REPETITION,
    "parent_expansion":[r for r in PC_GROUP_DIAGNOSTICS if r["configuration"]==GALLERY_CONFIGURATION],
    "all_fixed_settings":CORPUS_DIAGNOSTICS,"oversized_comparisons":OS_MATCHED_COMPARISONS,
    "history_shares":H_SAMPLE_SHARES,"metadata_placement":M_PLACEMENT}
C_INDEPENDENT_FINGERPRINT=fingerprint(C_INDEPENDENT)
require(all(r["sample_coverage_pct"]==100 for r in C_METRICS),"A selected strategy representation does not cover the frozen sample.")
print("Independent comparison frozen before mentor loading:",C_INDEPENDENT_FINGERPRINT)

### 11c. Separate read-only mentor evaluation path

Only now read the reference JSONL inside the unchanged source archive. Nothing is extracted,
executed or passed to chunk builders. The evaluation allowlist excludes mentor IDs, timestamps,
`embedding_text`, topic/city/district/status/page fields and implementation metadata. Archive line
numbers below are evaluation locations, not identity comparisons. The 94 curated entries are not
an exhaustive ground truth; source-document coverage is not judged from their count.

Document correspondence uses reviewed exact titles plus source URLs. The only URL adjustment
allowed here is removing a fragment for document identity (EU Article fragments remain visible).
No fuzzy title mapping or internet lookups occur. A document match alone does not establish a
provision match. Unmatched/ambiguous cases remain unresolved, never evidence that source is missing.

Separate `[EN summary]` suffixes and `[EN original]` markers from excerpt text without changing the
stored reference. English original prose remains eligible source text; English summaries do not.
The read-only builder line documents the added title/section/heading prefix; its claims about
retrieval are not adopted. No `embedding_text` equality or content comparison is performed.

In [ ]:
import zipfile
from urllib.parse import urldefrag
require(fingerprint(C_INDEPENDENT)==C_INDEPENDENT_FINGERPRINT,"Complete independent review before mentor evaluation.")
C_ARCHIVE=ROOT/corpus["source"]["archive"]
C_ARCHIVE_SHA=hashlib.sha256(C_ARCHIVE.read_bytes()).hexdigest()
with zipfile.ZipFile(C_ARCHIVE,"r") as archive:
    reference_members=[name for name in archive.namelist() if name.endswith("/corpus/chunks.jsonl")]
    require(len(reference_members)==1,"Reference path must be unique.")
    C_REFERENCE_MEMBER=reference_members[0]
    reference_bytes=archive.read(C_REFERENCE_MEMBER)
    C_REFERENCE_ROWS=[]
    for line_number,line in enumerate(reference_bytes.decode("utf-8").splitlines(),1):
        if not line.strip():continue
        raw=json.loads(line)
        C_REFERENCE_ROWS.append({"archive_line":line_number,**{key:raw.get(key,"") for key in ("document","source_ref","citation","heading","text")}})
    builder_member=next(name for name in archive.namelist() if name.endswith("/scripts/build_corpus.py"))
    builder_text=archive.read(builder_member).decode("utf-8")
C_PREFIX_EVIDENCE=[{"archive_member":builder_member,"line":i,"verbatim_code":line}
    for i,line in enumerate(builder_text.splitlines(),1) if '"embedding_text":' in line]
show_table(C_PREFIX_EVIDENCE)
# Reviewed document matches: title and defragmented URL must agree with Stage 1, never IDs.
C_DOCUMENT_MAP=[]
for title,url in dict.fromkeys((r["document"],r["source_ref"]) for r in C_REFERENCE_ROWS):
    candidates=[did for did,d in M_REGISTRY.items() if d["source_titles"]["title"]["raw_text"]==title
                and any(urldefrag(raw_url)[0]==urldefrag(url)[0] for raw_url in raw_values(did,"source_url"))]
    C_DOCUMENT_MAP.append({"mentor_title":title,"mentor_url":url,"source_document_ids":candidates,
        "status":"exact" if len(candidates)==1 else "ambiguous" if candidates else "unmatched",
        "review_basis":"Reviewed title and source-URL agreement; only URL fragment ignored for document identity",
        "source_title":M_REGISTRY[candidates[0]]["source_titles"]["title"]["raw_text"] if len(candidates)==1 else None,
        "source_urls":raw_values(candidates[0],"source_url") if len(candidates)==1 else []})
C_DOC_LOOKUP={(r["mentor_title"],r["mentor_url"]):r for r in C_DOCUMENT_MAP}
show_table(C_DOCUMENT_MAP)


def reference_components(text):
    marker="[EN summary]";index=text.find(marker)
    body=text if index<0 else text[:index]
    summary="" if index<0 else text[index:]
    prefix="[EN original]" if body.startswith("[EN original]") else ""
    lo=len(prefix)
    while lo<len(body) and body[lo].isspace():lo+=1
    hi=len(body)
    while hi>lo and body[hi-1].isspace():hi-=1
    return {"raw_reference_text":text,"source_excerpt":body[lo:hi],"reference_excerpt_offsets":[lo,hi],
            "english_summary_separate":summary,"language_marker_separate":prefix,
            "generated_context_prefix":"Document title + generated § section + heading, per builder evidence; excluded from excerpt alignment"}
C_REFERENCE_FINGERPRINT=fingerprint(C_REFERENCE_ROWS)
C_REFERENCE_COMPONENTS={r["archive_line"]:reference_components(r["text"]) for r in C_REFERENCE_ROWS}
show_table([{"archive_line":r["archive_line"],"citation":r["citation"],**C_REFERENCE_COMPONENTS[r["archive_line"]]}
            for r in C_REFERENCE_ROWS if r["archive_line"] in (1,4,30)])

### 11d. Citation and literal-excerpt alignment (evaluation only)

An **exact** provision correspondence requires one literal occurrence of the complete comparison
excerpt plus a compatible leading source Article/§ label when the reference supplies one. Exact
means that excerpt only, never full-rule coverage. **Partial** requires a literal source anchor
(at least 60 characters) and a unique citation-compatible source candidate, or one literal anchor
when no legal label exists. Ellipses/rewording prevent a full-excerpt exact result. **Ambiguous**
means multiple candidates/occurrences or conflicting citation evidence. **Unmatched** means no
adequate literal evidence under these conservative rules; it does not mean absent content.

Citation candidates are Stage 1 leading marker locations. A look-ahead window to the next
leading marker is only an evaluation search window, **not a new structural unit or hierarchy**.
For metadata-only pages, use literal source anchors without inventing legal labels. Full source
maps are retained for every displayed match. No normalization, fuzzy semantic matching or translation
is used. Subparagraph citations are preserved but not automatically verified; Article-level matches
can still need manual subparagraph/rule review.

In [ ]:
C_LEADING=re.compile(r"^\s*(?:Чл\.|Член|§)\s*(\d+[а-яА-Я]?)\b",re.I)
C_CITATION=re.compile(r"(?:чл\.|член|§)\s*(\d+[а-яА-Я]?)\b",re.I)
C_CITATION_LOCATIONS={}
for did,view in VIEWS.items():
    locations=[]
    for bid in view["block_ids"]:
        block=BLOCKS[bid];match=C_LEADING.match(block["raw_text"])
        markers=block["structural_annotations"]["legal_marker_candidates"]
        if match and any(m["kind"] in ("article","paragraph_sign") for m in markers):
            _,lo,hi=BLOCK_SPANS[bid]
            locations.append({"label_number":match.group(1).lower(),"label_family":"paragraph_sign" if "§" in match.group(0) else "article",
                "start":lo,"heading_end":hi,"block_id":bid,"position":block["position"],"raw_heading":block["raw_text"],"stage1_evidence":markers})
    for i,loc in enumerate(locations):loc["evaluation_end"]=locations[i+1]["start"] if i+1<len(locations) else len(view["text"])
    C_CITATION_LOCATIONS[did]=locations


def literal_occurrences(text,excerpt):
    if not excerpt:return []
    found=[];start=0
    while True:
        i=text.find(excerpt,start)
        if i<0:break
        found.append((i,i+len(excerpt)));start=i+1
    return found


def align_reference(reference):
    mapping=C_DOC_LOOKUP[(reference["document"],reference["source_ref"])]
    result={"archive_line":reference["archive_line"],"mentor_citation":reference["citation"],"mentor_heading":reference["heading"],
            "document_mapping_status":mapping["status"],"status":"unmatched","source_document_id":None,"source_matches":[],"citation_candidates":[],
            "scope_limit":"Literal excerpt/citation evidence only; full legal-rule equivalence unverified"}
    if len(mapping["source_document_ids"])!=1:
        result.update(status=mapping["status"],reason="Document mapping unresolved");return result
    did=mapping["source_document_ids"][0];result["source_document_id"]=did;text=VIEWS[did]["text"]
    component=C_REFERENCE_COMPONENTS[reference["archive_line"]];excerpt=component["source_excerpt"]
    citation=C_CITATION.search(reference["citation"])
    family="paragraph_sign" if citation and "§" in citation.group(0) else "article"
    candidates=[p for p in C_CITATION_LOCATIONS[did] if citation and p["label_number"]==citation.group(1).lower() and p["label_family"]==family]
    result["citation_candidates"]=candidates
    def compatible(a,b):return [p for p in candidates if p["start"]<=a and b<=p["evaluation_end"]]
    exact=literal_occurrences(text,excerpt)
    if exact:
        good=[(a,b) for a,b in exact if not citation or compatible(a,b)]
        status="exact" if len(exact)==1 and len(good)==1 else "ambiguous"
        result.update(status=status,reason="Full literal excerpt occurrence; citation compatibility checked" if status=="exact" else "Repeated literal text or incompatible/unresolved leading citation")
        matches=exact
    else:
        # Fixed-size literal probes are evaluation anchors, not corpus chunks.
        probes=list(dict.fromkeys(excerpt[i:i+80] for i in range(0,max(1,len(excerpt)-59),40) if len(excerpt[i:i+80])>=60))
        matches=[]
        for probe in probes:
            hits=literal_occurrences(text,probe)
            matches.extend((a,b) for a,b in hits if not citation or compatible(a,b))
        matches=sorted(set(matches))
        candidate_hits={p["block_id"] for a,b in matches for p in compatible(a,b)}
        if matches and ((citation and len(candidate_hits)==1) or (not citation and len(matches)==1)):
            result.update(status="partial",reason="Literal anchor(s) found; complete excerpt differs, is abbreviated, or has ellipses")
        elif matches:result.update(status="ambiguous",reason="Literal anchors occur in multiple candidate contexts")
        else:result.update(status="unmatched",reason="No citation-compatible literal anchor of at least 60 characters; unresolved, not evidence of missing source")
    result["source_match_count"]=len(matches)
    # Preserve all match offsets, but attach verbose maps only to the first six for inspection.
    result["all_source_match_spans"]=[[a,b] for a,b in matches]
    result["source_matches"]=[{"start":a,"end":b,"verbatim_source":text[a:b],"source_spans":source_map(did,a,b)} for a,b in matches[:6]]
    return result
C_ALIGNMENTS=[align_reference(r) for r in C_REFERENCE_ROWS]
show_table([{"status":status,"evaluation_rows":sum(a["status"]==status for a in C_ALIGNMENTS),
             "meaning":"Reference correspondence only; not corpus completeness or retrieval accuracy"}
             for status in ("exact","partial","ambiguous","unmatched")])
show_table([{k:a.get(k) for k in ("archive_line","mentor_citation","document_mapping_status","status","reason","source_match_count")} for a in C_ALIGNMENTS])

### 11e. Inspect reference excerpts, additions and source context

Show one observed example of each correspondence category, plus the long Article reference and
EU examples. Categories absent from this reference set are not fabricated. Synthetic checks below
exercise repeated literal evidence, missing anchors and component extraction separately. Added English summaries and the generated
prefix remain visually separate. A partial/exact excerpt can omit other clauses or history in its
source Article, so correspondence alone does not establish rule completeness.

The table records a qualitative question for rule coverage, boundary choice, context completeness
and citation usefulness. Inspect the actual source windows and citation locations before rating a
reference's legal adequacy. References outside the four experimental documents may be aligned to
Stage 1 source, but no new chunk strategies are run on those documents.

In [ ]:
C_REFERENCE_REVIEW=[]
selected_lines=[]
for status in ("exact","partial","ambiguous","unmatched"):
    example=next((a for a in C_ALIGNMENTS if a["status"]==status),None)
    if example:selected_lines.append(example["archive_line"])
for a in C_ALIGNMENTS:
    if a["source_document_id"] in (DOC_BY_ORDINAL[2]["document_id"],DOC_BY_ORDINAL[11]["document_id"]):selected_lines.append(a["archive_line"])
selected_lines=list(dict.fromkeys(selected_lines))
for line in selected_lines:
    alignment=next(a for a in C_ALIGNMENTS if a["archive_line"]==line)
    reference=next(r for r in C_REFERENCE_ROWS if r["archive_line"]==line);component=C_REFERENCE_COMPONENTS[line]
    C_REFERENCE_REVIEW.append({"archive_line":line,"citation":reference["citation"],"mapping":alignment["status"],
        "rule_coverage_rating":"UNCERTAIN","rule_coverage":"Excerpt correspondence cannot establish all conditions/exceptions; compare source context below",
        "boundary_rating":"CONCERN" if alignment["status"]=="partial" else "UNCERTAIN",
        "boundary_choice":alignment["reason"],"context_completeness":"UNCERTAIN: summaries/ellipsis are not replacements for omitted source clauses or history",
        "citation_rating":"PASS" if alignment["status"] in ("exact","partial") else "UNCERTAIN",
        "citation_usefulness":"Source Article/anchor location is inspectable; subparagraph citation and legal scope still require review" if alignment["status"] in ("exact","partial") else "Mapping unresolved; do not use as a verified source citation",
        "context_rating":"CONCERN" if "…" in component["source_excerpt"] or "..." in component["source_excerpt"] else "UNCERTAIN",
        "source_match_spans":alignment.get("all_source_match_spans",[]),"source_document_id":alignment["source_document_id"]})
    body=details_html("Mentor excerpt only",component["source_excerpt"])
    body+=details_html("Mentor-added English summary — excluded from source correspondence",component["english_summary_separate"])
    body+=details_html("Marker / generated-prefix accounting",json.dumps({k:component[k] for k in ("reference_excerpt_offsets","language_marker_separate","generated_context_prefix")},ensure_ascii=False,indent=2))
    body+=details_html("Citation candidates, uncertainty and matched source maps",json.dumps(alignment,ensure_ascii=False,indent=2))
    did=alignment["source_document_id"]
    if did:
        for match in alignment["source_matches"][:2]:
            lo=max(0,match["start"]-180);hi=min(len(VIEWS[did]["text"]),match["end"]+240)
            body+="<h4>Unchanged source context</h4>"+highlight_html(VIEWS[did]["text"][lo:hi],lo,focus=(match["start"],match["end"]))
        if not alignment["source_matches"]:
            for candidate in alignment["citation_candidates"][:2]:body+=details_html("Unresolved candidate heading — not a match",candidate["raw_heading"])
    display(HTML("<details><summary>Reference archive line "+str(line)+" / "+esc(reference["citation"])+" / "+esc(alignment["status"])+"</summary>"+body+"</details>"))
show_table(C_REFERENCE_REVIEW)

### 11f. Validation and evaluation isolation

No reference object is accepted by a corpus-building function in this section. The allowlisted
reference dictionaries are evaluation-only records, never Stage 1 views. Verify all canonical
inputs, frozen independent observations and approved report bytes after reference inspection.
Validate literal-match maps and label all interpretation limits. The archive is read in place and
its hash must remain unchanged. Counts here describe this evaluation, never equality targets.

In [ ]:
require(fingerprint(C_SPEC)==C_RUN_ID and C_SPEC["display_configuration"]==GALLERY_CONFIGURATION
        and C_SPEC["samples"]==[s["sample_id"] for s in BASELINE_SAMPLES],"Comparison configuration changed; rerun Step 11.")
require(fingerprint(C_REFERENCE_ROWS)==C_REFERENCE_FINGERPRINT,"Reference evaluation inputs changed; rerun evaluation.")
require(comparison_input_fingerprint()==C_INPUT_FINGERPRINT,"Reference evaluation changed corpus inputs/results.")
require(fingerprint(C_INDEPENDENT)==C_INDEPENDENT_FINGERPRINT,"Mentor evidence changed independent observations.")
require(hashlib.sha256(C_ARCHIVE.read_bytes()).hexdigest()==C_ARCHIVE_SHA,"Source archive changed.")
require(all(Path(p).read_bytes()==data for p,data in C_APPROVED_BYTES.items()),"Approved artifact/report changed.")
require(all(set(r)=={"archive_line","document","source_ref","citation","heading","text"} for r in C_REFERENCE_ROWS),"Reference allowlist violated.")
for r in C_REFERENCE_ROWS:
    c=C_REFERENCE_COMPONENTS[r["archive_line"]];lo,hi=c["reference_excerpt_offsets"]
    require(c["source_excerpt"]==r["text"][lo:hi],"Reference excerpt offsets changed.")
for a in C_ALIGNMENTS:
    require(a["status"] in ("exact","partial","ambiguous","unmatched"),"Unknown correspondence category.")
    for match in a["source_matches"]:
        require(match["verbatim_source"]==reconstruct(match["source_spans"])
            ==VIEWS[a["source_document_id"]]["text"][match["start"]:match["end"]],"Reference source evidence mismatch.")
require(literal_occurrences("абв абв","абв")==[(0,3),(4,7)],"Repeated literal evidence lost.")
require(literal_occurrences("абв","липсва")==[],"Unmatched literal evidence invented.")
require(reference_components("[EN original] abc [EN summary] explanation")["source_excerpt"]=="abc","English original incorrectly removed.")
require(reference_components("текст [EN summary] summary")["english_summary_separate"]=="[EN summary] summary","Summary not separated.")
require(C_LEADING.match("Чл. 2а. Текст").group(1)=="2а" and C_CITATION.search("§ 6, ал. 1").group(1)=="6","Cyrillic suffix or § citation lost.")
require(all(r["rating"] in ("PASS","CONCERN","FAIL","UNCERTAIN") for r in C_RUBRIC),"Invalid reviewer rating.")
ensure_metadata_current()
print("PASS: independent comparison frozen; reference path read-only; source mappings, approved artifacts and corpus inputs unchanged.")

### 11g. Evidence table and stop after Step 11

The final table separates observed strengths, weaknesses and unresolved questions. Refer back to
the sample rubric, full source panels and explicit source IDs for each observation. Mentor
correspondence is supplementary evidence only. It cannot establish exhaustive coverage, legal
correctness, retrieval performance or production parameters. No artificial winner score is used.

**STOP — Step 11 complete; waiting for review.** No Hybrid strategy, embeddings, vector storage,
retrieval, RAG or agents are implemented. No final production strategy is declared.

In [ ]:
C_FINAL_EVIDENCE=[
 {"strategy":"Fixed-Length","strength":"PASS: exact source slices, bounded character size and measured overlap",
  "weakness":"CONCERN: reviewed provisions/paragraphs can be split; history-boundary witnesses show note/context separation",
  "unresolved":"UNCERTAIN: whether overlap restores necessary legal context or improves retrieval",
  "evidence":{"sample_metrics":[r for r in C_METRICS if r["strategy"]=="fixed" and r["sample_id"] in ("normal_article","long_article","publication_history")],"history_boundary_witnesses":H_WITNESSES}},
 {"strategy":"Document-Aware Structural","strength":"PASS: complete uncapped provisional source units preserve text without peer overlap",
  "weakness":"CONCERN: uneven sizes and residual/uncertain boundaries; largest residual parent is 45,829 characters",
  "unresolved":"UNCERTAIN: correctness of unreviewed boundaries and a suitable future size policy",
  "evidence":{"sample_metrics":[r for r in C_METRICS if r["strategy"]=="structural" and r["sample_id"] in ("long_article","long_paragraph","unclassified_heading")],"size_triggers":OS_THRESHOLD_ROWS}},
 {"strategy":"Parent-Child","strength":"PASS: explicit contained children resolve to complete source parents; small and large context remain inspectable",
  "weakness":"CONCERN: children still split provisions; Chapter context can be excessive and include additional units",
  "unresolved":"UNCERTAIN: which parent context is useful under a later token budget; metadata resolution is not retrieval validation",
  "evidence":{"sample_metrics":[r for r in C_METRICS if r["strategy"]=="parent_child" and r["sample_id"] in ("normal_article","nested_structure","long_article")],"same_child_alternatives":IDENTICAL_CHILD_COMPARISONS}}]
show_table([{k:v for k,v in row.items() if k!="evidence"} for row in C_FINAL_EVIDENCE])
C_REPORT={"report_version":1,"scope":"Step 11 only — existing strategies and read-only mentor comparison","run_id":C_RUN_ID,
    "independent":C_INDEPENDENT,"independent_fingerprint":C_INDEPENDENT_FINGERPRINT,
    "reference_provenance":{"archive_sha256":C_ARCHIVE_SHA,"member":C_REFERENCE_MEMBER,"member_sha256":hashlib.sha256(reference_bytes).hexdigest()},
    "document_mappings":C_DOCUMENT_MAP,"reference_components":list(C_REFERENCE_COMPONENTS.items()),
    "generated_prefix_evidence":C_PREFIX_EVIDENCE,"reference_alignments":C_ALIGNMENTS,"reference_review":C_REFERENCE_REVIEW,
    "final_evidence":C_FINAL_EVIDENCE,"decision":"No final strategy or parameters selected; waiting for Step 11 review"}
comparison_report_path=OUTPUT_DIR/f"strategy_comparison_report.{fingerprint(C_REPORT)}.json"
comparison_bytes=(canonical(C_REPORT)+"\n").encode("utf-8")
if comparison_report_path.exists():require(comparison_report_path.read_bytes()==comparison_bytes,"Existing comparison report differs.")
else:comparison_report_path.write_bytes(comparison_bytes)
require(json.loads(comparison_report_path.read_text(encoding="utf-8")) == json.loads(canonical(C_REPORT)), "Comparison report round trip failed.")
print("Strategy/reference report:",comparison_report_path.relative_to(ROOT))
print("STOP AFTER STEP 11 — evidence ready for review; no final strategy or Hybrid implementation.")

## Step 12 — Hybrid experiment, conclusions and Stage 2 review

**What:** Test a provisional composition of the already-tested structural units and Step 8 child
spans. No new splitting technique is introduced. Steps 1–11 code and their saved artifacts remain
unchanged; earlier STOP messages are historical approved checkpoints. This is the Stage 2 review,
not permission to run embeddings, vector storage, retrieval, reranking, RAG, LLMs or agents.

**Evidence gate:** The user approved Step 11 and explicitly requested this experiment. Recorded
independent observations, the rule ledger and Step 8 diagnostics must exist. Otherwise display
“hybrid hypothesis pending review” and skip the experiment/export cells. Approval does not turn
provisional legal boundaries into verified legal structure.

**Lead hypothesis for later evaluation:** Keep each existing structural unit whole through 4,000
characters. For a larger unit, retain its complete span as a parent and reuse the Step 8
paragraph/whitespace-aware children at 2,000 characters, initially with **0% overlap**. Preserve
uncertain/unclassified/residual units as source spans, never assigning them invented legal ancestry.
Their oversized spans can likewise be retained as explicitly uncertain source containers with the
same already-tested child mechanism. Short heading/residual units remain visible; do not merge them.

The small shortlist also tests 10% requested overlap and a 2,000-character trigger. These are
sensitivity alternatives, not optimized parameters. The source remains the same four complete
baseline documents; the 12 frozen samples select inspection windows, not splitting boundaries.

In [ ]:
HY_STEP11_APPROVED = True  # Explicit user approval and request for Step 12 in this session.
def hybrid_review_ready(approved, independent, oversized, observations):
    return bool(approved and independent and independent.get("rubric") and oversized and observations)
HY_READY=hybrid_review_ready(HY_STEP11_APPROVED,globals().get("C_INDEPENDENT"),
                           globals().get("OS_DIAGNOSTICS"),globals().get("C_FINAL_EVIDENCE"))
if not HY_READY:
    print("hybrid hypothesis pending review")
else:
    ensure_metadata_current()
    require(fingerprint(C_INDEPENDENT)==C_INDEPENDENT_FINGERPRINT and comparison_input_fingerprint()==C_INPUT_FINGERPRINT,
            "Approved observations or source results changed; review before building the hybrid.")
    print("Evidence gate PASS: Step 11 approved; independent observations and Step 8 diagnostics recorded.")

In [ ]:
if HY_READY:
    HY_PRIOR_PATHS=[APPROVED_MANIFEST_PATH,report_path,structural_report_path,pc_report_path,os_report_path,
                    history_report_path,metadata_report_path,comparison_report_path,
                    ARTIFACT,C_ARCHIVE,ROOT/"notebooks/01_document_ingestion_parsing.ipynb",
                    ROOT/"docs/stage-02-chunking-experiments-plan.md"]
    HY_PRIOR_HASHES={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in HY_PRIOR_PATHS}
    HY_APPROVED_CODE_HASHES={'stage02-setup-03': 'b5bf00ab071ecd34694fe26dd84be7eefcf3187c41018a72f25167088155f187', 'stage02-setup-04': '2ed606c9bdc1eb11209df04e9c33fb3ad6bba96194b34c3cb0d5d96d5935e03f', 'stage02-setup-06': '6b1b9f545d1a0667721d2430797dc3b091a1a0f020584cb2269b81db28b2571b', 'stage02-setup-08': 'a75889f44e648896a9f813df126edccae3a9570dae908659f370f09d145d4e29', 'stage02-setup-10': 'cae940587794bb1c860d42e218a6717ad47824b6a01b3052fc7fd749d88be1c1', 'stage02-setup-12': '005b7c95c7b1f5a4b39adb2d8c4550f9cc9d572a8d99d306b714913ef4315dcd', 'stage02-setup-13': '41acbfa3a8e66052761632fa2f1d5edc02e256500a1dceffe46758afad75ef86', 'stage02-setup-15': 'a29e052a01fa86376e316e60bcec6f3b6d239a9f7bc5950d5e3b523276673e97', 'stage02-setup-17': '5067645b5cf4b425b2641c457a010abf81e5519c26e60ade977d525a21a8be2b', 'stage02-setup-19': '33ecc726610f0f29ccd6622d107ce1268801d4745280fe3fa28438b46d995a4a', 'stage02-setup-21': '92e42ed9ece3e3adbec824941a749be94f5a655196e9abefbad80d629ac7c376', 'stage02-setup-23': 'ac85fe3740e75203fd786374b75c57969411ac9fede3f26f88ceeaf811cd6282', 'stage02-fixed-02': '8cd2f189a83371182972b6687f684647f2933ae8eadd7d8758130a64aadda983', 'stage02-fixed-04': 'e2643fc984f49953692a7cc0481f5880ebd6d4ede24b30d1fa256a810f4d0e99', 'stage02-fixed-06': 'aa73f6706b58469caa7668c4286fe4c9dc88328a4948d631ffd03a7a8779db70', 'stage02-fixed-08': 'b63e3903756335cd77e74bcc81263ab974d523263d0e0730ba7ac21821b275e5', 'stage02-fixed-10': '3315dca54b9d92f00cfc965a67ea128c2b9fc059871dc9bd22094717c7d11466', 'stage02-fixed-12': 'bb33cda73edca11604a693fc2fbcfd6ec96c708c5c983cc156470a75f96985e8', 'stage02-fixed-13': '9110708f6337aa67579d36d7954d2eb327534e2e2c564d64f13b0cb84ccd9eae', 'stage02-fixed-15': 'd6696284816bee28149f205210034a1425532c3624af2df2114c56fd942c0264', 'stage02-fixed-17': 'c637a554dc703805b296d38d44dbe31f73f4a95d67a1590169e030bd4d7098cd', 'stage02-structural-02': '2889960d83d30ef1bf340f9cafe4d3f39a6d03e7ee778aa04a5a92fc5eeffcb4', 'stage02-structural-04': 'a92f84af485fdb7804c76cf3c6b7a8933fee4ac054294e1777e23004706c2a79', 'stage02-structural-06': '76acbe313788f6d06deef5f89d0c2085b2a4e69f3891b61c3fe32e06457b3c19', 'stage02-structural-08': 'b4b5899ecbe661bf356982aa83a4764be8b04c82121a7979c23ab82db5a7f1dc', 'stage02-structural-10': 'f892e009a45b5a79912fdb6720d70853b6e3e48734aabf54877449b98c9d13c1', 'stage02-structural-12': 'd2d778baf0b36300826a5e40a13496aa8eea187b52ee944c704ee8c812cc7d8f', 'stage02-structural-14': '8f7c35584a7f4d0ef711d86f0fb9e56c8c258c4775073582008653803f8bb795', 'stage02-structural-16': '87ce63e37ec1507250c44908a3c344d8d8f4a4f7b7f013a063f50a20b39fa485', 'stage02-structural-18': 'e90a3e42e7c049b92b8809ab8dcd630d8955fef63f73b4da8a3758b44158a2f4', 'stage02-parent-child-02': 'fe3da9af62d8b00f5ef36029697f8624fc89f6a458a74aaa6019bc7d51bfc667', 'stage02-parent-child-04': 'f27b6237a99a0a982a8839c0c82b2b321964d5fefce9245aa65741aad5c8e6d9', 'stage02-parent-child-06': '7a370b7c563fd8f52c24699684055f21cf1aa195544a8e575775be15d3adf27a', 'stage02-parent-child-08': '8064763a94de19751022fd9bb048a2e102d742b5d77e3d985d530f516c29bc30', 'stage02-parent-child-10': 'cea88b9fc8ca3f470acb0845b3455f06db22d6ca471f312b990f255a4232aac0', 'stage02-parent-child-12': '8a0b3ba47e5daf07cb0e989408b236843d3c0531a76867f2495b387add99eb66', 'stage02-parent-child-14': '07f4ba8904b0c7fafa142218c32eddae43e590872c5d0dfee7e7b957051de96f', 'stage02-parent-child-16': '174567506b4abd017fdfd54d888aae695ca7518b9afaf5dc12e6c824bb5a517b', 'stage02-parent-child-18': 'bc30d84b93ebea38b362a0fb6a2c9366813150d35d3a0010bfebc21788e751db', 'stage02-parent-child-20': '029699a81e8e16472fb03d357722b0017c644734b286d7287ee8861bc4aa92bd', 'stage02-oversized-01': '6c41b524434462067d54c86021af3243e22fd8c79870ccd0c330f28e241e94ce', 'stage02-oversized-03': '44f87795f3838dc4b3c2274363fe27ed0acbad9a2617b9d98b42b3b23a56a67b', 'stage02-oversized-05': 'cd59b82aa728a57f6c65d56b0071e09fbb0fef577770d2006a870e7dfb1b6e44', 'stage02-oversized-07': '312c05d85a996ecc6bc521d248ee28e80d0b93675d227b392e14c2926cc02649', 'stage02-oversized-09': '0b6c31369471dc666b7248049ac5f7a25163d4ce5bf224b250ef23b5ae0392d3', 'stage02-oversized-11': '512ea45f97ad6e91b4f50632629fca13775e170212adc2ca86aa197ebc5ad1a8', 'stage02-oversized-13': '298792d336a81e0628c59d41c4cb59a2e6bf2a64f67a6d41dade12ef04578d5a', 'stage02-history-01': '68855890a33bbd522715dddc1379783045d51149f488e1c69eec7d68cbcba397', 'stage02-history-03': 'b6221f9133dcaccabc9652e66034ea10df83a187d511ad28a57ee3f494f27b94', 'stage02-history-05': 'c72757a63c466e6bc7d1cbcf475f302c88dc94d48a3ac552b90b739b53c4170a', 'stage02-history-07': 'e131ddf87f66af94312be046c07dbd855571a892922647cb949763fb0d3ca274', 'stage02-history-09': 'c535afee385b7923ced7fb18b6c7b3d8852dca4435da3abc63fe26c349d7a063', 'stage02-history-11': 'ddf9a755ea0d05c104cdc7c46c5e46b1cd49108b96fc700130126223ad55e869', 'stage02-metadata-01': '15dfab3299ba22a2004cbadedb5946eb09ae0c17996aacd61bba6b0d040063fd', 'stage02-metadata-03': '8cd93ba2ce5db862b5ccdb0af8a3619c727b61dd66e40f3c88249ffe47d19e02', 'stage02-metadata-05': 'cb079538c1d20ceb129fe2d05b915c7787de3e5ae710ba38f4c96d0470a90f2d', 'stage02-metadata-07': '81f6fa3954f23bb1444da50241b7de1239e49655b9bbf443cf4de669dcf0077b', 'stage02-metadata-09': 'ae4661f234f8217b7a2df97a1afade55eae1f98d2def4fc00c19365b3aeda5f9', 'stage02-metadata-11': 'f877b8a78820c4784d8875571d4c2c43c6531ff306454e8959a9a30490b0b689', 'stage02-comparison-01': '345d0d9dab0752a319f25e4d67076e2721e14298fcb63539324133da2ef1a654', 'stage02-comparison-03': 'a29b5849c2348b132fc6e9b36c1b22f19d510301d1d4bde7bf6da53a9524df26', 'stage02-comparison-05': '9f566ddc43c637a8f4ede50781a966775a98b6bacc1d79ddf6c8b11d514f404e', 'stage02-comparison-07': 'a5d0e864b2d2de1a8b7023e8f63e5ececbce7e426e625ccf156acfd94e4f36c6', 'stage02-comparison-09': '886fd7b5e7002b4facfc204d76d2d04239fa1702883209fc22319d97c4402398', 'stage02-comparison-11': 'd52742e3608544c1b0ead76e73fd8399483cc720c3b91f127eeaef4b42c0a4fb', 'stage02-comparison-13': 'afa34922d7221fe98bc2491fdd9b7a8d24e2c7059d96cc50ab04b83639227fa4', 'stage02-comparison-15': '7902ad5e0d00ea0086f48cb5f8edf749da06d25624c6d4342cdebf5bec5f9732'}
    HY_SHORTLIST=[
        {"id":"hybrid-4000-aware-2000-overlap0","threshold":4000,"setting":"boundary_aware-chars-2000-overlap-0pct","role":"provisional lead"},
        {"id":"hybrid-4000-aware-2000-overlap10","threshold":4000,"setting":"boundary_aware-chars-2000-overlap-10pct","role":"overlap sensitivity"},
        {"id":"hybrid-2000-aware-2000-overlap0","threshold":2000,"setting":"boundary_aware-chars-2000-overlap-0pct","role":"lower size-trigger sensitivity"}]
    HY_LEAD=HY_SHORTLIST[0]["id"]
    HY_RULES=[
     {"id":"retain_units","rule":"Retain each existing structural unit whole when size <= trigger; no merging or new hierarchy",
      "observed_tradeoff":"Fixed windows cut reviewed provisions or combine neighboring provisions; uncapped units preserve source boundaries but have uneven sizes",
      "evidence":{"step11_metrics":[r for r in C_METRICS if r["sample_id"] in ("normal_article","short_neighbors","transitional_provision")],"structural_run":STRUCTURAL_ID},
      "support":"Supported for exact preservation; legal boundary correctness remains provisional"},
     {"id":"size_trigger","rule":"Lead trigger 4,000 characters; retain larger units as complete parents",
      "observed_tradeoff":"2,000 triggers 120 units, 4,000 triggers 43, 8,000 triggers 11; 4,000 leaves 77 additional 2–4k units intact versus 2,000",
      "evidence":{"step8_thresholds":OS_THRESHOLD_ROWS},"support":"Provisional compromise, not a proven token/context budget"},
     {"id":"controlled_children","rule":"Reuse Step 8 paragraph/whitespace-aware children at 2,000 characters",
      "observed_tradeoff":"At trigger 4,000 and zero overlap: aware cuts 6 paragraphs versus 169 exact; aware makes 236 children versus 219 exact",
      "evidence":{"step8_rows":[r for r in OS_DIAGNOSTICS if r["threshold"]==4000 and r["setting"] in ("exact-chars-2000-overlap-0pct","boundary_aware-chars-2000-overlap-0pct")]},
      "support":"Supported mechanical paragraph preservation, with more/underfilled children; semantic focus untested"},
     {"id":"overlap","rule":"Start at 0%; retain 10% requested overlap as sensitivity alternative",
      "observed_tradeoff":"Aware 0% has 0 duplicate characters and 6 paragraph cuts; aware 10% has 13,543 duplicate characters and 7 cuts at trigger 4,000",
      "evidence":{"step8_rows":[r for r in OS_DIAGNOSTICS if r["threshold"]==4000 and r["setting"].startswith("boundary_aware-chars-2000")]},
      "support":"Provisional: avoids observed duplication; loses repeated local context whose retrieval value is unknown"},
     {"id":"parent_policy","rule":"Link oversized children to the original complete structural unit, not an automatically widened Section/Chapter",
      "observed_tradeoff":"Step 7 identical children can point to much larger parents with additional units; a complete parent can still be 45,829 characters",
      "evidence":{"same_child_parent_comparisons":IDENTICAL_CHILD_COMPARISONS},"support":"Supported containment/traceability; future parent inclusion needs an explicit context-budget policy"},
     {"id":"uncertainty","rule":"Keep unclassified/residual spans and existing uncertainty explicit; oversized unknowns get source-container links only",
      "observed_tradeoff":"Weak structure, publication preambles and unclassified headings do not establish legal ancestry",
      "evidence":{"samples":[s for s in BASELINE_SAMPLES if s["sample_id"] in ("weak_structure","unclassified_heading","long_paragraph","publication_history")]},
      "support":"Supported conservative source accounting; tiny headings and large prose contexts remain counterexamples"},
     {"id":"history_metadata","rule":"Keep all history verbatim; resolve separate annotations, raw document fields and provenance through existing registries",
      "observed_tradeoff":"History can dominate chunks; attribution remains uncertain. Registry joins aid inspection without copying long notes or inventing effective dates",
      "evidence":{"history_shares":H_SAMPLE_SHARES,"history_witnesses":H_WITNESSES,"metadata_run":M_RUN_ID},
      "support":"Supported source fidelity; metadata-only history removal and filtering effectiveness remain untested"}]
    HY_SPEC={"version":1,"setup_id":SETUP_ID,"source_fingerprint":C_INPUT_FINGERPRINT,
             "structural_fingerprint":STRUCTURAL_RESULTS_FINGERPRINT,"step8_fingerprint":OS_DATA_FINGERPRINT,
             "independent_observations":C_INDEPENDENT_FINGERPRINT,"shortlist":HY_SHORTLIST,
             "rules_fingerprint":fingerprint(HY_RULES),"stage11_approved":HY_STEP11_APPROVED,
             "meaningful_provisional_kinds":["article","paragraph_sign","annex","part","chapter","section"],
             "uncertain_kinds":["unclassified","residual"],"mentor_builder_input":False}
    HY_RUN_ID=fingerprint(HY_SPEC)
    show_table([{k:r[k] for k in ("id","rule","observed_tradeoff","support")} for r in HY_RULES])
    show_table(HY_SHORTLIST)

### 12a. Construct the hypothesis from existing source spans

The builder accepts a shortlist configuration only. It reads the approved structural results and
Step 8 children; mentor evaluation objects cannot be supplied as corpus inputs. A kept unit is a
single output span. An oversized unit is stored once as a complete parent with ordered child links.
The future candidate payload layer consists of kept units plus children, not parents added again.
No embeddings or indexing are performed.

A `chapter`/`section` unit here is exactly the Step 6 flat span; it does not acquire descendants.
Unclassified/residual parents have `ancestry_assertion: None` and an explicit source-container role.
Every record retains the original unit ID, provisional kind, boundary evidence and uncertainty.
Text is read directly from the unchanged document stream; IDs include source/configuration/spans.

In [ ]:
if HY_READY:
    def hybrid_action(size,threshold):
        return "retain_whole" if size<=threshold else "reuse_tested_children"


    def build_hybrid(config):
        require(config in HY_SHORTLIST,"Only reviewed shortlist configurations are allowed.")
        require(fingerprint(STRUCTURAL_RESULTS)==HY_SPEC["structural_fingerprint"] and fingerprint(list(OS_GROUPS.values()))==HY_SPEC["step8_fingerprint"],
                "Structural or tested child evidence changed; rebuild/review before composing hybrid.")
        parents,leaves,links,unit_rows={},{},{},[]
        document_ordinals=Counter()
        for did in BASELINE_DOCUMENT_IDS:
            for unit in STRUCTURAL_RESULTS[did]:
                uid=unit["id"];size=unit["end"]-unit["start"]
                uncertain=unit["unit_kind"] in HY_SPEC["uncertain_kinds"]
                common={"document_id":did,"setup_id":SETUP_ID,"configuration":config["id"],
                    "run_ref":HY_RUN_ID,"document_registry_ref":did,"structural_unit_id":uid,
                    "source_kind":unit["unit_kind"],"source_boundary_evidence_ids":unit["evidence_ids"],
                    "source_uncertainty":unit["uncertainty"],"ancestry_assertion":None,
                    "interpretation":"explicitly uncertain source span" if uncertain else "provisional structural role; legal meaning unverified"}
                action=hybrid_action(size,config["threshold"])
                if action=="retain_whole":
                    pieces=[{"start":unit["start"],"end":unit["end"],"split_reason":"preserved_original_unit_at_or_below_trigger"}]
                    parent_id=None;tested_group_id=None
                else:
                    tested=OS_GROUPS[(uid,config["setting"])];tested_group_id=tested["id"]
                    require(config["threshold"] in tested["eligible_thresholds"],"Step 8 did not test this trigger/parent pair.")
                    parent_id="hybrid-parent:"+fingerprint([HY_RUN_ID,config["id"],uid,unit["start"],unit["end"]])
                    parents[parent_id]={**common,"id":parent_id,"ordinal":unit["ordinal"],"start":unit["start"],"end":unit["end"],
                        "text":VIEWS[did]["text"][unit["start"]:unit["end"]],"source_spans":source_map(did,unit["start"],unit["end"]),
                        "block_ids":unit["block_ids"],"record_kind":"uncertain source container" if uncertain else "complete provisional structural parent",
                        "parent_role":"uncertain source container" if uncertain else unit["unit_kind"],"tested_group_ref":tested_group_id}
                    pieces=tested["children"]
                child_ids=[]
                for order,piece in enumerate(pieces,1):
                    a,b=piece["start"],piece["end"]
                    cid="hybrid-span:"+fingerprint([HY_RUN_ID,config["id"],uid,a,b])
                    document_ordinals[did]+=1
                    record={**common,"id":cid,"ordinal":document_ordinals[did],"start":a,"end":b,
                        "text":VIEWS[did]["text"][a:b],"source_spans":source_map(did,a,b),
                        "record_kind":"hybrid child" if parent_id else "hybrid preserved unit",
                        "split_reason":piece["split_reason"],"action":action,
                        "tested_child_ref":piece.get("id") if parent_id else None,
                        "actual_previous_overlap":max(0,pieces[order-2]["end"]-a) if order>1 else 0,
                        "actual_next_overlap":max(0,b-pieces[order]["start"]) if order<len(pieces) else 0}
                    if parent_id:
                        record["parent_id"]=parent_id
                        rid="hybrid-link:"+fingerprint([HY_RUN_ID,config["id"],parent_id,cid,order])
                        record["relationship_id"]=rid
                        links[rid]={"id":rid,"parent_id":parent_id,"child_id":cid,"order":order,
                                    "relationship_type":"contained_source_span","tested_group_ref":tested_group_id}
                    leaves[cid]=record;child_ids.append(cid)
                unit_rows.append({"source_unit_id":uid,"document_id":did,"source_kind":unit["unit_kind"],
                    "original_size":size,"action":action,"parent_id":parent_id,"output_ids":child_ids,
                    "uncertain_source_role":uncertain,"tested_group_ref":tested_group_id})
        return {"configuration":config,"parents":parents,"outputs":leaves,"links":links,"unit_decisions":unit_rows}
    HY_RESULTS={config["id"]:build_hybrid(config) for config in HY_SHORTLIST}
    HY_RESULT_FINGERPRINT=fingerprint(HY_RESULTS)
    HY_LEAD_RESULT=HY_RESULTS[HY_LEAD]
    print("Provisional lead composed:",len(HY_LEAD_RESULT["outputs"]),"candidate payload spans;",len(HY_LEAD_RESULT["parents"]),"complete parents retained.")

### 12b. Inspect the same frozen samples before aggregate diagnostics

Use the already-tested **2,000 / 0%** fixed and parent-child settings as matched controls for the
lead's child setting; keep the uncapped structural baseline. Parent choice in the parent-child
control stays exactly as selected in Step 11. Its configuration changes only to the already-generated
zero-overlap alternative. Step 11's 10% displays remain available as historical controls.

For comparison metrics, include only payload records intersecting the same sample in *every*
strategy. This avoids counting all children of a large parent on one side and just the focus on
another. Complete parents and all children are still expandable for inspection; they are not added
to payload overlap totals. Extra source text outside the blue focus is measured explicitly.

In [ ]:
if HY_READY:
    HY_CONTROL_CONFIGURATION="chars-2000-overlap-0pct"
    HY_SAMPLE_VIEWS={};HY_SAMPLE_METRICS=[]
    for sample in BASELINE_SAMPLES:
        did,lo,hi=sample["document_id"],sample["start"],sample["end"]
        intersects=lambda r:r["document_id"]==did and intersection_length((lo,hi),(r["start"],r["end"]))>0
        fixed=[r for r in FIXED_RESULTS[HY_CONTROL_CONFIGURATION][did] if intersects(r)]
        structural=[r for r in STRUCTURAL_RESULTS[did] if intersects(r)]
        pc_records=[];pc_parents=[];pc_groups=[]
        for old_gid in C_SELECTIONS[sample["sample_id"]]["parent_group_ids"]:
            old=PC_GROUPS[old_gid];group=PC_GROUP_BY_KEY[(old["case"],old["mode"],HY_CONTROL_CONFIGURATION)]
            parent,children=relationship_display_records(group["id"])
            pc_groups.append(group["id"]);pc_parents.append(parent);pc_records.extend(c for c in children if intersects(c))
        hybrid=[r for r in HY_LEAD_RESULT["outputs"].values() if intersects(r)]
        parent_ids=list(dict.fromkeys(r["parent_id"] for r in hybrid if "parent_id" in r))
        hybrid_parents=[HY_LEAD_RESULT["parents"][pid] for pid in parent_ids]
        HY_SAMPLE_VIEWS[sample["sample_id"]]={"fixed":fixed,"structural":structural,"parent_child":pc_records,
            "hybrid":hybrid,"pc_group_ids":pc_groups,"hybrid_parent_ids":parent_ids}
        for strategy,records,parents in (("fixed",fixed,[]),("structural",structural,[]),("parent_child",pc_records,pc_parents),("hybrid",hybrid,hybrid_parents)):
            row=comparison_metrics(sample,strategy,records,parents)
            row["metric_scope"]="payload spans intersecting frozen sample; complete context parents excluded from overlap"
            ratios=[(p["end"]-p["start"])/(r["end"]-r["start"]) for r in records for p in parents if r.get("parent_id")==p["id"]]
            row["parent_expansion_ratios"]=length_summary(ratios)
            HY_SAMPLE_METRICS.append(row)
        source=VIEWS[did]["text"][lo:hi]
        panels=[]
        for strategy,records in (("fixed",fixed),("structural",structural),("parent_child",pc_records),("hybrid",hybrid)):
            body="".join(span_card(r,sample,records,REVIEWED_PROVISIONS) for r in records)
            if strategy=="parent_child":
                for gid in pc_groups:
                    parent,children=relationship_display_records(gid)
                    body+="<details><summary>Complete control parent and all children</summary>"+relationship_html(parent,children,sample,REVIEWED_PROVISIONS)+"</details>"
            if strategy=="hybrid":
                body+=details_html("Hybrid decisions and uncertainty",json.dumps([{k:r[k] for k in ("id","structural_unit_id","source_kind","interpretation","ancestry_assertion","source_uncertainty","action","split_reason")} for r in records],ensure_ascii=False,indent=2))
                for parent in hybrid_parents:
                    children=[r for r in HY_LEAD_RESULT["outputs"].values() if r.get("parent_id")==parent["id"]]
                    body+="<details><summary>Complete hybrid parent and all children — no truncation</summary>"+relationship_html(parent,children,sample,REVIEWED_PROVISIONS)+"</details>"
            panels.append("<section><h4>"+esc(strategy)+"</h4>"+body+"</section>")
        display(HTML("<details><summary>"+esc(sample["sample_id"])+" — hybrid and matched existing controls</summary>"
            +"<h4>Same frozen source span</h4>"+highlight_html(source,lo)
            +"<div style='display:grid;grid-template-columns:repeat(4,minmax(320px,1fr));gap:12px;overflow-x:auto'>"+"".join(panels)+"</div></details>"))
    show_table([{k:v for k,v in r.items() if k not in ("source_evidence","reviewed_legal_units_cut")} for r in HY_SAMPLE_METRICS])

### 12c. Improvements, regressions and the shortlist's measured costs

Deltas below mean **lead hybrid minus the named baseline on the same frozen sample**. Fewer cut
paragraphs or duplicated positions is a mechanical improvement, not a relevance score. More
records can mean additional processing and smaller context; fewer can hide broader topics.
Longer payloads are a future budget concern, even if they preserve a complete rule. No scalar winner
score combines these effects. Zero overlap can remove duplicated local context that later proves useful.

Across the four complete documents, the output layer covers each source position, including added
separators. Parent containment is counted separately. Repeated wording at distinct source positions
remains original source text, not deduplicated material. All 662 original structural units remain
available as the unchanged control; the experiment does not replace the approved Stage 6 artifact.

In [ ]:
if HY_READY:
    HY_DELTAS=[]
    for sample in BASELINE_SAMPLES:
        rows={r["strategy"]:r for r in HY_SAMPLE_METRICS if r["sample_id"]==sample["sample_id"]};hy=rows["hybrid"]
        for name in ("fixed","structural","parent_child"):
            base=rows[name]
            HY_DELTAS.append({"sample_id":sample["sample_id"],"baseline":name,
                "paragraphs_cut_delta":hy["paragraph_blocks_cut"]-base["paragraph_blocks_cut"],
                "reviewed_units_cut_delta":len(hy["reviewed_legal_units_cut"])-len(base["reviewed_legal_units_cut"]),
                "records_combining_reviewed_units_delta":hy["records_combining_reviewed_units"]-base["records_combining_reviewed_units"],
                "overlap_duplicate_characters_delta":hy["overlap_duplicate_characters"]-base["overlap_duplicate_characters"],
                "payload_count_delta":hy["records"]-base["records"],
                "max_payload_size_delta":hy["size_distribution"]["max"]-base["size_distribution"]["max"],
                "extra_source_characters_delta":hy["extra_characters_outside_sample"]-base["extra_characters_outside_sample"],
                "evidence":{"hybrid":hy["source_evidence"],"baseline":base["source_evidence"]}})
    show_table([{k:v for k,v in r.items() if k!="evidence"} for r in HY_DELTAS])
    HY_AGGREGATES=[];HY_SENSITIVITY_SAMPLES=[]
    for config in HY_SHORTLIST:
        result=HY_RESULTS[config["id"]];records=list(result["outputs"].values());sizes=[len(r["text"]) for r in records]
        parents=list(result["parents"].values());total=sum(len(VIEWS[did]["text"]) for did in BASELINE_DOCUMENT_IDS)
        cut_paragraphs={bid for r in records for bid in paragraph_cuts(r)}
        ratios=[len(parents_record["text"])/len(r["text"]) for r in records if "parent_id" in r for parents_record in [result["parents"][r["parent_id"]]]]
        HY_AGGREGATES.append({"configuration":config["id"],"role":config["role"],"source_characters":total,
            "original_structural_units":len(OS_CONTROL),"retained_whole_units":sum(d["action"]=="retain_whole" for d in result["unit_decisions"]),
            "oversized_parents":len(parents),"uncertain_source_containers":sum(p["source_kind"] in HY_SPEC["uncertain_kinds"] for p in parents),
            "children":len(result["links"]),"candidate_payload_count":len(records),"payload_sizes":length_summary(sizes),
            "payloads_above_2000":sum(s>2000 for s in sizes),"payloads_below_100":sum(s<100 for s in sizes),
            "paragraph_blocks_cut":len(cut_paragraphs),"duplicate_peer_characters":sum(sizes)-total,
            "complete_parent_sizes":length_summary([len(p["text"]) for p in parents]),"parent_expansion":length_summary(ratios),
            "uncertain_output_spans":sum(r["source_kind"] in HY_SPEC["uncertain_kinds"] for r in records)})
        for sample in BASELINE_SAMPLES:
            relevant=[r for r in records if r["document_id"]==sample["document_id"] and intersection_length((r["start"],r["end"]),(sample["start"],sample["end"]))]
            metrics=comparison_metrics(sample,"hybrid",relevant)
            HY_SENSITIVITY_SAMPLES.append({"configuration":config["id"],**metrics})
    show_table(HY_AGGREGATES)
    display(HTML("<details><summary>All three shortlist configurations on all frozen samples</summary>"
        +table_html([{k:v for k,v in r.items() if k!="source_evidence"} for r in HY_SENSITIVITY_SAMPLES])+"</details>"))
    HY_TRADEOFFS=[
     {"observation":"Fewer avoidable paragraph cuts in oversized units","rating":"PASS","limit":"Mechanical evidence only; children still split long provisions","evidence_rule":"controlled_children"},
     {"observation":"Whole small Articles/§ units preserved instead of forced child splitting","rating":"PASS","limit":"Whole units can reach 4,000 characters, above the 2,000-character child target","evidence_rule":"retain_units"},
     {"observation":"Versus uncapped structural units, long rules are now fragmented in the candidate payload layer","rating":"CONCERN","limit":"Complete parents remain available, but later expansion must be deliberately budgeted","evidence_rule":"parent_policy"},
     {"observation":"Tiny headings/residual units remain; no automatic merging solves weak structure","rating":"CONCERN","limit":"Uncertain spans remain traceable; usefulness as future search targets untested","evidence_rule":"uncertainty"},
     {"observation":"Zero-overlap lead avoids duplication but may lose useful adjacent context","rating":"UNCERTAIN","limit":"10% sensitivity retains variable actual overlap; retrieval tests must decide","evidence_rule":"overlap"},
     {"observation":"Parent context remains as large as the source requires","rating":"CONCERN","limit":"45,829-character residual context and 18,756-character Article are not assumed to fit a model budget","evidence_rule":"parent_policy"},
     {"observation":"History remains intact and separately inspectable","rating":"PASS","limit":"History share/metadata usefulness does not establish retrieval quality or effective dates","evidence_rule":"history_metadata"}]
    show_table(HY_TRADEOFFS)

### 12d. Validate the composition and preservation contract

Reconstruct every document by removing only measured overlaps from ordered candidate payloads.
Check that preserved units are exact originals, every child span is identical to its Step 8 record,
and complete parents resolve without cycles. Validate IDs, source maps, relationship order,
registry references and artifact/DOCX fingerprints. Unclassified/residual labels must survive
unchanged. The notebook's approved earlier code cells and all prior data/report files must remain
unchanged. These are fidelity checks, not legal validation.

In [ ]:
if HY_READY:
    def ensure_hybrid_current():
        require(HY_READY and HY_STEP11_APPROVED,"hybrid hypothesis pending review")
        require(fingerprint(HY_SPEC)==HY_RUN_ID and HY_SPEC["shortlist"]==HY_SHORTLIST
                and fingerprint(HY_RULES)==HY_SPEC["rules_fingerprint"],"Hybrid rules/settings changed; rebuild Step 12.")
        require(fingerprint(HY_RESULTS)==HY_RESULT_FINGERPRINT,"Hybrid results changed; rebuild comparisons.")
        require(fingerprint(C_INDEPENDENT)==HY_SPEC["independent_observations"] and comparison_input_fingerprint()==HY_SPEC["source_fingerprint"],
                "Approved source/evidence changed; review and rebuild the hybrid.")
        require([s["sample_id"] for s in BASELINE_SAMPLES]==C_SPEC["samples"] and fingerprint(frozen_samples())==fingerprint(BASELINE_SAMPLES),
                "Frozen sample selection changed; rebuild Stage 2 comparisons.")


    HY_VALIDATION_COUNTS=[]
    for config in HY_SHORTLIST:
        result=HY_RESULTS[config["id"]];parents=result["parents"];records=result["outputs"]
        require(len(records)==len(set(records)) and len(result["links"])==len(set(result["links"])),"Duplicate hybrid identity.")
        for decision in result["unit_decisions"]:
            unit=STRUCTURAL_BY_ID[decision["source_unit_id"]]
            outputs=[records[cid] for cid in decision["output_ids"]]
            require(decision["action"]==hybrid_action(len(unit["text"]),config["threshold"]),"Incorrect size branch.")
            if decision["action"]=="retain_whole":
                require(len(outputs)==1 and outputs[0]["text"]==unit["text"] and "parent_id" not in outputs[0],"Preserved unit changed or got an invented parent.")
            else:
                parent=parents[decision["parent_id"]];tested=OS_GROUPS[(unit["id"],config["setting"])]
                require(parent["text"]==unit["text"] and parent["source_spans"]==source_map(unit["document_id"],unit["start"],unit["end"]),"Complete parent changed.")
                require("parent_id" not in parent and parent["id"] not in records,"Cycle or parent included as candidate payload.")
                require(parent["id"]=="hybrid-parent:"+fingerprint([HY_RUN_ID,config["id"],unit["id"],unit["start"],unit["end"]]),"Parent ID changed.")
                require([(c["start"],c["end"],c["text"]) for c in outputs]==[(c["start"],c["end"],c["text"]) for c in tested["children"]],"New child splitter or modified spans introduced.")
                relative=[{"start":c["start"]-parent["start"],"end":c["end"]-parent["start"],"text":c["text"]} for c in outputs]
                check_os_windows(parent["text"],relative,tested["setting"]["target"],tested["setting"]["overlap"])
                if unit["unit_kind"] in HY_SPEC["uncertain_kinds"]:
                    require(parent["parent_role"]=="uncertain source container" and parent["ancestry_assertion"] is None,"Invented ancestry for uncertain content.")
                for order,child in enumerate(outputs,1):
                    link=result["links"][child["relationship_id"]]
                    require(link["parent_id"]==parent["id"] and link["child_id"]==child["id"] and link["order"]==order,"Broken link/order.")
                    require(link["id"]=="hybrid-link:"+fingerprint([HY_RUN_ID,config["id"],parent["id"],child["id"],order]),"Link ID changed.")
                    require(parent["document_id"]==child["document_id"] and parent["start"]<=child["start"]<child["end"]<=parent["end"],"Child escaped its parent.")
            for child in outputs:
                require(child["text"]==reconstruct(child["source_spans"])==VIEWS[child["document_id"]]["text"][child["start"]:child["end"]],"Source reconstruction failed.")
                require(child["id"]=="hybrid-span:"+fingerprint([HY_RUN_ID,config["id"],unit["id"],child["start"],child["end"]]),"Output identity changed.")
                require(child["document_registry_ref"] in M_REGISTRY and child["source_kind"]==unit["unit_kind"]
                        and child["source_boundary_evidence_ids"]==unit["evidence_ids"] and child["source_uncertainty"]==unit["uncertainty"],"Provenance/uncertainty lost.")
        for did in BASELINE_DOCUMENT_IDS:
            ordered=sorted((r for r in records.values() if r["document_id"]==did),key=lambda r:(r["start"],r["end"]))
            end=0;restored=[]
            for r in ordered:
                require(r["start"]<=end<r["end"],"Gap or redundant candidate payload.")
                require(r["actual_previous_overlap"]==end-r["start"],"Measured peer overlap mismatch.")
                restored.append(r["text"][end-r["start"]:]);end=r["end"]
            require(end==len(VIEWS[did]["text"]) and "".join(restored)==VIEWS[did]["text"],"Whole document does not reconstruct exactly.")
        require(fingerprint(build_hybrid(config))==fingerprint(result),"Hybrid IDs/results are not reproducible.")
        HY_VALIDATION_COUNTS.append({"configuration":config["id"],"outputs_checked":len(records),"parents_checked":len(parents),"relationships_checked":len(result["links"]),"coverage":"100% exact document reconstruction"})
    require(all(r["sample_coverage_pct"]==100 for r in HY_SAMPLE_METRICS+HY_SENSITIVITY_SAMPLES),"A frozen sample is not fully covered.")
    require(not hybrid_review_ready(False,C_INDEPENDENT,OS_DIAGNOSTICS,C_FINAL_EVIDENCE)
            and not hybrid_review_ready(True,{},OS_DIAGNOSTICS,C_FINAL_EVIDENCE),"Observation/approval gate failed.")
    require(hybrid_action(0,4000)=="retain_whole" and hybrid_action(4000,4000)=="retain_whole"
            and hybrid_action(4001,4000)=="reuse_tested_children","Threshold edge case changed.")
    expect_value_error(lambda:build_hybrid({"text":"mentor reference","threshold":4000}),"Only reviewed shortlist configurations")
    old_threshold=HY_SHORTLIST[0]["threshold"]
    try:
        HY_SHORTLIST[0]["threshold"]=3999
        expect_value_error(ensure_hybrid_current,"Hybrid rules/settings changed")
    finally:HY_SHORTLIST[0]["threshold"]=old_threshold
    require(all(hashlib.sha256((ROOT/p).read_bytes()).hexdigest()==digest for p,digest in HY_PRIOR_HASHES.items()),"An original artifact/report/archive changed.")
    # Read notebook JSON with the standard library; verify original code cell sources byte-for-byte.
    current_notebook=json.loads((ROOT/"notebooks/02_chunking_experiments.ipynb").read_text(encoding="utf-8"))
    current_code={c["id"]:hashlib.sha256("".join(c["source"]).encode()).hexdigest() for c in current_notebook["cells"] if c["cell_type"]=="code"}
    require(all(current_code.get(cid)==digest for cid,digest in HY_APPROVED_CODE_HASHES.items()),"Earlier approved notebook code changed.")
    require(hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()==artifact_sha256 and corpus["source"]["sha256"]==M_RUN["source"]["sha256"],"Source fingerprints differ.")
    ensure_hybrid_current()
    show_table(HY_VALIDATION_COUNTS)
    print("PASS: exact source coverage, contained complete parents, tested child spans, stable IDs, provenance, uncertainty, unchanged prior artifacts/code, and review gate.")

### 12e. What is supported, provisional and unresolved?

| Area | Stage 2 supports | Provisional carry-forward choice | Later evaluation question |
|---|---|---|---|
| Structural units | Exact preserved spans and reviewed boundary evidence | Existing provisional units; no merging/reparenting | Which unreviewed boundaries, short headings and mixed-topic units need correction? |
| Oversized trigger | Measured 2k/4k/8k effects | 4,000 characters; 2,000 sensitivity | Do actual Bulgarian token budgets make 4k intact units impractical? |
| Child method | Aware cuts reduce avoidable paragraph breaks | Reuse paragraph → whitespace → exact fallback from Step 8 | Does paragraph preservation improve evidence relevance and completeness? |
| Child size / overlap | 2k/0% avoids duplication and cuts fewer paragraphs than its 10% counterpart here | 2,000 characters / 0%; 10% sensitivity | Does overlap recover missed conditions or merely repeat content? |
| Parent policy | Exact explicit containment and context-expansion measurements | Original oversized unit; no automatic Chapter expansion | Which parent portions fit context budgets and actually restore missing context? |
| Metadata/provenance | Registry joins, raw fields, IDs and source maps resolve | Document/run registries with light chunk and relationship references | Which fields support meaningful filters; how to handle raw version semantics? |
| Uncertain text | Residual/unclassified content can be preserved without invented roles | Keep explicit uncertainty; same tested children inside oversized source containers | Should headings/navigation remain separate future search targets or only source context? |
| History | Original notes remain traceable; separate annotations expose their share | Keep history in original text plus separate candidate annotations | Does history help temporal questions or dilute focus, and can scope be verified? |

No production metadata schema, token budget, parent-expansion policy or optimal strategy is established.
The mentor correspondences remain qualitative references; they do not justify corpus-building rules,
parameter selection or exhaustive coverage claims. Stage 2 supports carrying this hypothesis into a
controlled next-stage evaluation because it is reversible, traceable, fully covered and exposes its
known weaknesses. It does **not** establish retrieval quality.

In [ ]:
if HY_READY:
    ensure_hybrid_current()
    HY_DECISION={"status":"provisional shortlist for later embedding/retrieval experiments; not production-optimal",
        "lead":HY_SHORTLIST[0],"sensitivity_alternatives":HY_SHORTLIST[1:],
        "structural_policy":"Keep original provisional units whole at/below trigger; no new boundaries, merging or ancestry",
        "parent_policy":"Retain original oversized source unit; explicit parent lookup, no automatic whole-parent LLM insertion or Chapter widening",
        "metadata_provenance":"Existing Stage 10 document/run registries plus source-unit, span, parent/link references; raw dates, never effective-date inference",
        "ambiguous_spans":"Retain residual/unclassified labels and warnings; oversized unknown spans remain uncertain source containers",
        "history":"Unchanged original text plus separate Stage 9 candidate annotations; no removal or metadata-only policy",
        "supported_decisions":["Exact source reconstruction and containment","Reuse of already-tested boundaries/child spans",
                               "Fewer avoidable paragraph cuts with aware splitting","Explicit uncertainty and provenance preservation"],
        "provisional_choices":["4,000-character trigger","2,000-character child target","0% lead overlap","Source-unit parent rather than widened Chapter"],
        "unresolved_tests":["Actual Bulgarian tokenizer sizes and model context limits","Held-out legal questions requiring conditions, exceptions and cross-references",
                            "Lead versus overlap/threshold sensitivity and strict fixed/uncapped structural controls",
                            "Child-only versus budgeted parent context, including long Article/residual counterexamples",
                            "Unclassified headings, weak structure and residual/navigation target usefulness",
                            "History scope attribution and qualitative-noise hypotheses","Metadata filtering semantics, raw version dates and citation completeness",
                            "All-21-document generalization beyond the four-document experiment population"],
        "next_stage_not_implemented":["embeddings","vector storage","retrieval","reranking","RAG","LLM calls","agents"]}
    show_json("Stage 2 provisional decision",HY_DECISION)
    HY_REPORT={"report_version":1,"scope":"Step 12 only — provisional hybrid, conclusions and Stage 2 review",
        "run_id":HY_RUN_ID,"spec":HY_SPEC,"rule_ledger":HY_RULES,"result_fingerprint":HY_RESULT_FINGERPRINT,
        "results":{config_id:{**{k:v for k,v in result.items() if k not in ("parents","outputs")},
            "parents":{pid:{k:v for k,v in p.items() if k!="text"} for pid,p in result["parents"].items()},
            "outputs":{cid:{k:v for k,v in c.items() if k!="text"} for cid,c in result["outputs"].items()}}
            for config_id,result in HY_RESULTS.items()},
        "sample_metrics":HY_SAMPLE_METRICS,"baseline_deltas":HY_DELTAS,"shortlist_aggregates":HY_AGGREGATES,
        "sensitivity_samples":HY_SENSITIVITY_SAMPLES,"tradeoffs":HY_TRADEOFFS,"decision":HY_DECISION,
        "provenance":{"artifact_sha256":artifact_sha256,"docx_sha256":corpus["source"]["sha256"],"registry_ref":M_RUN_ID,
                      "unchanged_prior_artifacts":HY_PRIOR_HASHES,"unchanged_prior_code":HY_APPROVED_CODE_HASHES},
        "validation":{"status":"PASS","configurations":HY_VALIDATION_COUNTS,"legal_validation":False}}
    hybrid_report_path=OUTPUT_DIR/f"hybrid_experiment_report.{fingerprint(HY_REPORT)}.json"
    hybrid_bytes=(canonical(HY_REPORT)+"\n").encode("utf-8")
    if hybrid_report_path.exists():require(hybrid_report_path.read_bytes()==hybrid_bytes,"Existing hybrid report differs.")
    else:hybrid_report_path.write_bytes(hybrid_bytes)
    require(json.loads(hybrid_report_path.read_text(encoding="utf-8"))==HY_REPORT,"Hybrid report round trip failed.")
    lead=HY_AGGREGATES[0]
    HY_SUMMARY=f"""# Stage 2 — provisional decision for review

    Carry forward the source-preserving hybrid as a **hypothesis for embedding/retrieval evaluation**, not a production-optimal strategy.

    - **Units:** preserve existing provisional structural units through 4,000 Unicode characters. Retain larger units as complete parents.
    - **Children:** reuse Step 8 paragraph/whitespace-aware splitting at 2,000 characters, initially 0% overlap. No new splitter.
    - **Sensitivity shortlist:** 4,000 / 2,000 / 10% requested overlap; 2,000 / 2,000 / 0% overlap.
    - **Parents:** original oversized unit only; no automatic Chapter expansion or assumption the complete parent fits an LLM budget.
    - **Uncertainty:** keep residual/unclassified spans and warnings; oversized unknowns remain source containers with no asserted legal ancestry.
    - **Metadata/history:** existing document/run registries, exact source maps and explicit links; dates and history remain verbatim, with history annotations separate.

    Why carry it forward: at the 4,000 trigger and 2,000/0% setting, aware splitting cuts 6 source paragraphs versus 169 with exact windows, with full coverage in both. It produces 236 children rather than 219: paragraph preservation costs more, sometimes underfilled, spans. Whole shorter units avoid unnecessary provision splitting. These are mechanical observations, not retrieval findings.

    The lead retains {lead['retained_whole_units']} units whole and {lead['oversized_parents']} complete parents, including {lead['uncertain_source_containers']} uncertain source containers. Its {lead['children']} children plus kept units make {lead['candidate_payload_count']} payload spans covering {lead['source_characters']:,} source characters across four complete documents. Peer overlap duplication is {lead['duplicate_peer_characters']} characters. All 12 frozen samples are covered.

    Regressions and limits: long rules are fragmented in the payload layer versus uncapped structural units; {lead['payloads_above_2000']} intact payloads still exceed 2,000 characters and {lead['payloads_below_100']} are shorter than 100. Complete parents can reach {lead['complete_parent_sizes']['max']:,} characters. Tiny headings, weak structure and uncertain boundaries remain. Zero overlap may lose useful local context. History may dominate some spans, but its retrieval effect is untested.

    Next evaluation must test actual Bulgarian token budgets; conditions, exceptions and cross-references; overlap/threshold sensitivities; budgeted parent expansion; ambiguous-content targets; history scope and metadata filters. Generalization beyond the four experimental documents is unresolved. The mentor material remains a qualitative reference, not exhaustive ground truth or a corpus-building input.

    Validation: exact source reconstruction, coverage, containment, deterministic IDs, provenance, and unchanged earlier artifacts/code. A restart reproducibility check accompanies the executed notebook. Legal correctness is not established by these checks.

    **Stopped after Step 12 for review. No embeddings, vector storage, retrieval, reranking, RAG, LLM calls or agents implemented.**
    """
    hybrid_summary_path=OUTPUT_DIR/f"stage2_decision_summary.{HY_RUN_ID}.md"
    if hybrid_summary_path.exists():require(hybrid_summary_path.read_text(encoding="utf-8")==HY_SUMMARY,"Existing decision summary differs.")
    else:hybrid_summary_path.write_text(HY_SUMMARY,encoding="utf-8")
    display(HTML(details_html("Concise Stage 2 decision summary",HY_SUMMARY)))
    print("Hybrid report:",hybrid_report_path.relative_to(ROOT))
    print("Stage 2 decision summary:",hybrid_summary_path.relative_to(ROOT))
    print("STOP AFTER STEP 12 — provisional hybrid and Stage 2 conclusions ready for review; no next-stage implementation.")

### Stage 2 review checkpoint

If the evidence gate passed, the local report, source-backed comparisons and concise decision
summary are ready for review. If observations or approval were absent, the experiment was skipped
with “hybrid hypothesis pending review”; no older results should be treated as a new run.

**STOP after Step 12.** Approve or revise the provisional shortlist before any next-stage work.
No production-optimal configuration is claimed, and no embedding/retrieval implementation follows.